<h1 align="center">Модель прогнозирования стоимости жилья для агентства недвижимости</h1>

## Введение

Динамика современного рынка недвижимости требует от агентств максимальной оперативности и точности при оценке объектов. В условиях жесткой конкуренции классические методы анализа замедляют бизнес-процессы и увеличивают риск финансовых потерь. Внедрение интеллектуальных алгоритмов позволяет автоматизировать рутинные расчеты и свести к минимуму человеческий фактор. 

**Задача проекта** — разработать модель, которая позволила бы агентству недвижимости обойти конкурентов по скорости и качеству совершения сделок.

**План реализации проекта**

Для достижения поставленной цели необходимо реализовать полный цикл разработки в рамках следующих этапов:

- *Загрузка и первичная обработка данных*: импорт данных и библиотек, очистка от выбросов, пропусков и дубликатов.
- *Разведочный анализ (EDA)*: выявление скрытых закономерностей и ключевых факторов ценообразования.
- *Проектирование признаков (Feature Engineering)*: генерация новых метрик для повышения точности модели.
- *Разработка и валидация*: обучение пула моделей, подбор гиперпараметров и выбор лучшего алгоритма.
- *Вывод в продакшн (Deployment)*: интеграция готовой модели в инфраструктуру агентства. 

Информационную основу исследования составляют реальные исторические данные о рынке жилой недвижимости США, включающие физические характеристики объектов (площадь, этажность, архитектурные особенности), их геолокацию, инфраструктурные параметры (ближайшие школы), а также текущий рыночный статус. Специфика датасета заключается в наличии естественных шумов, пропусков и неструктурированных текстовых полей, что требует глубокой предварительной обработки.

Начнем с подготовки окружения: загрузим ключевые библиотеки и определим базовые функции для автоматизации рутинных операций

## Инициализация проекта и импорт библиотек

### Импорт базовых библиотек 

In [1]:
# Базовые библиотеки и предобработка
import pandas as pd
import numpy as np
import re
from typing import List
from ydata_profiling import ProfileReport
from pprint import pprint
import ast
import random
import time
from geopy.geocoders import Nominatim
import json

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns

# Настройка стилей графиков
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
plt.rcParams["patch.force_edgecolor"] = True

# Статистика, продакшн и настройки
import scipy.stats as stats
import warnings
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

### Базовые функции

In [2]:
def find_non_numeric_chars(series: pd.Series) -> List[str]:
    """Находит все уникальные нечисловые символы в текстовой колонке DataFrame.

    Функция сканирует переданный признак, игнорирует пропуски (NaN) и с помощью
    регулярных выражений извлекает любые символы, которые не являются цифрами 
    от 0 до 9 (буквы, знаки препинания, пробелы, спецсимволы).

    Args:
        series (pd.Series): Анализируемая колонка датасета (обычно типа object).

    Returns:
        List[str]: Отсортированный список всех уникальных нечисловых символов.
    """
    # Превращаем все значения в строки, исключая честные NaN
    strings = series.dropna().astype(str)
    
    # Находим всё, что НЕ является цифрой от 0 до 9
    non_digits = strings.apply(lambda x: re.findall(r'[^0-9]', x))
    
    # Собираем все найденные символы в один плоский список и берем уникальные
    unique_chars = sorted(list(set([char for sublist in non_digits for char in sublist])))
    
    return unique_chars

def find_text_patterns(series: pd.Series) -> List[str]:
    """Находит все уникальные текстовые слова и фразы в зашумленной числовой колонке.

    Функция сканирует переданный признак, игнорирует пропуски (NaN) и с помощью
    регулярных выражений извлекает все непрерывные последовательности букв и слов,
    очищая их от цифр и знаков препинания. Результат возвращается в виде списка
    уникальных фраз, отсортированных по алфавиту.

    Args:
        series (pd.Series): Анализируемая колонка датасета (обычно типа object).

    Returns:
        List[str]: Отсортированный список всех уникальных текстовых слов и фраз.
    """
    # Превращаем все значения в строки, исключая честные NaN
    strings = series.dropna().astype(str)
    
    # Находим все последовательности латинских букв, пробелов и двоеточий
    # Игнорируем цифры, запятые и знаки доллара
    raw_patterns = strings.apply(lambda x: re.findall(r'[a-zA-Z\s:]+', x))
    
    # Собираем все найденные фразы в один плоский список
    flat_list = [phrase.strip() for sublist in raw_patterns for phrase in sublist]
    
    # Очищаем список от пустых строк и берем только уникальные значения
    unique_patterns = sorted(list(set([phrase for phrase in flat_list if phrase])))
    
    return unique_patterns

def get_gip(p_value, alpha=0.05):
    """Функция, помогающая принять или отвергнуть нулевую гипотезу.
    
    Args:
        p_value (float): p-value статистического теста
        alpha (float): уровень значимости, по умолчанию 0.05
        
    Returns:
        str: подсказка о результате проверки гипотезы
    """
    print(f'p-value = {p_value:.3f}')
    
    if p_value <= alpha:
        print(f'p-значение меньше, чем заданный уровень значимости {alpha}. '
              'Отвергаем нулевую гипотезу.')
    else:
        print(f'p-значение больше, чем заданный уровень значимости {alpha}. '
              'У нас нет оснований отвергнуть нулевую гипотезу.')

## Анализ структуры и типов данных

### Знакомство со структурой данных

In [3]:
# Загрузка данных
data = pd.read_csv('data/data.zip')
print('Размер данных: {}'.format(data.shape))
data.head(3)

Размер данных: (377185, 18)


,status,private pool,propertyType,street,baths,homeFacts,fireplace,city,schools,sqft,zipcode,beds,state,stories,mls-id,PrivatePool,MlsId,target
0,Active,NaN,Single Family Home,240 Heather Ln,3.5,"{'atAGlanceFacts': [{'factValue': '2019', 'fac...",Gas Logs,Southern Pines,"[{'rating': ['4', '4', '7', 'NR', '4', '7', 'N...",2900,28387,4,NC,NaN,NaN,NaN,611019,"$418,000"
1,for sale,NaN,single-family home,12911 E Heroy Ave,3 Baths,"{'atAGlanceFacts': [{'factValue': '2019', 'fac...",NaN,Spokane Valley,"[{'rating': ['4/10', 'None/10', '4/10'], 'data...","1,947 sqft",99216,3 Beds,WA,2.0,NaN,NaN,201916904,"$310,000"
2,for sale,NaN,single-family home,2005 Westridge Rd,2 Baths,"{'atAGlanceFacts': [{'factValue': '1961', 'fac...",yes,Los Angeles,"[{'rating': ['8/10', '4/10', '8/10'], 'data': ...","3,000 sqft",90049,3 Beds,CA,1.0,NaN,yes,FR19221027,"$2,895,000"


**Описание исходных признаков датасета:**

Данные представляют собой 377 185 объявлений о продаже различных объектов недвижимости, каждое объявление описано 18 признаками (фактически 15 характеристик, так как 1 признак имеет дубль данных, и имеется 2 идентификатора):

| Название признака | Описание признака | Тип / Специфика данных |
| -- | -- | -- |
| `target` | Цена объекта недвижимости | Числовой (**целевой признак**) |
| `status` | Статус продажи (активен, под контрактом, аукцион и др.) | Категориальный (содержит синонимы и сокращения) |
| `propertyType` | Тип объекта недвижимости (апартаменты, кондо, дом и т.д.) | Категориальный (требует стандартизации) |
| `street` | Адрес объекта | Текстовый |
| `city` | Город | Категориальный |
| `state` | Штат | Категориальный |
| `zipcode` | Почтовый индекс | Категориальный / Числовой |
| `sqft` | Площадь объекта в квадратных футах | Числовой |
| `beds` | Количество спален | Числовой |
| `baths` | Количество ванных комнат | Текстовый / Числовой (содержит шумы) |
| `stories` | Количество этажей | Текстовый / Числовой |
| `fireplace` | Наличие камина | Категориальный / Булев |
| `private pool` / `PrivatePool` | Наличие собственного бассейна | Дублирующиеся признаки (требуют объединения) |
| `mls-id` / `MlsId` | Идентификатор в системе мультилистинга | Идентификаторы (дубли, малозначимы для оценки) |
| `homeFacts` | Сведения о строительстве объекта (год, ремонт и т.д.) | **Словарь JSON** (требует десериализации) |
| `schools` | Сведения о школах в районе | **Словарь JSON** (требует десериализации) |

### Первичная очистка данных

На этапе знакомства с данными необходимо выполнить базовые технические операции над всем массивом, чтобы подготовить его к дальнейшему анализу. Текущий размер датасета составляет 18 столбцов. Мы исключим из него полные дубликаты строк, а также удалим неинформативные системные столбцы `mls-id` и `MlsId` (технические идентификаторы объявлений), поскольку они уникальны для каждой записи и не несут экономического или физического смысла для моделирования стоимости недвижимости.

In [4]:
# Удаляем полные дубликаты строк
data.drop_duplicates(inplace=True)

# Удаляем технические идентификаторы
data.drop(columns=['mls-id', 'MlsId'], inplace=True)

# Фиксируем новый размер датасета
print(f"Размер датасета после первичной очистки: {data.shape}")

Размер датасета после первичной очистки: (377135, 16)


### Основная статистическая информация

In [5]:
# Общая информация о типах и строках
print("Общая информация о структуре данных:")
data.info()

Общая информация о структуре данных:
<class 'pandas.core.frame.DataFrame'>
Int64Index: 377135 entries, 0 to 377184
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   status        337218 non-null  object
 1   private pool  4181 non-null    object
 2   propertyType  342402 non-null  object
 3   street        377133 non-null  object
 4   baths         270827 non-null  object
 5   homeFacts     377135 non-null  object
 6   fireplace     103112 non-null  object
 7   city          377101 non-null  object
 8   schools       377135 non-null  object
 9   sqft          336585 non-null  object
 10  zipcode       377135 non-null  object
 11  beds          285881 non-null  object
 12  state         377135 non-null  object
 13  stories       226462 non-null  object
 14  PrivatePool   40310 non-null   object
 15  target        374655 non-null  object
dtypes: object(16)
memory usage: 48.9+ MB


In [6]:
# Основная статистика по текстовым полям
data.describe(include=['object'])

,status,private pool,propertyType,street,baths,homeFacts,fireplace,city,schools,sqft,zipcode,beds,state,stories,PrivatePool,target
count,337218,4181,342402,377133,270827,377135,103112,377101,377135,336585,377135,285881,377135,226462,40310,374655
unique,159,1,1280,337076,229,321009,1653,2026,297365,25405,4549,1184,39,348,2,43939
top,for sale,Yes,single-family home,Address Not Disclosed,2 Baths,"{'atAGlanceFacts': [{'factValue': '', 'factLab...",yes,Houston,"[{'rating': [], 'data': {'Distance': [], 'Grad...",0,32137,3 Beds,FL,1.0,yes,"$225,000"
freq,156058,4181,92199,672,52458,7174,50353,24441,4204,11854,2141,53454,115434,67451,28792,1462


In [7]:
# Расчет и вывод таблицы пропусков
missing_data = data.isnull().sum().to_frame(name='Количество пропусков')
missing_data['Доля пропусков, %'] = round(
    (missing_data['Количество пропусков'] / len(data)) * 100, 2
)
display(missing_data.sort_values(by='Количество пропусков', ascending=False))

,Количество пропусков,"Доля пропусков, %"
private pool,372954,98.89
PrivatePool,336825,89.31
fireplace,274023,72.66
stories,150673,39.95
baths,106308,28.19
beds,91254,24.20
sqft,40550,10.75
status,39917,10.58
propertyType,34733,9.21
target,2480,0.66


In [ ]:
"""
# Отбираем для отчета только текстовые и числовые признаки (исключаем JSON)
columns_for_report = [col for col in data.columns if col not in ['homeFacts', 'schools']]

# Генерируем отчет по облегченной выборке данных
profile = ProfileReport(data[columns_for_report], title="Real Estate Analysis Report", minimal=True)

# Сохраняем результат в HTML-файл
profile.to_file("data/real_estate_analysis_report.html")
print("Интерактивный аналитический отчет сохранен как 'real_estate_analysis_report.html'")
"""

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Интерактивный аналитический отчет сохранен как 'real_estate_analysis_report.html'


**Анализ статистической информации и выявленные аномалии**

Первичный комплексный анализ структуры данных показал, что датасет находится в «сыром» состоянии и требует обязательной предобработки. На данном этапе зафиксированы следующие ключевые проблемы и инсайты:

1. **Проблема типов данных и нечисловых символов**: Ключевые непрерывные метрики — стоимость (`target`), площадь (`sqft`), количество этажей (`stories`), спален (`beds`) и ванных комнат (`baths`) — распознаются системой как текстовые строки (`object`). Это указывает на наличие скрытых нечисловых символов (знаков валют, единиц измерения, разделителей), которые блокируют математические операции.
2. **Пропуски в целевом признаке (`target`)**: В колонке цен зафиксировано 2 480 пропущенных значений. Поскольку это целевой признак для модели машинного обучения, строки с пропусками подлежат удалению на следующем этапе.
3. **Объединение и бинаризация признаков бассейнов**: Колонки `private pool` и `PrivatePool` дублируют друг друга по смыслу и содержат критический объем пропусков (от 89% до 98%). Высока вероятность, что пропуск означает отсутствие бассейна (логический ноль). Эти признаки необходимо будет объединить в один чистый бинарный флаг (1 — есть бассейн, 0 — нет).
4. **Масштабные пропуски в качественных характеристиках**: Камины (`fireplace`), этажность (`stories`), ванные комнаты (`baths`) и спальни (`beds`) имеют от 24% до 72% пропущенных значений, что потребует детального изучения и разработки стратегии их восстановления.
5. **Высокая избыточность категорий (`propertyType`, `status`)**: Поля содержат аномально большое количество уникальных значений (1279 и 156 соответственно) из-за отсутствия стандартизации названий и регистров букв.
6. **Критические вложенные структуры (`homeFacts`, `schools`)**: Первичный осмотр строк и автоматический анализ выявили, что внутри этих колонок зашиты сложные словари и списки JSON. Огромный объем неструктурированного текста в этих полях вызывает переполнение оперативной памяти (MemoryError) при попытке комплексного профилирования. Использование этих признаков в исходном виде невозможно.

**План дальнейших действий:**
Проведение качественного разведочного анализа (EDA) в текущем виде не представляется возможным. В связи с этим мы переходим к разделу **«Попризнаковый анализ и глубокая предобработка данных»**, где последовательно, колонка за колонкой, исследуем скрытые символы регулярными выражениями, очистим шум, исключим аномалии (такие как объявления об аренде), восстановим пропуски, проведем бинаризацию и десериализуем сложные структуры данных.

## Детальная предобработка признаков и стандартизация данных

Первоначально необходимо распаковать вложенные структуры для признаков 'homeFacts', 'schools'

### Десериализация JSON-описания объекта ('homeFacts')

In [9]:
# Посмотрим структуру
data["homeFacts"][0]

"{'atAGlanceFacts': [{'factValue': '2019', 'factLabel': 'Year built'}, {'factValue': '', 'factLabel': 'Remodeled year'}, {'factValue': 'Central A/C, Heat Pump', 'factLabel': 'Heating'}, {'factValue': '', 'factLabel': 'Cooling'}, {'factValue': '', 'factLabel': 'Parking'}, {'factValue': None, 'factLabel': 'lotsize'}, {'factValue': '$144', 'factLabel': 'Price/sqft'}]}"

In [10]:
# Найдем названия новых признаков, проверим что не меняется от строки к строке
factLabel = data["homeFacts"].str.findall(r"\bfactLabel': ([\s\S]+?)[}\b]")
factLabel.value_counts()

['Year built', 'Remodeled year', 'Heating', 'Cooling', 'Parking', 'lotsize', 'Price/sqft']    377135
Name: homeFacts, dtype: int64

In [11]:
# Создаем список без кавычек
list_label = ",".join(factLabel[0]).replace("'", "").split(",")
# Извлекаем значения
values = data["homeFacts"].str.findall(r"\bfactValue': ([\s\S]+?), 'factLabel\b")

# Цикл заполнения признаков (с добавлением очистки кавычек)
for i, val in enumerate(list_label):
    # Добавляем .str.strip("'\" "), чтобы убрать кавычки по краям значений
    # И .replace, чтобы превратить текстовые 'None' и пустые строки в честный np.nan
    data[val] = (
        values.apply(lambda x: x[i])
        .str.strip("'\" ")
        .replace(["", "None", "none", "null"], np.nan)
    )
# Смотрим результат
display(data[list_label].head())
# Удаляю исходный
data.drop('homeFacts', axis=1, inplace=True)
# Фиксируем новый размер датасета
print(f"Размер датасета после распаковки 'homeFacts': {data.shape}")

,Year built,Remodeled year,Heating,Cooling,Parking,lotsize,Price/sqft
0,2019,NaN,"Central A/C, Heat Pump",NaN,NaN,NaN,$144
1,2019,NaN,NaN,NaN,NaN,5828 sqft,$159/sqft
2,1961,1967,Forced Air,Central,Attached Garage,"8,626 sqft",$965/sqft
3,2006,2006,Forced Air,Central,Detached Garage,"8,220 sqft",$371/sqft
4,NaN,NaN,NaN,NaN,NaN,"10,019 sqft",NaN


Размер датасета после распаковки 'homeFacts': (377135, 22)


Обработаем по очереди распакованные признаки

### Хронологические параметры объектов недвижимости ('Year built', 'Remodeled year')

In [12]:
# Проверяем на скрытые символы и текстовые патерны
year_cols = 'Year built', 'Remodeled year'
for col in year_cols:
    print(f'\nУникальные нечисловые символы в {col}:')
    pprint(find_non_numeric_chars(data[col]), width=120, compact=True)
    print(f'\nУникальные текстовые паттерны в {col}:')
    pprint(find_text_patterns(data[col]), width=120, compact=True)


Уникальные нечисловые символы в Year built:
[' ', 'D', 'N', 'a', 'o', 't']

Уникальные текстовые паттерны в Year built:
['No Data']

Уникальные нечисловые символы в Remodeled year:
[]

Уникальные текстовые паттерны в Remodeled year:
[]


In [13]:
# Очищаем
for col in year_cols:
    # Переводим в числа, весь явный текст отправляется в nan
    data[col] = pd.to_numeric(data[col], errors='coerce')
    # Очищаем аномальные года (все, что больше 2026 года или нереалистично малы, например, до 1700 года)
    data.loc[(data[col] > 2026) | (data[col] < 1700), col] = np.nan
    data[col] = data[col].astype('Int16')

print(f"Диапазон значений года постройки:"
      f"{data['Year built'].min()} - {data['Year built'].max()}")
print(f"Диапазон значений года реконструкции:"
      f"{data['Remodeled year'].min()} - {data['Remodeled year'].max()}")

Диапазон значений года постройки:1700 - 2025
Диапазон значений года реконструкции:1738 - 2021


### Удельные стоимостные показатели ('Price/sqft')

На мой взгляд ценнейший признак, так как в рамках реальной оценки имущества всегда используют удельный показатель стоимости. В данном же проекте целевая переменная общая стоимость, что логично с точки зрения обычного пользователя нашей будущей программы. Очевидно, что данный показатель будет сильно коррелировать с целевой, и очевидно, что его необходимо исключить из финального набора признаков во избежание утечки целевой переменной (data leakage). Но тем не менее на данном этапе его ценность в том, что можно восстановить пропущенные значения в таргете (их 2480 строк) с их помощью, поэтому очищаю, перевожу в числовой формат и оставляю, до момента обработки таргета.

In [14]:
# Проверяем на скрытые символы и текстовые патерны
print(f"\nУникальные нечисловые символы в {'Price/sqft'}:")
pprint(find_non_numeric_chars(data['Price/sqft']), width=120, compact=True)
print(f"\nУникальные текстовые паттерны в {'Price/sqft'}:")
pprint(find_text_patterns(data['Price/sqft']), width=120, compact=True)


Уникальные нечисловые символы в Price/sqft:
[' ', '$', ',', '.', '/', 'C', 'D', 'F', 'I', 'N', 'S', 'a', 'c', 'e', 'f', 'g', 'm', 'n', 'o', 'q', 'r', 's', 't']

Уникальные текстовые паттерны в Price/sqft:
['Contact manager', 'Ft', 'No Data', 'No Info', 'Sq', 'sqft']


In [15]:
# Переводим в строку и нижний регистр для точного поиска
data['Price/sqft'] = data['Price/sqft'].astype(str).str.lower()

# Проверим строки, где есть слова-маркеры И при этом есть хотя бы одна цифра
potential_lost_data = data[
    (data['Price/sqft'].str.contains('contact|info|data', na=False)) & 
    (data['Price/sqft'].str.contains(r'\d', regex=True))
]['Price/sqft']

print(f"Количество строк, где есть И текст, И цифры: {len(potential_lost_data)}")

Количество строк, где есть И текст, И цифры: 0


In [16]:
# Стираем все буквы, знаки доллара, косые черты и пробелы
# Оставляем только цифры и точку
data['Price/sqft'] = (
    data['Price/sqft'].astype(str).str.replace(r'[^\d.]', '', regex=True)
)
# Переводим в числовой формат
data['Price/sqft'] = pd.to_numeric(data['Price/sqft'], errors='coerce')
print(f"Диапазон цен за кв. фут: {data['Price/sqft'].min()} - {data['Price/sqft'].max()}")

Диапазон цен за кв. фут: 0.0 - 5950000.0


In [17]:
# Посмотрим на строки с аномально высокой и нулевой ценой за кв. фут
high_price_examples = data[data['Price/sqft'] > 100000][['Price/sqft', 'sqft', 'target', 'city', 'propertyType']]
zero_price_examples = data[data['Price/sqft'] == 0][['Price/sqft', 'sqft', 'target', 'city', 'propertyType']]

print(f"Количество строк с ценой за кв.фут > $100k: {len(high_price_examples)}")
display(high_price_examples.head(5))

print(f"\nКоличество строк с ценой за кв.фут == 0: {len(zero_price_examples)}")
display(zero_price_examples.head(5))


Количество строк с ценой за кв.фут > $100k: 65


,Price/sqft,sqft,target,city,propertyType
3763,217700.0,Total interior livable area: 1 sqft,"$217,700",Pottstown,Single Family
5961,1499000.0,1 sqft,"$1,499,000",Panama City,lot/land
16440,1300000.0,1 sqft,"$1,300,000+",Chicago,condo
18537,480000.0,1 sqft,"$480,000",Chicago,multi-family
29107,245000.0,1 sqft,"$245,000",Chicago,lot/land



Количество строк с ценой за кв.фут == 0: 165


,Price/sqft,sqft,target,city,propertyType
1357,0.0,"2,544 sqft","$1,000",Detroit,single-family home
2282,0.0,"2,048 sqft","$1,000",Detroit,single-family home
6862,0.0,"2,650 sqft","$1,000",Detroit,Single Family
10349,0.0,"2,130 sqft","$1,000",Detroit,single-family home
15745,0.0,"2,270 sqft","$1,000",Detroit,single-family home


По представленным данным прослеживаются следующие аномалии:
- Некоторые строки удельной цены заполнились общей ценой в связи с тем что площадь указана как 1 кв.фут. Здесь  очевидно необходимо площадь исправить на NaN как неизвестную, и удельную цену также исправить на NaN
- Некоторые поля заполнились 0 значением в связи с маленькой целевой суммой - которые по факту являются аномалиями в датасете, так как цена не реальная, скорее всего данные предложения из категории аренды или стартовой цены аукциона.

Для соблюдения верной структуры данного отчета, я вернусь к разбору аномалий позже, когда все признаки датасета будут очищены, так как в настоящий момент можно случайно потерять или не учесть какие-то данные. Перейдем к следующему распакованному признаку 'lotsize'.

### Физические масштабы земельных участков ('lotsize')

In [18]:
# Проверяем на скрытые символы и текстовые патерны
print(f"\nУникальные нечисловые символы в {'lotsize'}:")
pprint(find_non_numeric_chars(data['lotsize']), width=120, compact=True)
print(f"\nУникальные текстовые паттерны в {'lotsize'}:")
pprint(find_text_patterns(data['lotsize']), width=120, compact=True)


Уникальные нечисловые символы в lotsize:
[' ', ',', '-', '.', 'A', 'D', 'F', 'N', 'S', 'a', 'c', 'e', 'f', 'l', 'o', 'q', 'r', 's', 't', '—']

Уникальные текстовые паттерны в lotsize:
['Acre', 'Acres', 'Ft', 'No Data', 'Sq', 'acre', 'acre lot', 'acres', 'acres lot', 'sqft', 'sqft lot']


В признаке 'lotsize' смешаны две единицы измерения: акры и квадратные футы. Для очистки переведем все акры в квадратные футы (1 акр = 43 560 кв. футов), удалим текстовый мусор и приведем столбец к числовому формату. Разбор пропусков оставим на потом, так как у квартир участка нет, а у домов он обязан быть.

In [19]:
def clean_lotsize(val):
    """Конвертирует размер участка в квадратные футы и очищает от текстового шума.

    Выделяет первое встреченное числовое значение. Если в исходной строке 
    присутствует маркер акров (acre), пересчитывает значение в кв. футы
    на основе константы (1 акр = 43 560 кв. футов).

    Args:
        val (str, float, int, None): Исходное значение размера участка 
            из датасета (может содержать текст, спецсимволы или пропуски).

    Returns:
        float: Размер участка в квадратных футах (sqft) или np.nan, 
            если числовые данные не обнаружены или передано пустое значение.
    """
    val_str = str(val).lower()
    
    # Пытаемся найти число. Если текста нет или цифры отсутствуют — вернет NaN
    num_match = re.search(r'([\d,.]+)', val_str)
    if not num_match:
        return np.nan
        
    # Очищаем число от запятых и переводим в float
    number = float(num_match.group(1).replace(',', ''))
        
    # Если это акры — конвертируем, иначе возвращаем как есть
    if 'acre' in val_str:
        return number * 43560.0
    return number


# Применяем чистку
data['lotsize'] = data['lotsize'].apply(clean_lotsize)

# Фиксируем успешное изменение типа данных и диапазон значений
print(f"Тип признака 'lotsize' после очистки: {data['lotsize'].dtype}")
print(f"Диапазон значений площади: {data['lotsize'].min()} - {data['lotsize'].max()} sqft")

Тип признака 'lotsize' после очистки: float64
Диапазон значений площади: 0.0 - 2147483647.0 sqft


Ого, диапазон показал гигантский земельный участок - явно шумовой артефакт. Но вернемся к аномалиям позже, а пока перейдем к следующему распакованному признаку - отопление ('Heating').

### Инженерные системы отопления ('Heating')

In [20]:
# Смотрим на топ-20 тегов признака Heating
print(data['Heating'].astype(str).str.lower().str.strip().value_counts().head(20))

forced air                     134308
nan                            109365
other                           29622
electric                        10216
gas                              9296
heat pump                        8851
no data                          8610
central air                      7814
central electric                 7112
central                          6247
central, electric                4253
baseboard                        3815
wall                             3301
electric heat                    3064
heating system                   2709
forced air, heat pump            1767
radiant                          1485
central air, ceiling fan(s)      1432
natural gas heat                 1383
central furnace                  1036
Name: Heating, dtype: int64


Признак `Heating` указывает на тип отопительной системы, установленной в объекте недвижимости. Это крайне зашумленный текстовый показатель, содержащий множество опечаток, дублирующих классов и технических заглушек риелторов. Наша задача на данном этапе — привести исходные текстовые данные к нижнему регистру, очистить их от изолированного шума кондиционирования и укрупнить редкие типы систем в единые технологические категории.

Чтобы не затереть исходный текстовый массив, который понадобится в последующих разделах для сквозного сбора бинарных флагов комфорта (бытовой техники, спец. комнат и элементов роскоши), мы не трогаем оригинальный столбец `Heating`. Результат работы функции стандартизации мы запишем в новый независимый признак — `heating_clean`.

In [21]:
# Фиксирую в качестве глобальных переменных регулярные выражения
cooling_regex = r"\b(ac|air|cool|refrigeration|window|condition|evaporat|fan|split|unit)\b"
heating_regex = r"\b(heating|heat|furnace|gas|pump|electric|boiler|radiat|stove|baseboard|forced)\b"
no_cooling_regex = r"\b(no cool|no air)\b"
no_heating_regex = r"\b(no heat|no heating)\b"


# Функция для классификации систем отопления
def clean_heating(val):
    """Стандартизирует тип отопительной системы и укрупняет редкие классы.

    Метод обрабатывает текстовые описания климатических систем, отсекает
    изолированный шум кондиционирования и преобразует технические заглушки в NaN.

    Args:
        val (str, float, int, None): Исходное текстовое значение из столбца
        'Heating'.

    Returns:
        str: Одна из 6 укрупненных категорий отопления:
            ['forced_air', 'electric', 'central', 'gas', 'heat_pump',
            'no_heating', 'alternative_or_other'].
        np.nan: Для чистых пропусков данных или изолированного шума охлаждения.
    """
    # Защита от системных пропусков данных (None, pd.NA, float('nan'))
    if pd.isna(val):
        return np.nan

    # Приводим к нижнему регистру и убираем случайные пробелы по краям
    val_str = str(val).lower().strip()

    # Фиксируем маркер физического отсутствия тепла
    if any(w in val_str for w in ["no heat", "no heating", "none", "0"]):
        return "no_heating"

    # Определяем, является ли строка чистым шумом кондиционирования без упоминания тепла
    is_pure_cooling = bool(re.search(cooling_regex, val_str)) and not bool(
        re.search(heating_regex, val_str)
    )

    # Отсекаем пустые заглушки в NaN и изолированные кондиционеры
    pure_nan_words = [
        "nan", "no data", "no info", "unknown", "", "contact manager",
    ]
    if (val_str in pure_nan_words) or is_pure_cooling:
        return np.nan

    # Классифицируем топ-5 основных технологических групп датасета
    if "pump" in val_str:
        return "heat_pump"
    if "forced" in val_str or "hot air" in val_str:
        return "forced_air"
    if any(w in val_str for w in ["gas", "furnace", "propane"]):
        return "gas"
    if "electric" in val_str:
        return "electric"
    if "central" in val_str:
        return "central"

    # Собираем все редкие типы (печи, геотермальное, солнечные панели,
    # настенные обогреватели и слово 'other') в один укрупненный класс
    return "alternative_or_other"


# Перезаписываем столбец Heating результатами работы функции (временно тип object для склейки)
data["heating_clean"] = data["Heating"].apply(clean_heating)
# Смотрим результат
data["heating_clean"].value_counts(dropna=False)

forced_air              140483
NaN                     129348
alternative_or_other     43200
electric                 26794
gas                      16117
heat_pump                13444
central                   7623
no_heating                 126
Name: heating_clean, dtype: int64

### Инженерные системы кондиционирования ('Cooling')

In [22]:
# Выводим топ-20 исходных уникальных значений охлаждения для анализа
print(data['Cooling'].astype(str).str.lower().str.strip().value_counts().head(20))

central                                            158744
nan                                                131319
central air                                         14384
no data                                             10615
has cooling                                          9730
central electric                                     6154
wall                                                 4017
central gas                                          3573
central heating                                      2807
cooling system                                       2700
central a/c                                          2051
other                                                1840
central a/c (electric), central heat (gas)           1646
central a/c (electric), central heat (electric)      1429
refrigeration                                        1075
central, electric                                    1060
electric                                             1012
evaporative   

Напишем функцию для стандартизации и классификации систем кондиционирования и охлаждения воздуха

In [23]:
# Функция очистки признака кондиционирования
def clean_cooling(val):
    """Стандартизирует тип кондиционирования и укрупняет редкие классы.

    Метод обрабатывает текстовые описания систем охлаждения, отсекает
    изолированный тепловой шум и преобразует технические заглушки в NaN.

    Args:
        val (str, float, int, None): Исходное текстовое значение из столбца
        'Cooling'.

    Returns:
        str: Одна из 5 укрупненных категорий кондиционирования:
            ['central_ac', 'wall_window_unit', 'evaporative_cooling',
            'no_cooling', 'alternative_or_other'].
        np.nan: Для чистых пропусков данных или изолированного шума отопления.
    """
    # Защита от системных пропусков данных (None, pd.NA, float('nan'))
    if pd.isna(val):
        return np.nan

    # Приводим к нижнему регистру и убираем случайные пробелы по краям
    val_str = str(val).lower().strip()

    # Определяем, является ли строка чистым шумом отопления без упоминания кондиционирования
    is_pure_heating = bool(re.search(heating_regex, val_str)) and not bool(
        re.search(cooling_regex, val_str)
    )

    # Зануление (NaN) для технических заглушек и теплового шума
    pure_nan_words = [
        "nan", "no data", "no info", "unknown", "", "contact manager",
    ]
    if (val_str in pure_nan_words) or is_pure_heating:
        return np.nan

    # Физическое отсутствие охлаждения
    if any(w in val_str for w in ["no cool", "no air", "none", "0"]):
        return "no_cooling"

    # Классификация основных категорий охлаждения рынка США
    if "evaporative" in val_str:
        return "evaporative_cooling"

    # Ловим central-системы, включая опечатку риелторов (приоритет поднят выше unit)
    central_words = [
        "central", "cenrtal", "a/c", "condition", "central air", "cooling",
    ]
    if any(w in val_str for w in central_words):
        return "central_ac"

    # Оконные и настенные блоки (смещены вниз, чтобы не перехватывать central_ac через слово unit)
    if any(w in val_str for w in ["wall", "window", "unit"]):
        return "wall_window_unit"

    # Собираем редкие типы в единый класс
    return "alternative_or_other"


# Перезаписываем столбец Cooling в новый чистый признак (временно тип object для склейки)
data["cooling_clean"] = data["Cooling"].apply(clean_cooling)
# Смотрим результат
data["cooling_clean"].value_counts(dropna=False)

central_ac              196805
NaN                     167165
alternative_or_other      6339
wall_window_unit          5337
evaporative_cooling       1317
no_cooling                 172
Name: cooling_clean, dtype: int64

Тщательный аудит сырых данных показал, что заполнение данных агентами по недвижимости носило несистемный характер: данные о кондиционировании (например, 'Central Air') вносились в поле отопления 'Heating', а маркеры тепла ('Gas Heat', 'Steam Heating') оказывались в поле охлаждения 'Cooling'. 

Чтобы восстановить эти скрытые чистые данные и закрыть до 30 000 взаимных пропусков, нами реализован алгоритм кросс-функционального восстановления пропущенных значений (Cross-Feature Imputation). Используя оригинальные, нетронутые сырые колонки-доноры, мы применим к ним соответствующие функции очистки по маскам регулярных выражений. Наполнение будет происходить строго стык-в-стык по оригинальным индексам рядов DataFrame, что полностью исключает смешивание или сдвиг данных.

In [24]:
# Сначала рассчитываем пропуски
cooling_before = data["cooling_clean"].isna().sum()
heating_before = data["heating_clean"].isna().sum()

# Приводим сырые тексты-доноры к нижнему регистру
heating_raw_lower = data["Heating"].astype(str).str.lower()
cooling_raw_lower = data["Cooling"].astype(str).str.lower()

# Выделяем сырые тексты-доноры строго по изолированным маскам
# Для охлаждения: берем из Heating то, где есть охлаждение, но НЕТ отопления и НЕТ запрета охлаждения
cool_from_heat = data["Heating"].where(
    heating_raw_lower.str.contains(cooling_regex, regex=True, na=False)
    & ~heating_raw_lower.str.contains(heating_regex, regex=True, na=False)
    & ~heating_raw_lower.str.contains(no_cooling_regex, regex=True, na=False),
    np.nan,
)

# Для отопления: берем из Cooling то, где есть отопление, но НЕТ охлаждения и НЕТ запрета отопления
heat_from_cool = data["Cooling"].where(
    cooling_raw_lower.str.contains(heating_regex, regex=True, na=False)
    & ~cooling_raw_lower.str.contains(cooling_regex, regex=True, na=False)
    & ~cooling_raw_lower.str.contains(no_heating_regex, regex=True, na=False),
    np.nan,
)

# Применяем родные функции чистки (они возвращают чистый текст / np.nan)
cooling_cross = cool_from_heat.apply(clean_cooling)
heating_cross = heat_from_cool.apply(clean_heating)

# Безопасно склеиваем текстовые object-колонки стык-в-стык
data["cooling_clean"] = data["cooling_clean"].combine_first(cooling_cross)
data["heating_clean"] = data["heating_clean"].combine_first(heating_cross)

# Приводим к финальному типу category
data["cooling_clean"] = data["cooling_clean"].astype("category")
data["heating_clean"] = data["heating_clean"].astype("category")

# 7. Выводим автоматический строгий расчет чистой дельты
print(f"-> Прирост кондиционирования: +{cooling_before - data['cooling_clean'].isna().sum()}")
print(f"-> Прирост отопления: +{heating_before - data['heating_clean'].isna().sum()}")

-> Прирост кондиционирования: +1087
-> Прирост отопления: +8689


### Парковочные пространства ('Parking')

In [25]:
# Смотрим на исходный ТОП-20 парковки перед очисткой
print(data['Parking'].astype(str).str.lower().str.strip().value_counts().head(20))

nan                                 177747
attached garage                      70748
2 spaces                             28061
1 space                              14252
no data                              13333
detached garage                      13200
carport                               7743
off street                            5279
3 spaces                              4724
carport, attached garage              3025
1                                     2936
4 spaces                              2917
2                                     2756
on street                             1707
attached garage, detached garage      1354
0                                     1114
attached garage, carport               993
parking desc                           900
6 spaces                               755
detached garage, attached garage       726
Name: Parking, dtype: int64


Главная сложность очистки заключается в смешении категориальных типов (garage, carport) и числовых данных о количестве мест (1 space, 2 spaces), что обусловлено спецификой различных типов жилой недвижимости (индивидуальные дома или многоквартирные жилые комплексы). Данные будут стандартизированы в несколько базовых категорий, отражающих наличие и тип парковки, а технический мусор превратится в NaN.

In [26]:
# Фиксирую в качестве глобальных переменных маркеры парковки
garage_words = ["garage", "attached", "detached"]
parking_nan_words = [
    "nan", "no data", "no info", "unknown", "", "parking desc",
    "parking type", "garage type", "parkingtype", "parkingfeatures",
    "parking description"
]
navigation_junk_regex = (
    r"(exit|toward|follow|turn|left|right|head|enter|minutes|property is|"
    r"proceed|from i-|from sr-|club|gym|fitness|pool|bbq|barbecue|insurance|"
    r"reserves|laundry|pets|play|tennis)"
)


def clean_parking(val):
    """Стандартизирует типы парковки и укрупняет текстово-числовые данные.

    Args:
        val (str, float, int, None): Исходное текстовое значение из столбца 'Parking'.

    Returns:
        str или np.nan: Одна из 5 укрупненных категорий парковки или np.nan.
    """
    if pd.isna(val):
        return np.nan

    val_str = str(val).lower().strip()

    # Честное зануление (NaN) для пустых технических заглушек и навигационного мусора
    if (val_str in parking_nan_words) or bool(
        re.search(navigation_junk_regex, val_str)
    ):
        return np.nan

    # Явное физическое отсутствие парковки
    if any(
        w in val_str
        for w in [
            "no parking", "none", "no garage", "0", "no covered parking", "no rv parking"
        ]
    ):
        return "no_parking"

    # Выделяем базовые типы парковочных конструкций
    has_garage = any(w in val_str for w in garage_words) or bool(
        re.search(r"\b(att|attch|dtach|adtch|enclosed|gar)\b", val_str)
    )
    has_carport = "carport" in val_str

    # Группа ГАРАЖЕЙ и НАВЕСОВ (включая сложные гибридные комбинации)
    if has_garage and has_carport:
        return "mixed_garage_carport"
    if has_garage:
        return "garage"
    if has_carport:
        return "carport"

    # Специализированные места и укрупненные категории
    if bool(re.search(r"\b(rv|boat|golf cart|motorcycle)\b", val_str)):
        return "special_rv_boat_parking"
    if bool(re.search(r"\b(assigned|reserved|deeded|private|pvt|conveys)\b", val_str)):
        return "assigned_or_reserved_space"

    # Все фиксированные места (spaces, цифры, driveway, off street)
    if (
        "space" in val_str or "lot" in val_str or "pad" in val_str
        or "driveway" in val_str or "open" in val_str or "covered" in val_str
        or "off street" in val_str or "drvwy" in val_str or val_str.isdigit()
    ):
        return "assigned_or_spaces"

    # Уличная парковка вдоль дороги (on street)
    if "on street" in val_str or "onstr" in val_str or val_str == "street":
        return "street_parking"

    return "assigned_or_spaces"


# Применяем чистку к новому столбцу, сохраняя оригинал
data["parking_clean"] = data["Parking"].apply(clean_parking)
# Смотрим результат
data["parking_clean"].value_counts(dropna=False)

NaN                           193048
garage                         95807
assigned_or_spaces             70235
carport                         8560
mixed_garage_carport            5258
street_parking                  1762
no_parking                      1454
assigned_or_reserved_space       734
special_rv_boat_parking          277
Name: parking_clean, dtype: int64

На данном этапе признак 'homeFacts' полностью распакован и очищен от шума. Распакуем теперь признак школ

### Десериализация JSON-массивов образовательной инфраструктуры ('schools')

In [27]:
# Посмотрим структуру
data["schools"][0]

'[{\'rating\': [\'4\', \'4\', \'7\', \'NR\', \'4\', \'7\', \'NR\', \'NR\'], \'data\': {\'Distance\': [\'2.7 mi\', \'3.6 mi\', \'5.1 mi\', \'4.0 mi\', \'10.5 mi\', \'12.6 mi\', \'2.7 mi\', \'3.1 mi\'], \'Grades\': [\'3–5\', \'6–8\', \'9–12\', \'PK–2\', \'6–8\', \'9–12\', \'PK–5\', \'K–12\']}, \'name\': [\'Southern Pines Elementary School\', \'Southern Middle School\', \'Pinecrest High School\', \'Southern Pines Primary School\', "Crain\'s Creek Middle School", \'Union Pines High School\', \'Episcopal Day Private School\', \'Calvary Christian Private School\']}]'

Здесь совершенно другая структура в отличие от homeFacts. Сразу возникает масса вопросов имеет значение название школы или нет или вытащить только средний рейтинг и среднее расстояние?  Но, может лучше минимальное расстояние, это логичнее? А, что если в округе нет старшей школы а покупатель с взрослыми детьми? Данные вопросы больше для раздела EDA, для начала выгрузим данные в отдельную таблицу, проведем очистку и обработку, потом вернем в исходный датасет очищенные данные.

In [28]:
# Читаем строки через ast
parsed = data['schools'].dropna().apply(ast.literal_eval)

# Превращаем всё в плоский список словарей
schools_list = [
    {'house_index': idx, 'school_name': n, 'school_rating': r, 'school_grades': g, 'school_distance': d}
    for idx, obj in parsed.items() if obj and isinstance(obj, list)
    for n, r, g, d in zip(obj[0].get('name', []), obj[0].get('rating', []), 
                          obj[0].get('data', {}).get('Grades', []), obj[0].get('data', {}).get('Distance', []))
]

# Собираем DataFrame школ
schools_df = pd.DataFrame(schools_list)

# Оценим результат
display(schools_df.head(10))

,house_index,school_name,school_rating,school_grades,school_distance
0,0,Southern Pines Elementary School,4,3–5,2.7 mi
1,0,Southern Middle School,4,6–8,3.6 mi
2,0,Pinecrest High School,7,9–12,5.1 mi
3,0,Southern Pines Primary School,NR,PK–2,4.0 mi
4,0,Crain's Creek Middle School,4,6–8,10.5 mi
5,0,Union Pines High School,7,9–12,12.6 mi
6,0,Episcopal Day Private School,NR,PK–5,2.7 mi
7,0,Calvary Christian Private School,NR,K–12,3.1 mi
8,1,East Valley High School&Extension,4/10,9-12,1.65mi
9,1,Eastvalley Middle School,None/10,3-8,1.32mi


Как видно из представленной таблицы, данные по школе требуют глубокой очистки из-за пропущенных значений, разнородного строкового формата и избыточной текстовой информации:
- **Неоднородный формат рейтингов (`school_rating`):** Часть данных представлена в виде дробной шкалы (например, `4/10`), часть в виде одиночных цифр (`4`, `7`), а пропуски зафиксированы строковыми индикаторами `NR` (Not Rated) и `None`. Требуется унификация к единому числовому формату с заменой текстовых пропусков на истинные `NaN`.
- **Текстовые суффиксы в расстояниях (`school_distance`):** Признак содержит метрику `mi` (мили, так как датасет американский) и лишние пробелы (например, `2.7 mi` и `1.65mi`), что переводит колонку в тип `object`. Необходимо очистить строки от единиц измерения для приведения к типу `float`.
- **Скрытые числовые диапазоны в классах (`school_grades`):** На этапе очистки этот строковый признак несет огромную пользу. Мы не будем удалять его, а извлечем из него минимальный и максимальный классы обучения. Это позволит математически точно выставить бинарные флаги ступеней образования (`is_elementary`, `is_middle`, `is_high`) по стандартам США.
- **Скрытые категории в названиях (`school_name`):** Текст содержит ценные маркеры статуса заведения (`Charter`, `Magnet`, `Private`, а также религиозные школы). Для исключения мультиколлинеарности мы сведем их в единую категориальную переменную `school_status` с иерархией приоритета. Дополнительно названия послужат «подушкой безопасности» для точечного заполнения пропусков в ступенях обучения. Вся остальная информация в именах (уникальные названия) является шумом и подлежит удалению.

In [29]:
# Для проверки какие именно текстовые фразы необходимо преобразовать
print(find_text_patterns(schools_df.school_rating))
print(find_text_patterns(schools_df.school_distance))
print(find_text_patterns(schools_df.school_grades))

['NA', 'NR', 'None']
['mi']
['A', 'K', 'K to', 'N', 'NA', 'PK', 'Pk to', 'Preschool to', 'to']


In [30]:
# 1. Очистка school_rating: удаление суффикса шкалы и приведение к NaN
schools_df['school_rating'] = (
    schools_df['school_rating'].astype(str).str.replace('/10', '', regex=False)
    .replace(['NA', 'NR', 'None'], np.nan)
)
schools_df['school_rating'] = pd.to_numeric(
    schools_df['school_rating'], errors='coerce'
).astype('Int8')


# 2. Очистка school_distance: удаление букв 'mi' и приведение к float

schools_df['school_distance'] = (
    schools_df['school_distance']
    .astype(str).str.replace('mi', '', case=False, regex=False).str.strip()
)
schools_df['school_distance'] = pd.to_numeric(
    schools_df['school_distance'], errors='coerce'
).astype('float32')


# 3. Очистка school_grades_clean: приведем к нижнему регистру, отсечем все nan, 
# определим min и max класс, создадим бинарные признаки градаций

# Приведение к нижнему регистру и базовая очистка пропусков
schools_df['school_grades'] = (
    schools_df['school_grades'].astype(str).str.lower()
    .replace(['n', 'na', 'none', 'nan', 'a'], np.nan)
)
# Переводим текстовые маркеры в число 0
schools_df['school_grades'] = (
    schools_df['school_grades'].str.replace(r'pk|k|preschool', '0', regex=True)
)
# Находим ВСЕ последовательности цифр в строке
all_numbers = schools_df['school_grades'].str.findall(r'\d+')
# Вытаскиваем самое маленькое и самое большое число из списка найденных
min_grade = pd.to_numeric(all_numbers.str[0], errors='coerce')
max_grade = pd.to_numeric(all_numbers.str[-1], errors='coerce')
# Выставляем флаги ступеней образования по числовой сетке
schools_df['is_elementary'] = ((min_grade <= 5) & (max_grade >= 0)).astype('int8')
schools_df['is_middle'] = ((min_grade <= 8) & (max_grade >= 6)).astype('int8')
schools_df['is_high'] = ((min_grade <= 12) & (max_grade >= 9)).astype('int8')


# 4. Точечное заполнение флагов по названию на месте пропусков в school_grades

nan_grades_mask = schools_df['school_grades'].isna()
# Переводим в нижний регистр
schools_df['school_name'] = (schools_df['school_name'].astype(str).str.lower())

# Добавляем маску-исключение для профессиональных и учебных центров (это старшая школа)
center_mask = schools_df['school_name'].str.contains(
    'center|training|vocational|career|adult', regex=True
)
# Заполняем флаги
# Старшая школа
high_pattern = r'high|\bjhs\b|center|training|vocational|career|adult'
schools_df.loc[
    nan_grades_mask & schools_df['school_name'].str.contains(high_pattern, regex=True),
    'is_high',
] = 1
# Сохраняем средние школы
middle_pattern = r'middle|junior|\bjhs\b|intermediate|\bis\b|\bintermed\b'
schools_df.loc[
    nan_grades_mask & schools_df['school_name'].str.contains(middle_pattern, regex=True),
    'is_middle',
] = 1
# Размечаем начальные школы
elem_pattern = r'elementary|elem|lower school|\bel\b|(^|\s)ps($|\s|\d)'
schools_df.loc[
    nan_grades_mask & schools_df['school_name'].str.contains(elem_pattern, regex=True),
    'is_elementary',
] = 1


# 5. Создания признака статуса школы

# Инициализируем базовый статус
schools_df['school_status'] = 'public'
# Размечаем частные и религиозные заведения (базовый приоритет для спец-статусов)
private_pattern = (
    r'private|academy|preparatory|\bprep\b|christ|'
    r'christian|catholic|lutheran|parochial|episcopal'
)
schools_df.loc[
    schools_df['school_name'].str.contains(private_pattern, regex=True),'school_status'
] = 'private'
# Размечаем магнитные школы
schools_df.loc[
    schools_df['school_name'].str.contains('magnet', regex=False), 'school_status'
] = 'magnet'
# Размечаем чартерные школы (наивысший приоритет)
# Если школа Charter Academy — это государственная Charter, а не коммерческая Private
schools_df.loc[
    schools_df['school_name'].str.contains('charter', regex=False),'school_status'
] = 'charter'
# Переводим в категориальный тип
schools_df['school_status'] = schools_df['school_status'].astype('category')

Расчет единого минимального расстояния до любого учебного заведения является неэффективным, так как начальные, средние и старшие школы в США представляют собой независимые инфраструктурные объекты. Локация начальных школ не имеет практической ценности для семей со взрослыми детьми, что делает обобщенный показатель неинформативным. Для устранения логического шума целесообразно рассчитать минимальные расстояния до ближайшей школы каждого типа отдельно, используя ранее выделенные бинарные флаги ступеней образования. 

В отношении оценок расчет минимального рейтинга не имеет практического смысла, поскольку наличие слабых школ в округе никого не интересует и не отменяет доступ к сильным. На этапе предобработки целесообразно извлечь максимум информации: зафиксировать максимальный рейтинг как показатель престижа локации, а также рассчитать среднее арифметическое и медиану. Окончательный выбор между средним и медианным значениями будет сделан в разделе исследовательского анализа данных (EDA) на основе анализа распределений и их корреляции с ценой недвижимости.

In [31]:
# Расчет минимальных расстояний до школ по ступеням образования
d_el = (schools_df[schools_df['is_elementary'] == 1]
        .groupby('house_index')['school_distance'].min()
        .rename('dist_elementary'))

d_mid = (schools_df[schools_df['is_middle'] == 1]
         .groupby('house_index')['school_distance'].min()
         .rename('dist_middle'))

d_high = (schools_df[schools_df['is_high'] == 1]
          .groupby('house_index')['school_distance'].min()
          .rename('dist_high'))

# Расчет агрегированных рейтингов школ для каждого объекта
rat = (schools_df.groupby('house_index')['school_rating']
       .agg(['max', 'mean', 'median'])
       .rename(columns={'max': 'school_max_rating',
                        'mean': 'school_mean_rating',
                        'median': 'school_median_rating'}))

# Сборка итоговой матрицы признаков инфраструктуры
schools_agg = pd.concat([d_el, d_mid, d_high, rat], axis=1)

print(f"Размерность матрицы агрегированных признаков школ: {schools_agg.shape}")

Размерность матрицы агрегированных признаков школ: (372851, 6)


In [32]:
# Финальная интеграция данных в основной датасет
# Склеиваем по индексам: индекс data стыкуется с house_index из schools_agg
data = data.join(schools_agg, how='left')

# Удаляем исходную грязную текстовую колонку 'schools'
data.drop(columns=['schools'], inplace=True, errors='ignore')

# Контроль результата
print(f"Текущая размерность основного датасета (строк, колонок): {data.shape}")
display(data[['dist_elementary', 'dist_middle', 'dist_high', 'school_max_rating']].head(5))

Текущая размерность основного датасета (строк, колонок): (377135, 30)


,dist_elementary,dist_middle,dist_high,school_max_rating
0,2.70,3.10,3.10,7
1,1.01,1.01,1.65,4
2,2.06,1.19,2.63,8
3,0.10,1.05,0.81,10
4,3.03,3.03,3.25,5


На этапе предобработки сложная вложенная структура признака schools была успешно десериализована, очищена от текстового шума и преобразована в плоский вид. На основе названий учебных заведений и оригинальной числовой сетки классов восстановлены ступени образования и сформирован категориальный статус школ. Интеграция агрегированных инфраструктурных метрик (расстояний по ступеням обучения и комплексных рейтингов) выполнена непосредственно на текущем этапе путем безопасного слияния по уникальным индексам объектов. Исходный зашумленный JSON-массив удален из выборки, а сформированные признаки полностью готовы к этапу обработки пропусков и разведочного анализа данных (EDA).

### Идентификация наличия бассейна ('private pool', 'PrivatePool')

In [33]:
# Приведем данные к одному формату
data['private pool'] = data['private pool'].astype(str).str.lower()
data['PrivatePool'] = data['PrivatePool'].astype(str).str.lower()

# Посмотрим уникальные значения
display(pd.DataFrame(data[['private pool', 'PrivatePool']].value_counts()))

0
private pool PrivatePool        
nan          nan          332644
             yes           40310
yes          nan            4181

Оба признака содержат маркер yes, сигнализирующий о наличии бассейна, однако их заполнение не совпадает из-за различий в регистре наименований на этапе первичного сбора данных. Для сохранения полноты информации признаки объединены в единую бинарную переменную is_pool, где 1 соответствует наличию бассейна (yes), а пропущенные значения (NaN) интерпретированы как его отсутствие (0). Исходные избыточные столбцы удалены из выборки.

In [34]:
# Создаю новый бинарный признак, удаляя старые избыточные колонки
pool_cols = ['private pool', 'PrivatePool']
data['is_pool'] = data[pool_cols].isin(['yes']).any(axis=1).astype('int8')
data.drop(columns=pool_cols, inplace=True)

print(f'Итоговый размер датасета после объединения бассейнов: {data.shape}')

Итоговый размер датасета после объединения бассейнов: (377135, 29)


### Геометрические параметры строений ('sqft')

In [35]:
# Ищем скрытые символы
print("Уникальные нечисловые символы в колонке sqft:")
print(find_non_numeric_chars(data['sqft']))
# Посмотрим какие фразы использованы
print("Уникальные текстовые паттерны в sqft:")
print(find_text_patterns(data['sqft']))
# Смотрим примеры строк с буквами в sqft
print("\nПримеры сложных строк в sqft:")
data[data['sqft'].astype(str).str.contains(r'[a-zA-Z:]', regex=True)]['sqft'].tail(10)

Уникальные нечисловые символы в колонке sqft:
[' ', ',', '-', ':', 'T', 'a', 'b', 'e', 'f', 'i', 'l', 'n', 'o', 'q', 'r', 's', 't', 'v']
Уникальные текстовые паттерны в sqft:
['Total interior livable area:', 'sqft']

Примеры сложных строк в sqft:


377169                                 1,740 sqft
377171                                 2,022 sqft
377172    Total interior livable area: 1,907 sqft
377173                                 2,505 sqft
377174                                   950 sqft
377175                                 1,792 sqft
377176                                 1,829 sqft
377181                                 2,000 sqft
377182                                 1,152 sqft
377183                                        NaN
Name: sqft, dtype: object

Площадь объекта недвижимости является важнейшим ценообразующим фактором, поэтому стандартное заполнение пропусков средним или медианой может сильно исказить распределение данных.Для решения этой проблемы применим следующую стратегию:
- С помощью регулярных выражений извлечем чистые числовые значения из текстовых строк, отсекая приписки (sqft, Total interior livable area:).
- Объединим явные пропуски (NaN) и логические аномалии (технические заглушки 0 и 1) в единую группу пропущенных данных.
- Спроектируем новый бинарный признак-индикатор is_sqft_unknown (1 — площадь изначально не указана, 0 — площадь известна), а в самом признаке sqft заменим пропуски и аномалии на 0. Это позволит модели машинного обучения корректно интерпретировать отсутствие информации без внесения искусственного шума в непрерывную величину, если мы далее будем заполнять пропуски математическими значениями.

In [36]:
# Сначала полностью удаляем запятые и пробелы (как разделители разрядов)
cleaned_sqft = data['sqft'].astype(str).str.replace(r'[,\s]', '', regex=True)

# Извлекаем первое встреченное число целиком
cleaned_sqft_series = cleaned_sqft.str.extract(r'(\d+)')[0]

# Переводим во временный числовой формат float
cleaned_sqft_numeric = pd.to_numeric(cleaned_sqft_series, errors='coerce')

# Создаем бинарный флаг: 1 ставим там, где изначально был NaN, либо получился 0, 
# либо аномальная заглушка 1
data['is_sqft_unknown'] = (
    (cleaned_sqft_numeric.isnull()) | 
    (cleaned_sqft_numeric == 0) | 
    (cleaned_sqft_numeric == 1)
).astype('int8')

# Записываем очищенную площадь обратно
data['sqft'] = cleaned_sqft_numeric.fillna(0).astype(int)
data.loc[data['is_sqft_unknown'] == 1, 'sqft'] = 0

# Проверяем, что получилось в итоге
print(f"Распределение в новом признаке is_sqft_unknown:"
      f"\n{data['is_sqft_unknown'].value_counts()}")
print(f"\nДиапазон площади: {data['sqft'].min()} - {data['sqft'].max()} sqft")

Распределение в новом признаке is_sqft_unknown:
0    323834
1     53301
Name: is_sqft_unknown, dtype: int64

Диапазон площади: 0 - 795979430 sqft


Успешное приведение признака к числовому типу позволило нам зафиксировать чистые параметры площади для 85% объектов выборки, изолировав неизвестные и аномальные значения с помощью специального флага. Теперь, имея в распоряжении точную площадь и ранее очищенную удельную стоимость, мы полностью готовы перейти к обработке целевой переменной target и реализовать алгоритм восстановления пропущенных цен.

### Исследование целевой переменной ('target')

Теперь перейдем к самому важному признаку - целевому 'target'. Необходимо определить, какие именно символы присутствуют в строковых значениях помимо числовых данных, очистить данные и перевести в числовой формат. Для определения "лишних" символов использую свою функцию `find_non_numeric_chars()`

In [37]:
# Ищем скрытые символы
print("Уникальные нечисловые символы в колонке target:")
print(find_non_numeric_chars(data['target']))
# Ищем скрытые фразы
print("\nУникальные текстовые паттерны в target:")
print(find_text_patterns(data['target']))
# Смотрим примеры строк с буквами в target
print("\nПримеры сложных строк в target:")
data[data['target'].astype(str).str.contains(r'[a-zA-Z/\-+]', regex=True)]['target'].tail(5)

Уникальные нечисловые символы в колонке target:
[' ', '$', '+', ',', '-', '/', 'm', 'o']

Уникальные текстовые паттерны в target:
['mo']

Примеры сложных строк в target:


376966    $234,990+
376972    $231,100+
376976    $1,900/mo
377002    $433,500+
377120          NaN
Name: target, dtype: object

Перебирая срезы данных с помощью регулярных выражений, мы можем зафиксировать несколько критически важных паттернов ценообразования и скрытых аномалий:

- **Формат записи**: Большое количество цен записано в стандартном американском текстовом формате `$nnn,nnn`. Для обучения модели необходимо очистить эти строки от знаков валюты, запятых-разделителей и привести к числовому типу данных `float`.
- **Наличие символа `+` (Начальная цена)**: В риелторской практике США знак плюс после стоимости (например, `$182,490+`) означает, что указана базовая стартовая цена объекта (чаще всего это касается строящихся новостроек). Чтобы не потерять этот ценный рыночный сигнал, мы создадим бинарный флаг `is_start_price` (1 — цена стартовая, 0 — фиксированная), после чего удалим символ из основной колонки.
- **Обнаружение скрытой аренды (`/mo`)**: Указание `/mo` (month) означает, что в датасет попали объявления о сдаче жилья в аренду, а не о его продаже. К этой же категории скрытого шума относятся объекты с нереально низкой фиксированной ценой (например, $1,000 которые мы видели ранее), которые фактически представляют собой стартовую ставку аукциона или скрытую стоимость аренды. Смешивание таких объявлений с реальными продажами недопустимо — порядки цифр различаются в сотни раз, что приведет к полной деградации предсказательной способности алгоритма. Все строки с маркером аренды и аномально низким порогом стоимости подлежат полному удалению из выборки.
- **Восстановление пропущенных данных**: Наличие в датасете очищенного признака площади `sqft` и удельной стоимости `Price/sqft` открывает возможность математически восстановить пропуски в целевой переменной по формуле $target = sqft \times Price/sqft$. Это позволяет вернуть ценные наблюдения в выборку вместо их прямого удаления на этапе очистки.

In [38]:
# Приводим к строке и нижнему регистру для надежности поиска
data['target'] = data['target'].astype(str).str.lower().str.strip()

# Удаляем всю аренду (Строки с '/mo' полностью выкидываем из датасета)
is_rental = data['target'].str.contains('/mo', na=False)
data = data[~is_rental].copy()
rental_count = is_rental.sum()

# Фиксируем стартовую цену: создаем флаг для знака '+'
data['is_start_price'] = data['target'].str.contains(r'\+', regex=True, na=False).astype('int8')

# Очищаем цену: удаляем знаки $, запятые, плюсы, буквы и пробелы
data['target'] = data['target'].str.replace(r'[^\d.]', '', regex=True)
# Переводим в числовой формат float
data['target'] = pd.to_numeric(data['target'], errors='coerce')

# Фиксируем строки с аномально низкой ценой (аукционы и скрытая аренда)
# Пропуски (NaN) не трогаем, они пойдут на восстановление ниже
is_cheap_auction = (data['target'] <= 1000) & data['target'].notna()
auction_count = is_cheap_auction.sum()
data = data[~is_cheap_auction].copy()

# Восстанавливаем пропуски
# Находим строки, где таргет пропущен (NaN), но есть И площадь, И цена за квадратный фут
can_recover_target = (
    data['target'].isna() & 
    (data['sqft'] > 0) & 
    data['Price/sqft'].notna()
)
# Считаем количество пропусков ДО восстановления
target_nan_before = data['target'].isna().sum()
# Рассчитываем стоимость и точечно заполняем пропуски
recovered_targets = data.loc[can_recover_target, 'sqft'] * data.loc[can_recover_target, 'Price/sqft']
data['target'] = data['target'].combine_first(recovered_targets)

# Выводим итоговый отчет на экран
print(f"Успешно восстановлено пропусков по формуле:"
      f" {target_nan_before - data['target'].isna().sum()} строк")
print(f"Количество удаленных строк из категории 'Аренда' (/mo): {rental_count} строк")
print(f"Количество удаленных строк из категории 'Аукцион/Дешевый шум' (<= $1000): {auction_count} строк")

Успешно восстановлено пропусков по формуле: 292 строк
Количество удаленных строк из категории 'Аренда' (/mo): 398 строк
Количество удаленных строк из категории 'Аукцион/Дешевый шум' (<= $1000): 1042 строк


Очистка целевой переменной прошла успешно, аренда удалена, стартовая цена зафиксирована, пропуски частично заполнены, признак переведен в числовой формат.

### Количественные параметры жилой площади ('beds')

Посмотрим какие текстовые паттерны встречаются в тексте признака 'beds':

In [39]:
# Посмотрим какие фразы использованы
print("Уникальные текстовые паттерны в beds:")
text_patterns = find_text_patterns(data['beds'])
pprint(text_patterns, width=120, compact=True)

# Смотрим на топ-10 тегов признака beds
print('-'*15)
print(data['beds'].astype(str).str.lower().str.strip().value_counts().head(10))

Уникальные текстовые паттерны в beds:
['Based on Redfin', 'Bath', 'Baths', 'Bedrooms', 'Beds', 'Cable TV Available', 'Dining Room', 'Eat', 'In Kitchen',
 'Living Room', 'Oven', 'Range', 'Refrigerator', 'acre', 'acres', 'bd', 'less than its current list price',
 'more than its current list price', 'or More Bedrooms', 's Raleigh data', 's St Johns data', 's value is', 'sqft',
 'st Floor', 'we estimate the home', 'which is']
---------------
nan       91093
3 beds    52847
4 beds    35171
3         31388
2 beds    26096
4         20019
2         16105
baths     15277
3 bd      12875
5 beds    11236
Name: beds, dtype: int64


In [40]:
# Проверяем строки, где в beds записана площадь (содержит sqft)
sqft_mask = data['beds'].astype(str).str.contains('sqft', na=False)
sqft_in_beds = data[sqft_mask]
print("Примеры строк, где площадь улетела в beds:")
display(sqft_in_beds[['beds', 'sqft', 'is_sqft_unknown']][5:10])

# Проверяем строки, где в beds записан размер участка (acre, acres)
acre_mask = data['beds'].astype(str).str.contains('acre|acres', na=False)
acres_in_beds = data[acre_mask]
print("\nПримеры строк, где площадь участка улетела в beds:")
display(acres_in_beds[['beds', 'sqft', 'is_sqft_unknown']][:5])

# Проверяем строки, где в beds записаны ванные комнаты (содержит Bath)
baths_mask = data['beds'].astype(str).str.contains('Bath', na=False)
baths_in_beds = data[baths_mask]
print("\nПримеры строк, где ванные улетели в beds:")
display(baths_in_beds[['beds', 'baths']][5:10])

Примеры строк, где площадь улетела в beds:


,beds,sqft,is_sqft_unknown
2451,"2,200 sqft",0,1
2701,"2,874 sqft",0,1
3431,"2,178 sqft",0,1
3434,"6,351 sqft",1076,0
3536,"4,356 sqft",0,1



Примеры строк, где площадь участка улетела в beds:


,beds,sqft,is_sqft_unknown
279,0.25 acres,0,1
491,0.44 acres,0,1
548,1.43 acres,0,1
734,0.32 acres,0,1
1205,9.7 acres,0,1



Примеры строк, где ванные улетели в beds:


,beds,baths
154,Baths,~
182,Baths,"4,000"
207,Baths,"2,000"
210,Baths,"3,000"
220,Bath,NaN


In [41]:
# Смотрим всю строку 3434 целиком, чтобы понять природу объекта
display(data.loc[[3434]])

,status,propertyType,street,baths,fireplace,city,sqft,zipcode,beds,state,...,parking_clean,dist_elementary,dist_middle,dist_high,school_max_rating,school_mean_rating,school_median_rating,is_pool,is_sqft_unknown,is_start_price
3434,Auction,SingleFamilyResidence,4665 14th Ave S,-- baths,NaN,Saint Petersburg,1076,33711,"6,351 sqft",FL,...,NaN,0.6,3.8,0.9,4,2.666667,2.0,0,0,0


Анализ признака количества спален (beds):

В колонке beds обнаружен структурный сдвиг данных. Вместо количества комнат здесь содержатся физическая площадь (sqft, acre), ванные комнаты (bath), бытовая техника и технические метаданные информационных платформ агрегаторов. 

Для сохранения скрытых рыночных сигналов в текущем разделе инициализируются новые бинарные признаки: 'has_appliances' (наличие техники) и 'has_extra_rooms' (дополнительные планировки), которые будут наполняться в ходе сквозной очистки. Исходное зашумленное поле 'beds' сохраняется в качестве источника данных.

Алгоритм пошаговой рокировки данных:
- Земельные участки (acre / acres): Числа извлекаются, переводятся в квадратные футы (1 акр = 43 560 кв. футов) и точечно заполняют пропуски в lotsize.
- Площадь здания (sqft): Извлекается числовое значение. Если родная площадь в sqft равна 0, данные переносятся туда со сбросом флага 'is_sqft_unknown' = 0. Если родная площадь уже заполнена (> 0), значение переносится в признак площади участка ('lotsize').
- Ванные комнаты (bath / baths): Факт упоминания фиксируется в новом флаге 'has_baths_mention', если родное поле 'baths' было пустым. Это необходимо, чтобы сохранить ценный рыночный сигнал о наличии удобств в доме и не потерять его при фильтрации текстового шума в признаке спален.
- Количество спален: Строки без информации о комнатах (логи агрегаторов) переводятся в NaN. Из остальных строк регулярными выражениями извлекаются чистые числа для 'beds_clean'. Все аномальные нули и явные опечатки также переводятся в NaN.

In [42]:
# Приводим к строке и нижнему регистру для создания точных масок
beds_raw = data["beds"].astype(str).str.lower().str.strip()
baths_raw = data["baths"].astype(str).str.lower().str.strip()

# Фиксируем факт упоминания ванных комнат, если родное поле 'baths' пустое
baths_mask = beds_raw.str.contains("bath", na=False)

# Учитываем как системные NaN, так и любые варианты пустых текстовых заглушек
baths_is_empty = data["baths"].isnull() | baths_raw.isin(
    ["", "nan", "none", "no data"]
)
data["has_baths_mention"] = (baths_mask & baths_is_empty).astype("int8")

print(f"Количество успешно восстановленных оценок ванных комнат: {data['has_baths_mention'].sum()}")

# Обработка земельных участков (acre / acres) -> перенос в lotsize
acre_mask = beds_raw.str.contains("acre", na=False)
extracted_acres = (
    data.loc[acre_mask, "beds"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"([\d.]+)")
    .squeeze()
)
extracted_acres = pd.to_numeric(extracted_acres, errors="coerce")

# Точечно дополняем lotsize там, где был NaN (1 акр = 43560 кв. футов)
data.loc[acre_mask, "lotsize"] = data.loc[acre_mask, "lotsize"].combine_first(
    extracted_acres * 43560
)

# Обработка площадей здания (sqft)
sqft_mask = beds_raw.str.contains("sqft", na=False)
extracted_sqft = (
    data.loc[sqft_mask, "beds"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
    .squeeze()
)
extracted_sqft = pd.to_numeric(extracted_sqft, errors="coerce")

# Если родная площадь равна 0 ИЛИ является NaN — восстанавливаем её
home_sqft_mask = sqft_mask & ((data["sqft"] == 0) | data["sqft"].isnull())
data.loc[home_sqft_mask, "sqft"] = extracted_sqft
data.loc[home_sqft_mask, "is_sqft_unknown"] = 0

# Если родная площадь уже строго заполнена (> 0) — уносим значение в lotsize
land_sqft_mask = sqft_mask & (data["sqft"] > 0)
data.loc[land_sqft_mask, "lotsize"] = data.loc[
    land_sqft_mask, "lotsize"
].combine_first(extracted_sqft)

# Выводим отчет
print(f"Успешно перенесено земельных участков (acres): {acre_mask.sum()} строк")
print(f"Восстановлено жилой площади дома (sqft -> sqft): {home_sqft_mask.sum()} строк")
print(f"Дополнено площадей участков из описаний (sqft -> lotsize): {land_sqft_mask.sum()} строк")

Количество успешно восстановленных оценок ванных комнат: 527
Успешно перенесено земельных участков (acres): 1639 строк
Восстановлено жилой площади дома (sqft -> sqft): 1423 строк
Дополнено площадей участков из описаний (sqft -> lotsize): 1329 строк


In [43]:
# Проведем диагностику данных, надо понять на сколько безопасно извлекать просто числа
# Фиксируем стартовые списки слов для мебели и комнат
appliances_regex = r"oven|range|refrigerator|cable|available|tv"
rooms_regex = r"kitchen|living|dining|eat|room|floor"
aggregator_regex = r"redfin|estimate|value|price|data|which|less|more"
noise_pattern = appliances_regex + r"|" + rooms_regex + r"|" + aggregator_regex
has_noise_word = beds_raw.str.contains(noise_pattern, regex=True, na=False)

# Формируем правила: что является стандартной записью спален, а что — чистым числом
is_text_bed = beds_raw.str.match(r"^\d+\s*(?:beds?|bd|bs)\.?$", na=False)
is_pure_number = pd.to_numeric(beds_raw, errors="coerce").notnull()

# Находим строки с ванными комнатными, чтобы принудительно убрать их из этого вывода
baths_mask = beds_raw.str.contains("bath", na=False)

# Выделяем аномалии: мусор есть, это НЕ спальни, НЕ чистые числа И НЕ строки с ванными
text_anomalies = data[
    has_noise_word & ~is_text_bed & ~is_pure_number & ~baths_mask
]
print("Примеры строк, где цифры не означают спальню:")
display(text_anomalies["beds"].head(10))

Примеры строк, где цифры не означают спальню:


126384    Based on Redfin's St Johns data, we estimate t...
200331    3 or More Bedrooms, Dining Room, Living Room, ...
267024    Based on Redfin's Raleigh data, we estimate th...
296657                                   3 or More Bedrooms
364340                                 # Bedrooms 1st Floor
Name: beds, dtype: object

Предполагаю, что символ `#` (строка с индексом 296657) прописан случайно нажатием клавиши Shift, и фактически это 3 спальни. Объекты с индексами 126384 и 267024 переводятся в NaN, так как они представляют собой логи парсера и не содержат данных о количестве спален. Данные аномалии отработаю вручную.

In [44]:
# Инициализируем бинарные признаки комфорта
data["has_appliances"] = np.zeros(len(data), dtype="int8")
data["has_extra_rooms"] = np.zeros(len(data), dtype="int8")

# Фиксирую глобальные переменные для сквозного поиска техники и доп. комнат в цикле
appliances_regex = r"oven|range|refrigerator|cable|available|tv"
rooms_regex = r"kitchen|living|dining|eat|room|floor"

# Выделение мусорных строк, не содержащих информацию о спальнях
beds_raw = data["beds"].astype(str).str.lower().str.strip()
not_a_bedroom_mask = beds_raw.str.contains(
    "bath|sqft|acre|redfin|cable|room|kitchen|oven|range|refrigerator|"
    "price|value|estimate|data|which|floor",
    na=False,
)
is_empty_dash = beds_raw.str.contains("--", na=False)

# Извлечение чисел, включая дробные форматы записи
bed_numbers = (
    data.loc[~(not_a_bedroom_mask | is_empty_dash), "beds"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")
    .squeeze()
)
beds_vector = pd.to_numeric(bed_numbers, errors="coerce")

# Принудительное зануление ложных нулей
beds_vector.loc[beds_vector == 0] = np.nan

# Исправление опечатки клавиатуры в строке с этажностью
if 296657 in beds_vector.index:
    beds_vector.loc[296657] = 3.0
    data.loc[296657, "stories"] = "1"

# Фиксация итогового числового признака количества спален
data["beds_clean"] = beds_vector.astype("Int16")

# Вывод результатов для проверки структуры данных
print(data["beds_clean"].value_counts(dropna=False).head(10))

<NA>    113095
3       105197
4        68683
2        49873
5        21664
6         6511
1         6120
7         1820
8         1219
9          486
Name: beds_clean, dtype: Int64


Вывод по очистке признака 'beds':

1. **Рокировка данных**: Устранен структурный сдвиг данных. Физическая площадь зданий перенесена в признак 'sqft' (в 1423 строках сброшен флаг 'is_sqft_unknown'), а площади участков конвертированы в квадратные футы и интегрированы в признак 'lotsize'.
2. **Инициализация признаков комфорта**: Подготовлены бинарные индикаторы 'has_appliances' и 'has_extra_rooms' для сквозного сбора скрытых рыночных сигналов о бытовой технике и планировке. Успешно сформирован флаг 'has_baths_mention' для фиксации упоминаний санузлов в текстовых аномалиях.
3. **Коррекция опечаток**: Текстовые логи платформы Redfin переведены в категорию пропусков. Обработана техническая ошибка ввода в строке с этажностью (индекс 296657): восстановлено истинное значение (3 спальни) и скорректирован параметр этажности ('stories = 1').
4. **Фильтрация и пропуски**: Количество пропущенных значений зафиксировано на уровне 113 095 строк (тип 'Int16'). Логика фильтрации:
    - Строки с упоминанием ванных комнат, площадей, метаданных агрегаторов, а также технические прочерки '--' отсечены с помощью регулярных выражений.
    - Значения '0' и '0.0' принудительно переведены в 'NaN' во избежание смещения предсказаний модели.
    - Значения > 10 на незастроенных земельных участках (где 'sqft' равен 0 или пропущен) идентифицированы как номера строительных лотов и переведены в пропуски.
    - Исходная колонка сохранена в качестве источника данных для последующего сбора бинарных флагов.

### Количественные параметры санитарных узлов ('baths')

In [45]:
# Посмотрим какие фразы использованы в ванных комнатах
print("Уникальные текстовые паттерны в baths:")
text_patterns_baths = find_text_patterns(data['baths'])
pprint(text_patterns_baths, width=120, compact=True)
print('-'*15)
print("Топ-10 сырых значений признака baths:")
print(data['baths'].astype(str).str.lower().str.strip().value_counts().head(10))

Уникальные текстовые паттерны в baths:
['Bathrooms:', 'Bathrooms: SemiMod', 'Baths', 'Ft', 'Sq', 'ba', 'baths']
---------------
Топ-10 сырых значений признака baths:
nan             105507
2 baths          52082
3 baths          35454
2                20451
2.0              16556
4 baths          14748
3.0              10863
3                10113
bathrooms: 2      9534
2.5               8113
Name: baths, dtype: int64


In [46]:
# Смотрим, что скрывается за паттернами Sq и Ft в ванных комнатах
display(data[data['baths'].astype(str).str.contains('Sq|Ft', case=False, na=False)]['baths'].value_counts())

Sq. Ft.     202
Name: baths, dtype: int64

Аудит текстовых паттернов в поле `baths` показал отсутствие критического структурного сдвига данных. Строки с метками `Sq. Ft.` являются пустыми заголовками агрегаторов, не содержат чисел и подлежат переводу в категорию пропущенных значений (NaN). Из строк извлекаются целочисленные и дробные значения санузлов, характерные для рынка недвижимости США. Данные конвертируются в формат `float32` с сохранением исходной структуры пропущенных значений.

In [47]:
def clean_baths(series: pd.Series) -> pd.Series:
    """Очищает признак ванных комнат от текстового шума и приводит к типу float32.

    Args:
        series (pd.Series): Исходный зашумленный столбец с данными о ванных.

    Returns:
        pd.Series: Очищенный числовой столбец типа float32 с сохраненными NaN.
    """
    # Приведение к строке, нижнему регистру и удаление крайних пробелов
    cleaned = series.astype(str).str.lower().str.strip()
    # Очистка системного мусора "sq. ft.", не несущего информации о ванных
    cleaned = cleaned.mask(cleaned.isin(["sq. ft.", "sq", "ft", "nan", "", "none"]))
    # Извлечение численных значений (целых или десятичных дробей)
    extracted = cleaned.str.extract(r"(\d+(?:\.\d+)?)")[0]
    # Принудительная конвертация в float32
    numeric_series = pd.to_numeric(extracted, errors="coerce").astype("float32")
    # Возвращаем NaN на места, где изначально были пропуски в исходной серии
    numeric_series[series.isna()] = np.nan
    return numeric_series


# Создаем чистый признак baths_clean
data["baths_clean"] = clean_baths(data["baths"])

print("Финальная структура признака baths_clean после очистки:")
print(data["baths_clean"].value_counts(dropna=False).head(10))

Финальная структура признака baths_clean после очистки:
NaN    107003
2.0    106404
3.0     67276
4.0     26316
1.0     22906
2.5     12832
5.0      9451
3.5      5469
6.0      4301
0.0      3915
Name: baths_clean, dtype: int64


Текстовые маркеры и пустые съехавшие заголовки парсера 'Sq. Ft.' успешно удалены. Извлечены чистые числовые значения с сохранением специфической для США дробной структуры санузлов (например, '2.5' и '3.5'). Признак успешно приведен к итоговому типу 'float32'. 

Осталось занести данные сюда из спален и дозаполнить флаговый признак 'has_baths_mention'.

In [48]:
# Паттерн ищет число, стоящее перед словом bath, или слово bath, после которого идет число
baths_regex_donor = r"(?:(\d+(?:\.\d+)?)\s*bath)|(?:bath\w*\s*[:\s]*(\d+(?:\.\d+)?))"
# Извлекаем сырой текст спален и ищем в нем целевые группы цифр для ванных
beds_raw_lower = data["beds"].astype(str).str.lower().str.strip()
extracted_baths_from_beds = beds_raw_lower.str.extract(baths_regex_donor)
# Объединяем две возможные группы захвата (число до или число после слова bath)
baths_donor_series = extracted_baths_from_beds[0].combine_first(extracted_baths_from_beds[1])
# Принудительно конвертируем полученный донорский вектор в float32
baths_donor_numeric = pd.to_numeric(baths_donor_series, errors="coerce").astype("float32")
# Запоминаем количество пропусков до операции перекрестного обогащения
baths_before = data["baths_clean"].isna().sum()
# Дополняем только пустые ячейки родного признака ванных комнат стык-в-стык
data["baths_clean"] = data["baths_clean"].combine_first(baths_donor_numeric)
# Выводим строгий технический отчет о спасенных числовых значениях санузлов
print(f"Успешно восстановлено точных числовых значений ванных из beds: +{baths_before - data['baths_clean'].isna().sum()}")

Успешно восстановлено точных числовых значений ванных из beds: +6


Не густо, но лучше чем ничего.

In [49]:
# Дозаполняем флаг единицами там, где в очищенных ванных теперь НЕ NaN
baths_not_nan = (data["baths_clean"] > 0).astype("int8")
data["has_baths_mention"] = data["has_baths_mention"] | baths_not_nan

# Контроль результата
print("Распределение флага 'Ванна физически есть в объекте':")
print(data["has_baths_mention"].value_counts())

Распределение флага 'Ванна физически есть в объекте':
1    265304
0    110391
Name: has_baths_mention, dtype: int64


### Классификация типов недвижимости ('propertyType')

In [50]:
# Пример данных в 'propertyType'
print("Пример текстовых паттернов в 'propertyType':\n")
property_patterns = find_text_patterns(data['propertyType'])
pprint(random.sample(property_patterns, 20), width=120, compact=True)
print("\nТоп-10 сырых значений propertyType:\n")
print(data['propertyType'].astype(str).str.lower().str.strip().value_counts().head(10))

Пример текстовых паттернов в 'propertyType':

['Coastal Modern', 'to', 'Row Home', 'Mediterranean', 'Georgian', 'Cottage', 'Traditional', 'Farmhouse', 'Historic',
 'Lower Level', 'Multi Generational', 'Units', 'Loft', 'Mid', 'Georgian Revival', 'Attached or', 'Quad Level', 'Square',
 'Expanded Ranch', 'Multi Family']

Топ-10 сырых значений propertyType:

single-family home               91118
single family                    62818
condo                            42521
nan                              34699
single family home               31727
lot/land                         20453
townhouse                        18345
land                             10927
multi-family                      7788
condo/townhome/row home/co-op     7701
Name: propertyType, dtype: int64


Анализ выявил сильный структурный сдвиг данных. Вместо простого типа недвижимости риелторы вписали сюда этажность, информацию о земле, ремонте, архитектурных стилях и выходе к воде. Простое укрупнение текста сотрет эти ценные рыночные сигналы.

Сформирован пошаговый план извлечения скрытых факторов во временные переменные перед финальной стандартизацией и укрупнением категорий самого признака:

1. **Тип жилья и формат владения**: Основа признака. Группируем объекты в сильные рыночные классы: частные дома ('single_family'), квартиры ('condo_coop_apartment'), таунхаусы ('townhouse'), дома на несколько семей ('multi_family'), участки ('land_or_lot'), коммерческие объекты ('commercial') и мобильные/модульные дома ('mobile_or_manufactured').
2. **Этажность ('stories')**: Выделяем упоминания этажей ('one story', 'two story', 'high rise'), чтобы позже точечно заполнить пропуски в родной колонке 'stories'.
3. **Размер участка ('lotsize')**: Изолируем строки с площадью в акрах ('acre') для пересчета в кв. футы и обогащения пропущенных данных в 'lotsize'.
4. **Стиль, экзотика и премиальность**: Фиксируем сильные ценообразующие маркеры: историческую ценность ('historic', 'antique'), элитный статус ('villa', 'manor'), современный дизайн ('modern', 'contemporary') и уникальное расположение у воды или необычный конструктив ('coastal', 'lake house', 'stilt house').

После выделения скрытых слоев, геометрические данные (этажность и земля) пойдут на заполнение NaN в родных колонках 'stories' и 'lotsize', а маркеры стиля и экзотики будут зафиксированы в новых независимых бинарных флагах. Сам признак 'propertyType' после этого будет очищен и приведен к основным 7 классам.

In [51]:
# Для начала проверим акры, есть ли площадь, которая может быть извлечена
# Фильтруем строки по наличию ключевого слова 'acre'
acre_mask = data["propertyType"].astype(str).str.contains("acre", case=False, na=False)
# Выводим распределение уникальных значений
acre_counts = data.loc[acre_mask, "propertyType"].value_counts()
print(acre_counts)

Residential (<1 Acre)    95
Residential (1+ Acre)    20
Name: propertyType, dtype: int64


В поле обнаружены маркеры 'Residential (<1 Acre)' и 'Residential (1+ Acre)'. Чтобы не потерять эти данные при стандартизации признака, мы создаем новую категориальную колонку 'land_scale'. Расчет масштаба выполняется по базовой колонке 'lotsize' с разделением объектов на две категории: 'under_acre' (меньше 1 акра или 43 560 кв. футов) и 'over_acre' (больше 1 акра). Извлеченные текстовые маркеры из 'propertyType' точечно дополняют этот признак, а во всех остальных случаях сохраняются честные 'NaN'.

In [52]:
# Приводим к нижнему регистру для точного поиска текстовых совпадений
prop_lower = data['propertyType'].astype(str).str.lower()

# Выделяем маски для обнаруженных текстовых маркеров по площади
is_under_acre_text = prop_lower.str.contains(r'<1\s*acre', regex=True, na=False)
is_over_acre_text = prop_lower.str.contains(r'1\+\s*acre', regex=True, na=False)

# Инициализируем новую колонку как object для корректной записи текста
data['land_scale'] = pd.Series(np.nan, index=data.index, dtype='object')

# Размечаем масштаб только на основе известных физических размеров
data.loc[data['lotsize'] < 43560, 'land_scale'] = 'under_acre'
data.loc[data['lotsize'] >= 43560, 'land_scale'] = 'over_acre'

# Заполняем пропуски на основе извлеченных текстовых маркеров с защитой чисел
data.loc[data['land_scale'].isna() & is_under_acre_text, 'land_scale'] = 'under_acre'
data.loc[data['land_scale'].isna() & is_over_acre_text, 'land_scale'] = 'over_acre'

# Оптимизируем память, переводя признак в категориальный тип
data['land_scale'] = data['land_scale'].astype('category')

# Смотрим результат
print("Распределение значений в новом признаке land_scale:")
print(data['land_scale'].value_counts(dropna=False))

Распределение значений в новом признаке land_scale:
under_acre    252108
NaN            92400
over_acre      31187
Name: land_scale, dtype: int64


В сырых текстовых описаниях содержатся сильные ценообразующие маркеры качества и уникальности объектов. Чтобы сохранить эти рыночные сигналы без взаимного затирания, мы фиксируем регулярные выражения для независимых бинарных флагов стиля и уникальности (`is_historic`, `is_modern`, `is_luxury`, `is_waterfront_exotic`). 

Аналогично логике техники и доп. помещений, точечное извлечение этих архитектурных маркеров в текущем шаге сделает признаки полупустыми, так как риелторы разбросали слова вроде `custom`, `modern` или `lake` и по другим сырым колонкам датасета. Чтобы максимизировать плотность ценных рыночных сигналов, мы сохраняем оригинальное поле `propertyType` нетронутым. Финальный сквозной Feature Engineering архитектурных стилей, премиального статуса и экзотического конструктива будет запущен единым поиском по регулярным выражениям в самом конце очистки по всем сохраненным сырым массивам датасета разом через побитовое ИЛИ.

In [53]:
# Создаю пока пустые признаки, обновлю значения в конце очистки все признаков
data['is_historic'] = np.zeros(len(data), dtype='int8')
data['is_modern'] = np.zeros(len(data), dtype='int8')
data['is_luxury'] = np.zeros(len(data), dtype='int8')
data['is_waterfront_exotic'] = np.zeros(len(data), dtype='int8')

# Фиксируем регулярные выражения стиля и уникальности для финального цикла
historic_regex = (
    r"historic|historical|antique|century|federal|georgian|victorian|"
    r"queen[^a-z]+anne|colonial|tudor|williamsburg|early[^a-z]+american"
)
modern_regex = r"modern|contemporary|art[^a-z]+deco|mid[^a-z]+century|modernist"
luxury_regex = r"villa|manor|chalet|penthouse|resort|custom|vacation"

# Короткие слова защищены проверкой отсутствия латинских букв по краям
exotic_regex = (
    r"coastal|beach|lake|houseboat|moorage|stilt|yurt|cabin|(?<![a-z])camp(?!"
    r"[a-z])|lodge|log[^a-z]+home|(?<![a-z])log(?![a-z])|(?<![a-z])dome(?!"
    r"[a-z])|underground|bermuda|florida|key[^a-z]+west"
)

Бинарные признаки стилей и уникальности успешно инициализированы, а регулярные выражения для них зафиксированы на будущее. 

Перейдем к следующему этапу — стандартизации самих классов недвижимости. Результат этой обработки будет записываться в новый очищенный признак `property_type_clean`, при этом оригинальная колонка `propertyType` сохранится в датасете до финала. Все данные самостоятельно приводятся к 7 основным рыночным классам а также вспомогательной категории 'other' для неклассифицируемых остатков распределения. 

In [54]:
# Словарь ключевых слов (порядок определяет приоритет проверки)
property_mapping = {
    'multi_family': ['multi-family', 'multi family', 'duplex', 'triplex', 'fourplex', 'multiplex'],
    'townhouse': ['townhouse', 'townhome', 'row home', 'attached'],
    'apartment': ['condo', 'condominium', 'coop', 'cooperative', 'apartment', 'studio', 'penthouse', 'op'],
    'house': [
        'single-family', 'single family', 'detached', 'bungalow', 'cottage', 'traditional', 
        'ranch', 'colonial', 'contemporary', 'transitional', 'florida', 'farm', 'cape cod', 
        'spanish', 'mediterranean', 'craftsman', 'singlefamilyresidence', 'custom', 'home'
    ],
    'land': ['land', 'lot'],
    'commercial': ['commercial', 'warehouse', 'store', 'industrial'],
    'mobile': ['mobile', 'manufactured', 'modular', 'prefab', 'wide', 'mfd', 'yurt']
}


def clean_property_type(val):
    """Стандартизирует тип недвижимости на основе правил учебного брифа.

    Args:
        val (str, float, int, None): Исходное значение из столбца propertyType.

    Returns:
        str: Одна из 8 укрупненных категорий недвижимости:
            ['house', 'apartment', 'townhouse', 'land', 'multi_family',
             'commercial', 'mobile', 'other'].
        np.nan: Для чистых пропусков данных и неизвестных заглушек.
    """
    val_str = str(val).lower().strip()
    if val_str in ['nan', 'unknown', 'see remarks', 'yes', '']:
        return np.nan
    
    # Пробегаем по основному словарю классов
    for category, words in property_mapping.items():
        if any(w in val_str for w in words):
            return category

    return 'other'

# Создаем новый признак
data['property_type_clean'] = data['propertyType'].apply(clean_property_type).astype('category')

# Проверяем результат
print("Итоговое распределение категорий в property_type_clean:")
print(data['property_type_clean'].value_counts(dropna=False))

Итоговое распределение категорий в property_type_clean:
house           211126
apartment        47903
NaN              34864
land             31389
townhouse        27113
multi_family     12385
other             8183
mobile            2702
commercial          30
Name: property_type_clean, dtype: int64


Исходное зашумленное поле приведено к 8 чистым рыночным классам жилья. Все редкие архитектурные стили, занимающие мизерную долю рынка, укрупнены в класс 'other', а пропущенные значения сохранены как честные NaN.

### Этажность зданий ('stories')

In [55]:
# Пример данных в 'stories'
print("Пример текстовых паттернов в 'stories':\n")
stories_patterns = find_text_patterns(data['stories'])
pprint(random.sample(stories_patterns, 20), width=120, compact=True)
print("\nТоп-10 сырых значений stories:\n")
print(data['stories'].astype(str).str.lower().str.strip().value_counts().head(10))

Пример текстовых паттернов в 'stories':

['Level', 'Apartments', 'Bungalow', 'Federal', 'Loft', 'Commercial', 'None', 'Split Plan', 'Cottage', 'Traditional',
 'Unimproved Commercial', 'Contemporary', 'Condo', 'Lot', 'Colonial', 'Multi Level', 'Sixplex', 'Double Wide', 'Chalet',
 'T']

Топ-10 сырых значений stories:

nan    149934
1.0     67073
2.0     55062
1       23031
2       18103
3.0     11271
0.0      7240
one      5758
0        4271
3        4228
Name: stories, dtype: int64


Признак этажности переводится в категориальный формат с укрупнением значений в логические интервалы, характерные для рынка недвижимости США. Индивидуальные числовые параметры (1.0, 1.5, 2.0, 3.0) сохраняются для объектов индивидуального жилого фонда. Среднеэтажные и высотные строения группируются в интервалы '4_to_7' (включая текстовый маркер 'Mid-Rise') и '8_and_more' (включая маркеры 'High-Rise' и 'Total Floors'). Данный подход предотвращает искажение распределения при оценке квартир в многоквартирных комплексах. Строки, не содержащие явных параметров этажности, переводятся в категорию пропущенных значений (NaN).

In [56]:
# Единый справочник для поиска текстовых маркеров с защитой коротких слов границами \b
text_rules = {
    "8_and_more": ["high-rise", "high rise", "total floors"],
    "4_to_7": ["mid-rise", "mid rise"],
    "3.0": [
        "three or more", "threeormore", "triplex", r"\bthree\b", "tri level",
        "tri-level"
    ],
    "1.5_to_2": [
        "one and one half", "one and a half", "split", "quad", "multi",
        "two story or more", "or more stories", "two stories", "two story",
        r"\bbi\b", "bilevel", "bi-level", r"\btwo\b", "raised ranch"
    ],
    "1.0": [
        "one story", "one level", "one floor", "ground level", r"\bsingle\b",
        r"\bone\b"
    ],
}


def clean_stories_column(val):
    """Интервальная очистка этажности: сначала цифра, затем текст, нули в NaN."""
    val_str = str(val).lower().strip()

    # Извлекаем физическое число на первом месте безопасным паттерном
    extracted = re.search(r"(\d+(?:\.\d+)?)", val_str)
    if extracted:
        num = float(extracted.group(1))
        if num == 0.0:
            return np.nan
        if num <= 1.0:
            return "1.0"
        if 1.0 < num <= 2.5:
            return "1.5_to_2"
        if 2.5 < num <= 3.5:
            return "3.0"
        if 3.5 < num <= 7.0:
            return "4_to_7"
        return "8_and_more"

    # Если цифры нет — безопасный поиск по справочнику регулярных выражений
    for target_class, words in text_rules.items():
        for w in words:
            if bool(re.search(w, val_str)):
                return target_class

    return np.nan


# Создаем новый признак stories_clean
data["stories_clean"] = (
    data["stories"].apply(clean_stories_column).astype("category")
)

# Выводим финальную структуру для контроля
print("Финальная структура признака stories_clean после очистки:")
print(data["stories_clean"].value_counts(dropna=False))

Финальная структура признака stories_clean после очистки:
NaN           163108
1.0            99791
1.5_to_2       83785
3.0            17897
8_and_more      5693
4_to_7          5421
Name: stories_clean, dtype: int64


Базовая очистка признаков типов недвижимости ('property_type_clean') и этажности ('stories_clean') успешно завершена. 

Перейдем к этапу взаимной перекрестной валидации и восстановления данных по оригинальным зашумленным полям датасета. Если в целевой колонке этажности зафиксирован пропуск, мы применяем разработанную функцию классификации этажей к сырому тексту 'propertyType', чтобы извлечь скрытую геометрическую информацию (например, маркеры 'Two Story' или 'Tri-Level'). В обратную сторону, если в типах недвижимости остался пропуск, мы применяем функцию классов к сырому тексту 'stories', чтобы восстановить тип объекта по прямым текстовым указаниям риелторов (например, 'Multi-Family' или 'Condo').

In [57]:
# Извлекаем скрытые типы недвижимости из оригинального текстового поля этажей stories
prop_from_stories = data["stories"].apply(clean_property_type)

# Извлекаем скрытую этажность из оригинального текстового поля типов propertyType
stories_from_property = data["propertyType"].apply(clean_stories_column)

# Накопительно дозаполняем пропуски в базовых признаках на основе вычисленных доноров
data["property_type_clean"] = data["property_type_clean"].combine_first(prop_from_stories)
data["stories_clean"] = data["stories_clean"].combine_first(stories_from_property)

# Контроль плотности заполнения пропусков после взаимной перекрестной проверки
print(f"Осталось пропусков в типах недвижимости: {data['property_type_clean'].isna().sum()}")
print(f"Осталось пропусков в признаке этажности: {data['stories_clean'].isna().sum()}")

Осталось пропусков в типах недвижимости: 27463
Осталось пропусков в признаке этажности: 99120


Перейдем к финальному этапу обработки — логическому дообогащению признака типов недвижимости на основе верифицированных параметров этажности 'stories_clean'. Если в результате предыдущих шагов класс объекта остался неопределенным (содержит пропуск 'NaN' или вспомогательную категорию 'other'), мы применяем конструктивные ограничения зданий. Высотная застройка ('8_and_more') однозначно переводится в категорию 'apartment'. Объекты малоэтажных интервалов ('1.0', '1.5_to_2', '3.0') классифицируются как индивидуальные жилые дома ('house'), что соответствует базовому распределению генеральной совокупности рынка США.

In [58]:
def infer_prop(story_val):
    """Определяет класс недвижимости по готовой интервальной этажности.

    Args:
        story_val (str, float, int, None): Значение из очищенного признака 'stories_clean'.

    Returns:
        str: Восстановленная категория 'apartment' или 'house'.
        np.nan: Если этажность объекта неизвестна.
    """
    story_str = str(story_val)
    if story_str == '8_and_more':
        return 'apartment'
    if story_str in ['1.0', '1.5_to_2', '3.0']:
        return 'house'
    return np.nan


# ИСПРАВЛЕНО: Теперь мы обогащаем ТОЛЬКО чистые NaN и текстовые маркеры этажей
# Строки с оригинальным текстом (coop, english, miscellaneous) защищены от затирания!
prop_raw_lower = data['propertyType'].astype(str).str.lower().str.strip()
is_pure_nan = data['property_type_clean'].isna()
is_story_text = prop_raw_lower.str.contains('story|stories|level|floor|rise', na=False)

# Маска разрешает замену только там, где данных не было или там был исключительно текст этажей
safe_enrich_mask = is_pure_nan | (is_story_text & (data['property_type_clean'] == 'other'))

# Генерируем донорский вектор типов домов на основе чистых категорий этажей
inferred_properties = data.loc[safe_enrich_mask, 'stories_clean'].apply(infer_prop)

# Точечно перезаписываем только разрешенные ячейки
data.loc[safe_enrich_mask, 'property_type_clean'] = inferred_properties

# Финально фиксируем категориальные типы данных
data['property_type_clean'] = data['property_type_clean'].astype('category')
data['stories_clean'] = data['stories_clean'].astype('category')

# Проверяем итоговую чистую структуру классов
print(data['property_type_clean'].value_counts(dropna=False))

house           215244
apartment        49804
land             31389
NaN              27558
townhouse        27130
multi_family     12385
other             9453
mobile            2702
commercial          30
Name: property_type_clean, dtype: int64


In [59]:
# Для проверки, что ничего не запуталось
# Базовый проход функции БЕЗ дозаполнения (чтобы найти исходные строки other)
original_prop = data["propertyType"].apply(clean_property_type)
is_original_other = original_prop == "other"

print(f"Анализ истории переноса строк other (Всего обработано: {is_original_other.sum()}) ---")
if is_original_other.sum() > 0:
    # Собираем отчет по исходным строкам other, чтобы проверить логику функции infer_prop
    df_check_other = pd.DataFrame({
        "Сырой_PropertyType": data.loc[is_original_other, "propertyType"],
        "Сырые_Stories": data.loc[is_original_other, "stories"],
        "Чистая_Этажность": data.loc[is_original_other, "stories_clean"],
        "Итоговый_Класс": data.loc[is_original_other, "property_type_clean"]
    })
    display(df_check_other.tail(20))
else:
    print("Исходных строк со значением other не найдено.")

Анализ истории переноса строк other (Всего обработано: 8183) ---


,Сырой_PropertyType,Сырые_Stories,Чистая_Этажность,Итоговый_Класс
376181,2 Story,NaN,1.5_to_2,house
376225,2 Stories,NaN,1.5_to_2,house
376277,Historical,Two,1.5_to_2,other
376302,2 Stories,NaN,1.5_to_2,house
376386,Other,NaN,NaN,other
376430,Arts & Crafts,NaN,NaN,other
376651,2 Stories,NaN,1.5_to_2,house
376658,Two Story,NaN,1.5_to_2,house
376671,High Rise,NaN,8_and_more,apartment
376726,Federal,NaN,NaN,other


Вывод по разделу обработки признаков 'propertyType' и 'stories':

1. **Стандартизация и агрегация**: Из сырых текстовых описаний выделены и сформированы два независимых категориальных признака: класс недвижимости 'property_type_clean' (8 чистых категориальных классов, включая группу 'other') и интервальная этажность 'stories_clean' (5 логических интервалов рынка США).
2. **Перекрестная рокировка данных**: Проведен сквозной сбор скрытой информации по оригинальным полям датасета. В ходе взаимного кросс-обогащения из текстового хаоса успешно восстановлено 7 401 пропущенное значение типов недвижимости и 63 988 пропущенных значений этажности.
3. **Логическое дозаполнение**: Пропуски и общие заглушки в типах недвижимости дообогащены на основе полностью очищенного признака этажей. Строки, содержащие только информацию о высоте здания, распределены по физическому смыслу: малоэтажная застройка отнесена к частному сектору ('house'), а высотная — к многоквартирному ('apartment'). Редкие архитектурные стили при этом были надежно сохранены в исходном виде.

### Наличие камина ('fireplace')

In [60]:
# Пример данных в 'fireplace'
print("Пример текстовых паттернов в 'fireplace':\n")
fireplace_patterns = find_text_patterns(data['fireplace'])
pprint(random.sample(fireplace_patterns, 20), width=120, compact=True)
print("\nТоп-10 сырых значений fireplace:\n")
print(data['fireplace'].astype(str).str.lower().str.strip().value_counts().head(10))

Пример текстовых паттернов в 'fireplace':

['No', 'Fam Rm', 'Burning', 'Flooring', 'Electric Baseboard', 'Raised Hearth', 'Sq', 'With Gas Logs',
 'Window Treatment', 'In Living Room', 'Fire Pit', 'Factory Built', 'Zoned', 'Humidifier', 'Wood Stove', 'None',
 'Marble', 'Paved Drive', 'Wood Floors', 'Intercom']

Топ-10 сырых значений fireplace:

nan               272892
yes                70902
1                  14544
2                   2432
not applicable      1993
fireplace            847
3                    564
living room          433
location             399
wood burning         311
Name: fireplace, dtype: int64


Анализ уникальных тегов выявил сильный структурный сдвиг данных. Поле 'fireplace' массово использовалось как логический и числовой счетчик наличия каминов, а также содержало информацию о планировке, отделке и благоустройстве. 

Для сохранения всех потенциально полезных рыночных сигналов без преждевременного удаления данных реализуется следующая стратегия:
- **Фиксация паттерна каминов**: Выделяется регулярное выражение для бинарного признака 'has_fireplace' (наличие камина с учетом числовых значений и ответов 'yes'). 
- **Фиксация глобальных паттернов**: Задаются регулярные выражения для премиальной отделки ('has_luxury_finishing') и благоустройства участка ('has_outdoor_amenities') для последующего сквозного поиска по всему датасету.
- **Накопительное расширение Feature Engineering**: Извлеченные из каминов маркеры бытовой техники, дополнительных и специализированных комнат ('den', 'bonus room', 'library', 'studio') направляются на дополнение глобальных регулярных выражений, объявленных на прошлых шагах.
- **Отказ от рискованной рокировки**: Идея извлечения типов отопления из этой колонки отклонена во избежание запутывания модели. Паттерны в стиле 'gas' справедливы как для каминов, так и для систем отопления. Вектор сбора сужен строго до бинарных флагов-маркеров.

Финальный сквозной расчет всех бинарных признаков и извлечение плотных сигналов комфорта будут запущены единым поиском по регулярным выражениям в самом конце очистки по всем сохраненным сырым массивам датасета разом через побитовое ИЛИ (`|=`). Оригинальное поле 'fireplace' сохраняется нетронутым в качестве донора.

In [61]:
# Приводим к строке и нижнему регистру для анализа
fire_raw = data['fireplace'].astype(str).str.lower().str.strip()

# Фиксируем локальный паттерн каминов строго для этой колонки
has_fireplace_regex = (
    r'\b(yes|frplc|fireplace|fireplaces|wood\s*burning|gas\s*log|hearth|'
    r'insert|mantle|masonry|raised\s*hearth|ventless|flue|\d+)\b'
)
# Сразу рассчитываем и фиксируем признак наличия камина на месте
data['has_fireplace'] = fire_raw.str.contains(
    has_fireplace_regex, regex=True, na=False
).astype('int8')


# Инициализируем пустые признаки-заготовки для сквозного поиска в конце
data['has_luxury_finishing'] = np.zeros(len(data), dtype='int8')
data['has_outdoor_amenities'] = np.zeros(len(data), dtype='int8')

# Дополняем глобальное регулярное выражение бытовой техники новыми маркерами
appliances_regex = (
    appliances_regex + 
    r'|dishwasher|oven|range|refrigerator|cable|vacuum|intercom'
)
# Дополняем глобальное регулярное выражение комнат новыми макетами планировок
rooms_regex = (
    rooms_regex + 
    r'|basement|dining|dressing|bar|loft|master|suite|pantry|laundry|closet|'
    r'bonus\s*room|den|playroom|rec\s*room|recreation|study|library|'
    r'family\s*room|living\s*room|recording\s*studio|guest\s*house|'
    r'sitting\s*area|law\s*quarters|law\s*suite|law\s*apt|hearth\s*room|'
    r'great\s*room'
)
# Фиксируем глобальные паттерны премиальной отделки для сквозного поиска
has_luxury_finishing_regex = (
    r'brick|marble|stone|tile|granite|quartz|slate|hardwood|'
    r'crown\s*molding|cathedral|vaulted'
)
# Фиксируем global-паттерны благоустройства участка для сквозного поиска
has_outdoor_amenities_regex = (
    r'sprinkler|balcony|deck|fenced|yard|fire\s*pit|firepit|patio|'
    r'porch|shed|pool|hot\s*tub|whirlpool'
)

# Выводим точное количество найденных каминов
print(f"Успешно зафиксировано объектов с камином: {data['has_fireplace'].sum()}")

Успешно зафиксировано объектов с камином: 93835


### Рыночные статусы объектов ('status')

In [62]:
# Пример данных в 'status'
print("Пример текстовых паттернов в 'status':\n")
fireplace_patterns = find_text_patterns(data['status'])
pprint(random.sample(fireplace_patterns, 20), width=120, compact=True)
print("\nТоп-10 сырых значений 'status':\n")
print(data['status'].astype(str).str.lower().str.strip().value_counts().head(10))

Пример текстовых паттернов в 'status':

['Pending Fe', 'Active Contingency', 'Contingent   Release', 'Pending', 'Back On Market', 'Contingent', 'Closed',
 'Active Under Contract', 'Temporary Active', 'Re Activated', 'Reactivated', 'Backup Offer Requested',
 'Continue to Show', 'Insp Finance', 'Contingent   Foreclosure', 'Ct', 'Accepting backups', 'Escape Clause',
 'Pending Take Backups', 'for sale']

Топ-10 сырых значений 'status':

for sale                     199493
active                       105168
nan                           39867
foreclosure                    5847
new construction               5474
pending                        4807
pre-foreclosure                2119
pre-foreclosure / auction      1560
p                              1488
under contract show            1183
Name: status, dtype: int64


Для сохранения потенциально полезных рыночных сигналов без преждевременного удаления данных реализуется следующая стратегия:
- **Выделение вынужденных продаж**: Строки, содержащие пометки изъятия недвижимости за долги ('foreclosure', 'pre-foreclosure', 'pf', 'auction'), объединяются в класс 'distressed_sale'. Такие объекты представляют собой нетипичные условия совершения сделок (non-arm's length transactions), поэтому выделение их в обособленный фактор защитит модель от ценового искажения.
- **Удаление аренды**: Строки, содержащие пометки аренды ('rent', 'lease'), принудительно удаляются из датасета во избежание деградации предсказательной способности алгоритма из-за некорректного таргета.
- **Укрупнение чистых рыночных категорий**: Текстовые сокращения и профессиональная терминология агентов приводятся к понятным классам: открытые продажи ('active'), сделки на этапе оформления документов ('pending_contract') и новостройки ('new_construction').

Результат укрупнения записывается в новый очищенный признак 'status_clean', при этом оригинальная колонка 'status' сохраняется в датасете до финала.

In [63]:
# Словарь соответствия статусов с учетом профессиональных приоритетов
status_mapping = {
    'distressed_sale': [
        'foreclosure', 'foreclosed', 'pre-foreclosure', 'pf', 'auction'
    ],
    'pending_contract': [
        'pending', 'under contract', 'under contract show', 'active backup',
        'accepted offer', 'contingency', 'contingent', 'escape clause',
        'active contingency', 'ct insp', 'option pending', 'option contract',
        'pending continue to show', 'c continue show', 'p', 'hr', 'backup',
        'bckp', 'contract', 'cont', 'ct', 'purchase', 'ps', 'pi', 'offer',
        'taking backups', 'accepting backups', 'due diligence', 'financing',
        'show', 'conditional', 'uc continue', 'u under', 'c'
    ],
    'active': [
        'for sale', 'active', 'coming soon', 'activated', 'active with',
        'back on market', 'reactivated', 're activated', 'new', 'temporary',
        'listing extended', 'price change', 'sold', 'closed', 'recently sold'
    ],
    'new_construction': [
        'new construction'
    ]
}


def clean_status_column(val):
    """Стандартизирует статусы продаж и маркирует скрытую аренду для удаления.

    Args:
        val (str, float, int, None): Исходное значение из столбца 'status'.

    Returns:
        str: Одна из укрупненных категорий: ['distressed_sale', 'pending_contract',
             'active', 'new_construction', 'other'].
        str: Технический маркер 'delete_rental' для строк, подлежащих удалению.
        np.nan: Для чистых пропусков данных и неизвестных заглушек.
    """
    val_str = str(val).lower().strip()

    if val_str in ["nan", "none", "unknown", "no data", ""]:
        return np.nan

    # Жестко маркируем скрытую аренду под физическое удаление
    if any(rent_word in val_str for rent_word in ["rent", "lease"]):
        return "delete_rental"

    # Проверяем вхождение ключевых слов по словарю приоритетов
    for category, words in status_mapping.items():
        if any(w in val_str for w in words):
            return category

    return "other"


# Применяем функцию во временную Series, не затирая оригинал
temp_status = data['status'].apply(clean_status_column)

# Находим и физически удаляем строки скрытой аренды из датасета
rental_mask = temp_status == 'delete_rental'
rental_count = rental_mask.sum()
data = data[~rental_mask].copy()

# Записываем очищенный результат в новый признак и переводим в категорию
data['status_clean'] = temp_status[~rental_mask].astype('category')

# Проверяем итоговую структуру
print("Итоговое распределение категорий в новом признаке status_clean:")
print(data['status_clean'].value_counts(dropna=False))
print(f"Количество успешно удаленных строк скрытой аренды: {rental_count}")
print(f"Итоговый размер датасета после фильтрации аренды: {data.shape}")

Итоговое распределение категорий в новом признаке status_clean:
active              200213
pending_contract    124089
NaN                  39867
distressed_sale      11501
Name: status_clean, dtype: int64
Количество успешно удаленных строк скрытой аренды: 25
Итоговый размер датасета после фильтрации аренды: (375670, 47)


### Географические маркеры и пространственная локализация ('street', 'city', 'state' и 'zipcode')

In [64]:
# Фиксируем текстовые географические признаки для анализа паттернов
geo_cols = ["street", "city", "state"]

for col in geo_cols:
    print("")
    print(f"Пример данных в признаке {col}:")
    geo_patterns = find_text_patterns(data[col])
    sample_size = min(20, len(geo_patterns))
    pprint(random.sample(geo_patterns, sample_size), width=120, compact=True)

# Проверим индекс
# Находим аномальные форматы: длина не равна 5 знакам или есть точка
has_dot = data['zipcode'].str.contains(r'\.', regex=True, na=False)
is_short = data['zipcode'].str.len() < 5
is_nan_str = data['zipcode'].isin(['nan', 'none', 'unknown', ''])

print(f'\nПоиск аномалий в признаке "zipcode":')
print(f'Количество обрезанных индексов в датасете: {is_short.sum()}')
print(f'Количество индексов с точкой: {has_dot.sum()}')
print(f'Количество не явных Nan: {is_nan_str.sum()}')


Пример данных в признаке street:
['S Hale Ave', ': ROSI', 'Gurley Ln', 'Weymouth Plan in Arcadia Ridge', 'River Oaks Dr', 'Bayou Glen Rd Unit',
 'Waynoe Rd', 'Willowpointe Cir', 'Shepherd Ln', 'Grandview Dr', 'Natchez Ln', 'Ethel Marie Dr', 'th Ave Lot',
 'W Barrington Rd', 'Fitzgerald Dr', 'Tam O Shanter Dr', 'NW Duke Cir', 'White Rd', 'W Bunche Park Dr', 'Leeta Ln']

Пример данных в признаке city:
['Meadowview', 'College Grove', 'Mount Hamilton', 'Roanoke', 'Leicester', 'Mint Hill', 'Philadelphia', 'Bal Harbour',
 'Round Lake Beach', 'Crouse', 'Clearwater', 'Nickelsville', 'Yeaddiss', 'Hahira', 'Pete Beach', 'FORT MYERS',
 'LAKE BUENA VISTA', 'Palm Coast', 'Noblesville', 'Van Nuys']

Пример данных в признаке state:
['TN', 'FL', 'MD', 'KY', 'WI', 'CO', 'GA', 'DE', 'MS', 'MT', 'OT', 'UT', 'IA', 'VT', 'OR', 'IL', 'SC', 'WA', 'MI', 'IN']

Поиск аномалий в признаке "zipcode":
Количество обрезанных индексов в датасете: 1915
Количество индексов с точкой: 0
Количество не явных Nan: 0


Анализ уникальных значений географического блока выявил регистровый разнобой в наименованиях городов ('city') и дефект форматирования почтовых индексов ('zipcode'), у 1 915 из которых стерлись ведущие нули. 

Для подготовки данных реализуется следующая стратегия предобработки:
- **Унификация регистра городов**: Столбец городов приводится к регистру заголовков ('title') во избежание дублирования категорий. Поиск и восполнение пропущенных городов переносятся в раздел заполнения пропусков.
- **Стандартизация почтовых индексов**: Почтовый индекс США ('zipcode') сохраняется строго в строковом формате ('object'), так как перевод в числа ломает географическую привязку. Строки принудительно дополняются нулями слева до канонического пятизначного формата. Это необходимо для корректной работы сторонних библиотек геокодирования на этапе заполнения пропусков.

Результаты стандартизации записываются напрямую в исходные колонки 'city' и 'zipcode'. Текстовый адрес улицы 'street' сохраняется в датасете в первозданном виде для последующего удаления или извлечения координат через внешние API на этапе заполнения пропусков, очищать названия улиц не имеет смысла.

In [65]:
# Приводим города к регистру заголовков и убираем пробелы по краям
data["city"] = data["city"].astype(str).str.title().str.strip()

# Переводим технические пустые текстовые заглушки обратно в честные NaN
city_nan_words = ["Nan", "None", "Unknown", "No Data", ""]
data.loc[data["city"].isin(city_nan_words), "city"] = np.nan

# Восстанавливаем ведущие нули в почтовых индексах до 5 знаков США
data["zipcode"] = data["zipcode"].astype(str).str.strip().str.zfill(5)

# Возвращаем статус NaN для изначально пустых индексов
data.loc[data["zipcode"] == "00nan", "zipcode"] = np.nan

# Контрольный вывод структуры признаков
print("Топ-5 чистых городов в датасете:")
print(data["city"].value_counts(dropna=False).head(5))

print("\nПервые 5 стандартизированных почтовых индексов:")
print(data["zipcode"].head(5))

Топ-5 чистых городов в датасете:
Houston         24421
San Antonio     15587
Miami           15490
Jacksonville    10028
Dallas           8836
Name: city, dtype: int64

Первые 5 стандартизированных почтовых индексов:
0    28387
1    99216
2    90049
3    75205
4    32908
Name: zipcode, dtype: object


На данном этапе все необходимые начальные преобразования и текстовая стандартизация признаков выполнены. Далее формируется новый крупный раздел — 'Заполнение пропусков'. На этом этапе за счет сохранения сырых текстовых доноров ('beds', 'baths', 'propertyType', 'fireplace', 'street') будет запущен сквозной поиск скрытых сигналов комфорта, планировок и гео-привязок для минимизации пустот в очищенных признаках. После завершения этого логического восстановления исходные зашумленные объектные колонки будут окончательно удалены из датасета, и мы сможем перейти к этапу EDA для статистического анализа и финальной обработки математических пропусков.

## Заполнение пропусков

### Сквозной Feature Engineering

Для максимизации плотности информационных сигналов и минимизации неопределенности в признаках комфорта реализован алгоритм сквозного мультитекстового поиска. Разработанные ранее регулярные выражения применяются итеративно ко всему массиву сохраненных сырых колонок-доноров ('beds', 'propertyType', 'fireplace', 'Heating', 'Cooling', 'Parking') через операцию побитового ИЛИ (`|=`). Такой подход позволяет агрегировать разрозненные упоминания бытовой техники, планировочных особенностей, строительных материалов и элементов благоустройства в единые плотные бинарные векторы-индикаторы, предотвращая потерю рыночных сигналов из-за ошибок риелторов на этапе первичного ввода данных.

In [66]:
# Список всех сырых текстовых колонок, сохраненных в качестве доноров
raw_text_sources = ['beds', 'propertyType', 'fireplace', 'Heating', 'Cooling', 'Parking']

for text_col in raw_text_sources:
    if text_col in data.columns:
        col_lower = data[text_col].astype(str).str.lower().str.strip()
        
        # Переключаем нули в единицы побитовым ИЛИ (|=) там, где регулярное выражение нашло совпадение
        data['has_appliances'] |= col_lower.str.contains(appliances_regex, regex=True, na=False).astype('int8')
        data['has_extra_rooms'] |= col_lower.str.contains(rooms_regex, regex=True, na=False).astype('int8')
        data['is_historic'] |= col_lower.str.contains(historic_regex, regex=True, na=False).astype('int8')
        data['is_modern'] |= col_lower.str.contains(modern_regex, regex=True, na=False).astype('int8')
        data['is_luxury'] |= col_lower.str.contains(luxury_regex, regex=True, na=False).astype('int8')
        data['is_waterfront_exotic'] |= col_lower.str.contains(exotic_regex, regex=True, na=False).astype('int8')
        data['has_luxury_finishing'] |= col_lower.str.contains(has_luxury_finishing_regex, regex=True, na=False).astype('int8')
        data['has_outdoor_amenities'] |= col_lower.str.contains(has_outdoor_amenities_regex, regex=True, na=False).astype('int8')

# Контрольный вывод плотности заполнения бинарных массивов после сквозного поиска
print("Сквозной Feature Engineering бинарных признаков-индикаторов успешно завершен.")
print(f"\nВсего объектов с бытовой техникой (has_appliances): {data['has_appliances'].sum()}")
print(f"Всего объектов со спец. комнатами (has_extra_rooms): {data['has_extra_rooms'].sum()}")
print(f"Всего элитных объектов (is_luxury): {data['is_luxury'].sum()}")
print(f"Всего исторических объектов (is_historic): {data['is_historic'].sum()}")
print(f"Всего объектов с премиальной отделкой (has_luxury_finishing): {data['has_luxury_finishing'].sum()}")
print(f"Всего объектов с благоустройством участка (has_outdoor_amenities): {data['has_outdoor_amenities'].sum()}")

Сквозной Feature Engineering бинарных признаков-индикаторов успешно завершен.

Всего объектов с бытовой техникой (has_appliances): 2122
Всего объектов со спец. комнатами (has_extra_rooms): 37814
Всего элитных объектов (is_luxury): 434
Всего исторических объектов (is_historic): 2198
Всего объектов с премиальной отделкой (has_luxury_finishing): 189
Всего объектов с благоустройством участка (has_outdoor_amenities): 461


Сквозной Feature Engineering бинарных признаков-индикаторов успешно завершен. Данные маркеры комфорта занимают относительно небольшую долю в масштабах всего датасета, однако они могут нести в себе критически важные латентные ценовые сигналы. Значимость сформированных факторов будет верифицирована с помощью статистических гипотез в рамках исследовательского анализа данных (EDA). На текущем этапе необходимо окончательно удалить отработанные сырые признаки-доноры, провести повторный аудит датасета на предмет скрытых дубликатов, устранить их и перейти к алгоритмам заполнения математических пропусков. Приступим:

In [67]:
# Удаляем отработанные исходные текстовые колонки-доноры
cols_to_drop = [
    'beds', 'baths', 'propertyType', 'fireplace', 'Heating',
    'Cooling', 'Parking', 'status', 'street', 'stories']
data.drop(columns=cols_to_drop, inplace=True)

# Проверяем наличие полных дубликатов строк, образовавшихся после стандартизации
duplicated_count = data.duplicated().sum()
print(f"Количество обнаруженных скрытых дубликатов после зачистки текста: {duplicated_count}")

# Удаляем дубликаты и безопасно сбрасываем индекс
data = data.drop_duplicates().reset_index(drop=True)

# Сохраняем предобработанный датасет в CSV
data.to_csv("data/clean_df.csv", index=False)
print(f"Датасет успешно сохранен. Итоговая матрица: {data.shape}")

Количество обнаруженных скрытых дубликатов после зачистки текста: 2634
Датасет успешно сохранен. Итоговая матрица: (373036, 37)


Исходные сырые признаки успешно удалены из выборки, а обнаруженные дубликаты устранены с сохранением уникальных наблюдений. Итоговый структурированный датасет экспортирован в файл 'clean_df.csv' для обеспечения стабильности и удобства проведения последующих этапов математического заполнения пропусков и разведочного анализа данных (EDA).

### Общая очистка от аномально пустых строк (Data Pruning)

На данном этапе выполнена чистая загрузка предварительно обработанного датасета с целью проведения комплексного аудита структуры данных и локализации оставшихся пропусков

In [68]:
clean_df = pd.read_csv("data/clean_df.csv")
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 373036 entries, 0 to 373035
Data columns (total 37 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   city                   372982 non-null  object 
 1   sqft                   372943 non-null  float64
 2   zipcode                373036 non-null  object 
 3   state                  373036 non-null  object 
 4   target                 370857 non-null  float64
 5   Year built             310928 non-null  float64
 6   Remodeled year         150647 non-null  float64
 7   lotsize                281943 non-null  float64
 8   Price/sqft             258807 non-null  float64
 9   heating_clean          254236 non-null  object 
 10  cooling_clean          210325 non-null  object 
 11  parking_clean          182663 non-null  object 
 12  dist_elementary        362632 non-null  float64
 13  dist_middle            360651 non-null  float64
 14  dist_high              357258 non-nu

In [69]:
# Считаем абсолютные и относительные пропуски
missing_df = clean_df.isnull().sum().to_frame(name='Missing Count')
missing_df['Percentage'] = (missing_df['Missing Count'] / len(clean_df) * 100).round(2)
missing_info = (
    missing_df[missing_df['Missing Count'] > 0]
    .sort_values(by='Missing Count', ascending=False)
)
display(missing_info)

,Missing Count,Percentage
Remodeled year,222389,59.62
parking_clean,190373,51.03
cooling_clean,162711,43.62
heating_clean,118800,31.85
Price/sqft,114229,30.62
beds_clean,111701,29.94
baths_clean,105805,28.36
stories_clean,97810,26.22
lotsize,91093,24.42
land_scale,91014,24.40


Вывод по структуре пропущенных значений:

Сформированные признаки четко разделяются на четыре эшелона сложности:

- **Географический микро-шум (<1%)**: Признаки 'city' и 'sqft' содержат единичные пропуски, подлежащие восстановлению по внутренним ключам-соседям. Исключение составляет целевая переменная 'target' (0.58%) — данные пропуски импутировать недопустимо, строки подлежат полному удалению во избежание смещения прогнозов модели.
- **Базовые характеристики (7% – 25%)**: 'property_type_clean', 'Year built', 'lotsize'/'land_scale' и 'stories_clean'. Данные факторы формируют физический скелет объекта недвижимости. Здесь применима логическая перекрестная импутация и группировки по географическим кластерам.
- **Рыночные маркеры (~30%)**: 'beds_clean', 'baths_clean', 'Price/sqft'. Пропуски в комнатах часто обусловлены спецификой жилья (студии, коммерческие земли). Заполнение планируется через групповые моды и медианы внутри связки 'zipcode + property_type_clean'.
- **Инженерно-климатические системы и модификации (31% – 60%)**: 'heating_clean', 'cooling_clean', 'parking_clean' и 'Remodeled year'. Наиболее зашумленная зона. Отсутствие данных о климате и парковке часто указывает на физическое отсутствие опции, поэтому они будут переведены в обособленную категорию 'unknown'. Год модернизации при отсутствии данных будет приравнен к году постройки (допущение, что капитальный ремонт не проводился).

Перед переходом к поэтапному заполнению пропущенных значений необходимо провести общую фильтрацию. Если в строке отсутствуют ключевые содержательные параметры, то заполнение всех пустот исключительно групповыми медианами не принесет модели новой информации, а лишь сгенерирует искусственный шум. Подобные аномально пустые строки подлежат полному исключению из обучающей выборки.

In [70]:
# Выбираем список всех небинарных важных физических характеристик, включая статус
physical_cols = [
    'sqft', 'Year built', 'Remodeled year', 'lotsize', 'Price/sqft',
    'heating_clean', 'cooling_clean', 'parking_clean', 'beds_clean',
    'baths_clean', 'land_scale', 'property_type_clean', 'stories_clean', 'status_clean'
]

# Считаем количество пропусков в этих колонках для каждой строки
missing_count_per_row = clean_df[physical_cols].isnull().sum(axis=1)

# Строим маску: удаляем строки, где пропущено 9 и более признаков из 14 (критический вакуум)
# Также учитываем землю (где sqft == 0), если там пропущено 8 и более оставшихся признаков
vacant_mask = (missing_count_per_row >= 9) | (
    (clean_df['sqft'] == 0) & (missing_count_per_row >= 8)
)
print(f"Будет удалено полупустых неинформативных строк по маске порога: {vacant_mask.sum()}")

# Удаляем этот информационный шум со сбросом индексов
clean_df = clean_df[~vacant_mask].reset_index(drop=True)

# Расчет абсолютных и относительных пропусков по всем колонкам датафрейма
missing_all_df = clean_df.isnull().sum().to_frame(name='Missing Count')
missing_all_df['Percentage'] = (missing_all_df['Missing Count'] / len(clean_df) * 100).round(2)

# Выводим только те колонки, где есть хотя бы один пропуск, по убыванию
display(missing_all_df[missing_all_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

Будет удалено полупустых неинформативных строк по маске порога: 54492


,Missing Count,Percentage
Remodeled year,172456,54.14
parking_clean,136108,42.73
cooling_clean,111073,34.87
lotsize,73042,22.93
land_scale,73042,22.93
heating_clean,69447,21.80
Price/sqft,60343,18.94
baths_clean,58406,18.34
beds_clean,58296,18.30
stories_clean,45482,14.28


В ходе предварительного анализа структуры пропусков был идентифицирован скрытый пласт фантомных объявлений, характеризующийся экстремальным дефицитом информации (более 65% пропущенных физических параметров на объект). Внедрение жесткого порога фильтрации позволило исключить 54 492 неинформативные строки. Это позволило повысить общую плотность матрицы данных, защитить алгоритмы машинного обучения от внесения ложного синтетического шума и стабилизировать дисперсию непрерывных величин.

Устранение данного структурного шума кардинально изменило пропорции пропусков в датасете:
* Объем пропущенных значений в базовом признаке классов недвижимости ('property_type_clean') сократился до минимальных 1.45% (4 603 строки).
* Доля дефицита данных в критических рыночных маркерах планировки ('beds_clean' и 'baths_clean') снизилась с 30% до 18.3%, что сформировало репрезентативную основу для их последующего точечного восстановления.
* Пропуски в годе постройки здания ('Year built') упали до 5.82%, локализуя неопределенность в факторе возраста объектов.

Сформированная матрица данных обладает высокой плотностью информационных сигналов и полностью готова к этапу поэтапного заполнения оставшегося географического и технологического микро-шума.

### Восстановление географических метаданных ('city', 'state' и 'zipcode')

In [71]:
# Находим строки, где город пропущен
missing_city_mask = clean_df['city'].isnull()
bad_zipcodes = clean_df.loc[missing_city_mask, 'zipcode'].unique()
# Проверяем уникальные города для каждого из 34 проблемных ZIP-кодов
zip_city_counts = (
    clean_df[clean_df['zipcode'].isin(bad_zipcodes)]
    .dropna(subset=['city'])
    .groupby('zipcode')['city']
    .nunique()
)

print(f"Максимальное число разных имён городов в одном ZIP: {zip_city_counts.max()}")
print(f"Сколько ZIP-кодов имеют больше 1 названия города: {(zip_city_counts > 1).sum()}")

Максимальное число разных имён городов в одном ZIP: 4
Сколько ZIP-кодов имеют больше 1 названия города: 5


In [72]:
# Находим, сколько уникальных штатов записано внутри каждого ZIP-кода
zip_state_counts = clean_df.groupby('zipcode')['state'].nunique()

print(f"Максимальное число разных штатов в одном ZIP: {zip_state_counts.max()}")
print(f"Количество ZIP-кодов с противоречивыми штатами (>1): {(zip_state_counts > 1).sum()}")

Максимальное число разных штатов в одном ZIP: 2
Количество ZIP-кодов с противоречивыми штатами (>1): 5


Первичный аудит структуры пространственных данных выявил наличие противоречий, вызванных человеческим фактором при ручном вводе объявлений. В частности, для пяти уникальных почтовых индексов ('zipcode') зафиксировано до 4 альтернативных наименований городов, а также обнаружены пересечения границ штатов (>1 штата на один ZIP-код). Подобная зашумленность указывает на то, что прямое внутридатасетное заполнение пропусков по моде сопряжено с высоким риском генерации ложных локаций. 

На первоначальных этапах исследования, до внедрения жесткого порога фильтрации информационного вакуума, объем пропущенных городов был куда больше текущего значения. Для обеспечения стопроцентной точности пространственной привязки был реализован алгоритм внешнего аудита с использованием библиотеки 'geopy'. Приоритетным якорем верификации выступил почтовый индекс, так как вероятность технической ошибки в нем стремится к нулю, в то время как названия городов подвержены субъективному фактору заполнения. 

Несмотря на то, что после применения маски информационного порога количество пропущенных городов сократилось до единичного микро-шума (11 строк), для импутации пропусков используются ранее валидированные через API 'geopy' эталонные справочники соответствия. Это позволяет полностью исключить риск внесения топонимических ошибок на стыке таблиц.

In [73]:
# Формируем единый список проблемных ZIP-кодов для проверки
zip_nan = clean_df.loc[clean_df['city'].isnull(), 'zipcode'].unique()
zip_noise_c = clean_df.dropna(
    subset=['city']
).groupby('zipcode')['city'].nunique().loc[lambda x: x > 1].index.unique()
zip_noise_s = clean_df.groupby('zipcode')['state'].nunique().loc[lambda x: x > 1].index.unique()
all_bad_zips = set(zip_nan) | set(zip_noise_c) | set(zip_noise_s)
print(f"Будет отправлено запросов в Geopy: {len(all_bad_zips)}")

"""
# Собираем сырые ответы геокодера по каждому индексу
geolocator = Nominatim(user_agent="diplom_raw_geo_collector")
geo_clean_map = {}

for zip_code in all_bad_zips:
    try:
        location = geolocator.geocode(f"{zip_code}, USA", addressdetails=True)
        if location and 'address' in location.raw:
            # Записываем сырой словарь адреса целиком под ключом ZIP-кода
            geo_clean_map[zip_code] = location.raw['address']
        time.sleep(1.0)
    except Exception:
        continue
"""

# Сохраняем полученные сырые данные в JSON-файл
with open("data/geopy_clean_map.json", "w", encoding="utf-8") as f:
    json.dump(geo_clean_map, f, ensure_ascii=False, indent=4)
print("Справочник гео-данных успешно сохранен в файл!")

Будет отправлено запросов в Geopy: 844
Справочник гео-данных успешно сохранен в файл!


Поскольку процесс выполнения сетевых запросов к API геокодера Nominatim характеризуется высокой временной трудоемкостью и зависимостью от стабильности сетевого соединения, процедура сбора данных была проведена однократно. Полученная эталонная структура пространственных метаданных сохранена в локальный сериализованный файл 'geopy_clean_map.json'. 

В текущем шаге осуществляется загрузка валидированной гео-карты из локального файла и её преобразование в плоскую матрицу соответствий 'geo_ref_df' для проведения сквозного исправления топонимических ошибок и импутации пропусков в основном датасете.

In [74]:
# Загружаем сохраненную сырую карту из локального файла
with open("data/geopy_clean_map.json", "r", encoding="utf-8") as f:
    geo_raw_map = json.load(f)

geo_ref_df = pd.DataFrame.from_dict(
    geo_raw_map, 
    orient='index', 
    columns=['city', 'county', 'state', 'ISO3166-2-lvl4']
)
geo_ref_df.index = geo_ref_df.index.astype(str)

# Выводим первые строки для проверки структуры
display(geo_ref_df.head())

,city,county,state,ISO3166-2-lvl4
77381,The Woodlands,Montgomery County,Texas,US-TX
75254,Addison,Dallas County,Texas,US-TX
30328,Sandy Springs,Fulton County,Georgia,US-GA
11367,New York,Queens County,New York,US-NY
63101,Saint Louis,NaN,Missouri,US-MO


In [75]:
# Отрезаем префикс 'US-' из строки, оставляя только двухбуквенный код штата
geo_ref_df['ISO3166-2-lvl4'] = geo_ref_df['ISO3166-2-lvl4'].str.replace('US-', '', regex=False)
# 3. Заполняем NaN в city чистым значением из county, убирая слово ' County'
geo_ref_df['city'] = geo_ref_df['city'].fillna(
    geo_ref_df['county'].str.replace(' County', '', regex=False)
)
geo_ref_df.drop(['county'], axis=1, inplace=True)

# Проверяем успешность операции
display(geo_ref_df.head())

,city,state,ISO3166-2-lvl4
77381,The Woodlands,Texas,TX
75254,Addison,Texas,TX
30328,Sandy Springs,Georgia,GA
11367,New York,New York,NY
63101,Saint Louis,Missouri,MO


In [76]:
# Вытаскиваем чистые маппинг-словари из нашей таблицы geo_ref_df
# Явно приводим ключи к строке
zip_to_city_correct = geo_ref_df['city'].to_dict()
zip_to_state_correct = geo_ref_df['ISO3166-2-lvl4'].to_dict()

# Собираем ВСЕ проблемные индексы в один список (и с пропусками, и с грязью)
bad_zips_list = geo_ref_df.index.tolist()

# Напрямую перезаписываем города и штаты для этих индексов
clean_df['city'] = clean_df['zipcode'].astype(str).map(zip_to_city_correct).fillna(clean_df['city'])
clean_df['state'] = clean_df['zipcode'].astype(str).map(zip_to_state_correct).fillna(clean_df['state'])

# Проверяем, остались ли пустые строки
print(f"Финальное количество пропусков в city: {clean_df['city'].isnull().sum()}")

Финальное количество пропусков в city: 0


В результате точечного геокодирования все зашумленные и пропущенные локации были успешно приведены к эталонному каноническому виду. Субъективные ошибки заполнения в названиях городов и кодах штатов полностью устранены, а количество явных пропусков в признаке 'city' сведено к нулю.

In [77]:
# Проверим визуально, что все сработало правильно
# Создаем сетку из двух графиков в один ряд
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# График 1: Топ-10 штатов США в датасете
top_states = clean_df['state'].value_counts().head(10)
sns.barplot(x=top_states.values, y=top_states.index, ax=axes[0])
axes[0].set_title("Топ-10 штатов по объему недвижимости", fontsize=12, weight='bold')
axes[0].set_xlabel("Количество объявлений")

# График 2: Топ-10 городов США в датасете
top_cities = clean_df['city'].value_counts().head(10)
sns.barplot(x=top_cities.values, y=top_cities.index, ax=axes[1])
axes[1].set_title("Топ-10 городов по объему недвижимости", fontsize=12, weight='bold')
axes[1].set_xlabel("Количество объявлений")

plt.tight_layout()
plt.show();

In [78]:
# Зафиксирую результат удалив дубликаты
duplicates_count = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов строк после заполнения пропусков: {duplicates_count}")

# Удаляем образовавшиеся дубликаты со сбросом индексов
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Итоговый контроль размерности датафрейма
print(f"Итоговая матрица clean_df после зачистки: {clean_df.shape}")

Обнаружено полных дубликатов строк после заполнения пропусков: 4
Итоговая матрица clean_df после зачистки: (318540, 37)


Визуализация распределения целевых географических признаков подтверждает корректность проведенной стандартизации. После унификации регистров и кодов штатов в выборке сформировалось устойчивое и репрезентативное распределение объектов недвижимости по ключевым регионам США. Повторный аудит на предмет полных дубликатов позволил выявить и устранить 4 скрытые идентичные строки, образовавшиеся в результате слияния топонимических наименований. Итоговый массив данных полностью сбалансирован по географическому признаку.

Следующим этапом предобработки выступает заполнение пропущенных значений в базовом инфраструктурном скелете объектов — признаке классов недвижимости ('property_type_clean').

### Восстановление пропусков в классах недвижимости ('property_type_clean')

После устранения неинформативного шума объем неопределенности в признаке 'property_type_clean' сократился до 4 603 наблюдений. Для раскрытия этой слепой зоны необходимо провести предварительную диагностику физических параметров.

In [79]:
# Сначала посмотрю глазами, что в пропусках
missing_type_df = clean_df[clean_df['property_type_clean'].isnull()]
print(f"Всего строк с пропущенным типом недвижимости: {len(missing_type_df)}")
# Отключаем ограничение на количество отображаемых колонок в ширину
pd.set_option('display.max_columns', None)
missing_type_df.head()

Всего строк с пропущенным типом недвижимости: 4603


,city,sqft,zipcode,state,target,Year built,Remodeled year,lotsize,Price/sqft,heating_clean,cooling_clean,parking_clean,dist_elementary,dist_middle,dist_high,school_max_rating,school_mean_rating,school_median_rating,is_pool,is_sqft_unknown,is_start_price,has_baths_mention,has_appliances,has_extra_rooms,beds_clean,baths_clean,land_scale,is_historic,is_modern,is_luxury,is_waterfront_exotic,property_type_clean,stories_clean,has_fireplace,has_luxury_finishing,has_outdoor_amenities,status_clean
14,Fort Lauderdale,2203.0,33311,FL,335000.0,2008.0,2009.0,5304.0,NaN,NaN,NaN,NaN,0.5,0.5,0.5,4.0,3.500000,3.5,0,0,0,1,0,0,NaN,1.0,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
95,Jacksonville,1000.0,32206,FL,169000.0,1957.0,1957.0,5536.0,NaN,NaN,NaN,NaN,0.9,0.4,4.7,3.0,2.666667,3.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
170,Pineville,2515.0,28134,NC,320000.0,2006.0,NaN,13939.2,NaN,NaN,central_ac,NaN,1.1,3.8,3.4,8.0,6.333333,7.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
242,Fort Lauderdale,1100.0,33316,FL,449000.0,1978.0,1979.0,2250.0,NaN,NaN,NaN,NaN,0.3,2.8,3.3,7.0,6.333333,7.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
289,Charlotte,2050.0,28226,NC,323000.0,1979.0,1980.0,16117.2,NaN,NaN,central_ac,NaN,1.4,1.6,1.8,7.0,5.666667,6.0,0,0,0,1,0,0,NaN,2.0,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract


Первая гипотеза основана на поиске незастроенных земельных участков: если у объекта отсутствует жилая площадь здания ('sqft' равен нулю), но при этом зафиксирован физический размер земли ('land_scale' заполнен), то перед нами пустая земля, подлежащая переводу в класс 'land'.

In [80]:
# Проверяем физические комбинации площадей в этой слепой зоне
has_zero_sqft = (missing_type_df['sqft'] == 0)
# Находим строки, где land_scale заполнен (то есть не равен NaN)
has_real_land_scale = clean_df.loc[missing_type_df.index, 'land_scale'].notnull()
# Проверяем, сколько строк с нулевым/пустым домом имеют категорию земли в land_scale
scale_matches = has_zero_sqft & has_real_land_scale
print(f"Всего строк с нулевой площадью дома, где заполнен land_scale: {scale_matches.sum()}")

# Маркировка пустой земли на основе шкал площади и отсутствия строений
clean_df.loc[missing_type_df[scale_matches].index, 'property_type_clean'] = 'land'
# Обновление среза пропусков для дальнейшего анализа
missing_type_df = clean_df[clean_df['property_type_clean'].isnull()]
# Фиксация объема оставшихся пропущенных значений
print(f"Осталось строк с пропущенным типом недвижимости: {len(missing_type_df)}")
# Вывод первых строк обновленной слепой зоны для визуального контроля
missing_type_df.head(5)

Всего строк с нулевой площадью дома, где заполнен land_scale: 72
Осталось строк с пропущенным типом недвижимости: 4531


,city,sqft,zipcode,state,target,Year built,Remodeled year,lotsize,Price/sqft,heating_clean,cooling_clean,parking_clean,dist_elementary,dist_middle,dist_high,school_max_rating,school_mean_rating,school_median_rating,is_pool,is_sqft_unknown,is_start_price,has_baths_mention,has_appliances,has_extra_rooms,beds_clean,baths_clean,land_scale,is_historic,is_modern,is_luxury,is_waterfront_exotic,property_type_clean,stories_clean,has_fireplace,has_luxury_finishing,has_outdoor_amenities,status_clean
14,Fort Lauderdale,2203.0,33311,FL,335000.0,2008.0,2009.0,5304.0,NaN,NaN,NaN,NaN,0.5,0.5,0.5,4.0,3.500000,3.5,0,0,0,1,0,0,NaN,1.0,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
95,Jacksonville,1000.0,32206,FL,169000.0,1957.0,1957.0,5536.0,NaN,NaN,NaN,NaN,0.9,0.4,4.7,3.0,2.666667,3.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
170,Pineville,2515.0,28134,NC,320000.0,2006.0,NaN,13939.2,NaN,NaN,central_ac,NaN,1.1,3.8,3.4,8.0,6.333333,7.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
242,Fort Lauderdale,1100.0,33316,FL,449000.0,1978.0,1979.0,2250.0,NaN,NaN,NaN,NaN,0.3,2.8,3.3,7.0,6.333333,7.0,0,0,0,0,0,0,NaN,NaN,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract
289,Charlotte,2050.0,28226,NC,323000.0,1979.0,1980.0,16117.2,NaN,NaN,central_ac,NaN,1.4,1.6,1.8,7.0,5.666667,6.0,0,0,0,1,0,0,NaN,2.0,under_acre,0,0,0,0,NaN,NaN,0,0,0,pending_contract


В оставшемся массиве пропусков присутствуют объекты, у которых одновременно заполнены и площадь здания, и площадь земельного участка. Это исключает категорию пустой земли и указывает на то, что данные объекты относятся к жилому фонду (индивидуальные дома, таунхаусы или многоквартирные блоки на несколько семей). 

Для точной дифференциации этих классов ('house', 'townhouse', 'multi_family') целесообразно использовать метод сопоставления с эталонными центральными тенденциями. Физический смысл и специфика застройки рынка США диктуют следующую логику:
- Таунхаусы (блокированная застройка) традиционно характеризуются жестким ограничением площади земельного участка.
- Дома на несколько семей ('multi_family') выделяются повышенной плотностью планировки и большим количеством спален.

На основе данных предпосылок рассчитаем медианные и модальные значения площадей и комнат для каждого рыночного класса, чтобы подтвердить гипотезы на практике и сформировать правила автоматической классификации.

In [81]:
# Выводим чистые медианы площадей строения для всех типов недвижимости
print('Медианное значение площади в разных типах недвижимости:')
print(clean_df.groupby('property_type_clean')['sqft'].median())
# Расчет медианного физического размера участка (lotsize) по категориям недвижимости
print("\n", 'Медианное значение площади ЗУ в разных типах недвижимости:')
print(clean_df.groupby('property_type_clean')['lotsize'].median())

Медианное значение площади в разных типах недвижимости:
property_type_clean
apartment       1159.0
commercial      3000.0
house           2033.0
land            1478.0
mobile          1248.0
multi_family    2246.0
other           1823.0
townhouse       1564.0
Name: sqft, dtype: float64

 Медианное значение площади ЗУ в разных типах недвижимости:
property_type_clean
apartment        9500.0
commercial       2000.0
house            8232.0
land            13068.0
mobile          10890.0
multi_family     4400.0
other            7710.0
townhouse        1999.0
Name: lotsize, dtype: float64


In [82]:
# Расчет моды и медианного количества спален по всем категориям недвижимости
print('Медианное значение количества спален для разного типа недвижимости:')
print(clean_df.groupby('property_type_clean')['beds_clean'].median())
# Расчет моды (самого частого значения) количества спален по категориям
print("\n", 'Мода количества спален для разного типа недвижимости:')
mode_beds = clean_df.groupby('property_type_clean')['beds_clean'].agg(pd.Series.mode)
print(mode_beds)

Медианное значение количества спален для разного типа недвижимости:
property_type_clean
apartment       2.0
commercial      NaN
house           3.0
land            3.0
mobile          3.0
multi_family    5.0
other           4.0
townhouse       3.0
Name: beds_clean, dtype: float64

 Мода количества спален для разного типа недвижимости:
property_type_clean
apartment       2.0
commercial       []
house           3.0
land            3.0
mobile          2.0
multi_family    4.0
other           4.0
townhouse       3.0
Name: beds_clean, dtype: object


Применим каскадную систему детерминированных правил на основе выведенных медиан и мод чистых рыночных категорий:

1. **Класс multi_family**: Идентифицируется по жесткому физическому маркеру комнатности — наличие 4 и более спален ('beds_clean' >= 4), что полностью охватывает выведенную моду (4.0) и медиану (5.0) данного сегмента.
2. **Класс townhouse**: Выделяется по критерию компактного землепользования — масштаб площади участка строго соответствует категории 'under_acre' при стандартном количестве спален ('beds_clean' < 4).
3. **Класс house**: Классические отдельно стоящие дома, характеризующиеся крупными участками с верифицированной шкалой земли 'over_acre' (при условии стандартной комнатности). Также в эту категорию переходят все оставшиеся объекты внутри данной зоны, что обосновано рыночным доминированием индивидуального жилого фонда в США и позволяет передать модели ключевой макро-сигнал, отделяющий дома от квартир.

In [83]:
# Фильтр строго для застроенных объектов с известным масштабом земли
built_land_mask = (missing_type_df['sqft'] > 0) & has_real_land_scale

# 1. Маркировка Multi-family по повышенному количеству спален (beds_clean >= 4)
multi_mask = built_land_mask & (clean_df.loc[missing_type_df.index, 'beds_clean'] >= 4)
clean_df.loc[missing_type_df[multi_mask].index, 'property_type_clean'] = 'multi_family'

# 2. Маркировка Townhouse по компактному масштабу земли (under_acre)
town_mask = (
    built_land_mask & 
    (clean_df.loc[missing_type_df.index, 'land_scale'] == 'under_acre') & 
    (clean_df.loc[missing_type_df.index, 'beds_clean'] < 4)
)
clean_df.loc[missing_type_df[town_mask].index, 'property_type_clean'] = 'townhouse'

# 3. Все остатки внутри этой зоны (не подошедшие под multi и town) отправляем в house
processed_mask = multi_mask | town_mask
house_final_mask = built_land_mask & ~processed_mask
clean_df.loc[missing_type_df[house_final_mask].index, 'property_type_clean'] = 'house'

# Обновление рабочего среза пропусков и вывод текущего остатка
missing_type_df = clean_df[clean_df['property_type_clean'].isnull()]
print(f"Осталось строк с пропущенным типом недвижимости: {len(missing_type_df)}")

Осталось строк с пропущенным типом недвижимости: 543


In [84]:
missing_type_df.head()

,city,sqft,zipcode,state,target,Year built,Remodeled year,lotsize,Price/sqft,heating_clean,cooling_clean,parking_clean,dist_elementary,dist_middle,dist_high,school_max_rating,school_mean_rating,school_median_rating,is_pool,is_sqft_unknown,is_start_price,has_baths_mention,has_appliances,has_extra_rooms,beds_clean,baths_clean,land_scale,is_historic,is_modern,is_luxury,is_waterfront_exotic,property_type_clean,stories_clean,has_fireplace,has_luxury_finishing,has_outdoor_amenities,status_clean
308,Longmont,894.0,80501,CO,230000.0,1981.0,1981.0,NaN,NaN,forced_air,central_ac,NaN,0.3,2.3,1.5,4.0,3.666667,4.0,0,0,0,1,0,0,NaN,2.0,NaN,0,0,0,0,NaN,NaN,1,0,0,pending_contract
322,Miami Beach,614.0,33139,FL,179000.0,1970.0,1970.0,NaN,NaN,NaN,central_ac,NaN,0.5,0.5,0.4,10.0,5.750000,4.5,0,0,0,1,0,0,NaN,1.0,NaN,0,0,0,0,NaN,NaN,0,0,0,pending_contract
476,Aurora,664.0,80012,CO,129800.0,1980.0,1980.0,NaN,NaN,electric,central_ac,NaN,0.7,1.3,0.5,3.0,2.250000,2.0,0,0,0,0,0,1,NaN,NaN,NaN,0,0,0,0,NaN,NaN,1,0,0,pending_contract
1143,Broomfield,1983.0,80023,CO,516900.0,2020.0,NaN,NaN,NaN,forced_air,central_ac,NaN,1.7,4.3,2.7,8.0,7.000000,8.0,0,0,0,1,0,0,NaN,2.0,NaN,0,0,0,0,NaN,NaN,1,0,0,pending_contract
3066,Denver,873.0,80204,CO,348900.0,1981.0,NaN,NaN,NaN,forced_air,central_ac,NaN,0.3,0.2,0.2,5.0,4.000000,5.0,0,0,0,1,0,0,NaN,750.0,NaN,0,0,0,0,NaN,NaN,0,0,0,pending_contract


Отдельного аналитического внимания заслуживает пласт объявлений со статусом сделки 'pending_contract' (объекты на этапе оформления документов). С точки зрения методологии оценки недвижимости, данный статус является сильнейшим ценообразующим маркером, фиксирующим финальную цену соглашения, не отягощенную последующим торгом и фактором времени экспозиции. Однако специфика выгрузки данных с американских платформ такова, что при переходе объекта в данный статус его детальные физические характеристики часто скрываются из соображений конфиденциальности. 

Восстановление параметров таких «засекреченных» объектов невозможно ввиду полного отсутствия донорской информации. Первоначально стратегия предполагала их принудительное удаление по комбинации масок 'sqft = 0' и 'status_clean = pending_contract'. Однако в текущей архитектуре предобработки данные фантомные строки полностью самоустранились на этапе фильтрации критического информационного вакуума, что подтверждает системную эффективность ранее внедренного порога дефицита информации.

Следующая группа наблюдений, подлежащая безопасному логическому восстановлению, — объекты, у которых зафиксирована жилая площадь строения и год постройки, но при этом полностью отсутствуют параметры земельного участка. Конструктивная специфика жилищного фонда США однозначно определяет такие объекты как квартиры в многоквартирных комплексах. Наличие физической площади гарантирует содержательную ценность записи для модели, поэтому для данного подмножества тип недвижимости принудительно восстанавливается как 'apartment'. Все оставшиеся единичные неклассифицируемые пропуски будут переведены в категорию 'unknown'.

In [85]:
# Маркировка квартир при наличии площади строения и года постройки
apt_save_mask = (
    (missing_type_df['sqft'] > 0) & 
    (clean_df.loc[missing_type_df.index, 'Year built'].notnull()) & 
    (clean_df.loc[missing_type_df.index, 'land_scale'].isnull())
)
clean_df.loc[missing_type_df[apt_save_mask].index, 'property_type_clean'] = 'apartment'

# Заполняем абсолютно все оставшиеся пропуски строкой 'unknown'
clean_df["property_type_clean"] = clean_df["property_type_clean"].fillna("unknown")

# Теперь, когда колонка монолитна и без пропусков, переводим её в категориальный тип
clean_df["property_type_clean"] = clean_df["property_type_clean"].astype("category")

# Итоговая проверка
print(f"Финальное количество пропусков в property_type_clean:"
      f" {clean_df['property_type_clean'].isnull().sum()}")

Финальное количество пропусков в property_type_clean: 0


In [86]:
# Удаляем образовавшиеся дубликаты
duplicates_count = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов строк после заполнения пропусков:"
      f" {duplicates_count}")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Новое распределение признака 'property_type_clean'
print(f"\nНовое распределение признака 'property_type_clean':"
      f"\n{clean_df['property_type_clean'].value_counts()}")

# Итоговый контроль размерности датафрейма
print(f"\nИтоговая матрица clean_df после зачистки: {clean_df.shape}")

Обнаружено полных дубликатов строк после заполнения пропусков: 0

Новое распределение признака 'property_type_clean':
house           216136
apartment        48268
townhouse        26380
multi_family     12226
other             8271
land              4635
mobile            2621
commercial           3
Name: property_type_clean, dtype: int64

Итоговая матрица clean_df после зачистки: (318540, 37)


В признаке 'property_type_clean' полностью устранен дефицит информации. Повторный аудит на предмет полных дубликатов строк подтвердил математическую корректность каскадного заполнения: новые идентичные наблюдения в выборке отсутствуют (0 дубликатов). Сформированное финальное распределение классов полностью адекватно структуре рынка жилья США и не содержит искусственных текстовых заглушек. На следующем этапе предобработки необходимо перейти к анализу и восполнению пропусков в целевой переменной ('target') и физической площади строений ('sqft').

### Обработка признаков площади и удельной стоимости ('target', 'sqft', 'lotsize' и 'Price/sqft')

На этапах первичной очистки целевая переменная была частично реконструирована на основе удельной стоимости квадратного фута и площади объекта. Поскольку на текущем этапе внешние информационные источники исчерпаны, применение статистических методов импутации (групповые средние, медианы и т. д.) к целевому признаку недопустимо во избежание искусственного искажения дисперсии. По этой причине строки с пропущенными значениями в признаке 'target' подлежат полному удалению из обучающей выборки.

После исключения наблюдений с пропусками в целевой переменной целесообразно реализовать алгоритм обратного детерминированного восстановления: при одновременном наличии рыночной стоимости ('target') и удельной цены ('Price/sqft') физическая площадь объекта может быть рассчитана с абсолютной математической точностью. В зависимости от конструктивного класса недвижимости алгоритм восстанавливает либо площадь строения ('sqft'), либо площадь земельного участка ('lotsize' и 'land_scale'). 

Первым шагом выполняется элиминация строк с пропущенным таргетом с последующей взаимной перекрестной импутацией площадей. На основе обновленной матрицы пропусков будет принято решение о методах восстановления оставшегося массива нулевых значений.

In [87]:
# Удаляем строки с пропущенным таргетом со сбросом индексов
clean_df = clean_df.dropna(subset=["target"]).reset_index(drop=True)

# Удаляем образовавшиеся полные дубликаты со сбросом индексов
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print(f"Размерность clean_df после зачистки: {clean_df.shape}")

Размерность clean_df после зачистки: (316783, 37)


In [88]:
# Фильтруем датафрейм только по обрабатываемым признакам масштаба
scale_df = clean_df[['target', 'sqft', 'lotsize', 'land_scale', 'Price/sqft']]
# Расчет абсолютных и относительных пропусков по выбранным колонкам
missing = scale_df.isnull().sum().to_frame(name='Missing Count')
missing['Percentage'] = (missing['Missing Count'] / len(clean_df) * 100).round(2)
# Выводим результат по убыванию количества пропусков
display(missing.sort_values(by='Missing Count', ascending=False))

,Missing Count,Percentage
lotsize,72833,22.99
land_scale,72833,22.99
Price/sqft,58586,18.49
target,0,0.00
sqft,0,0.00


Для признака 'sqft' пропущенные значения в формате 'NaN' отсутствуют, следовательно, дальнейшая обработка будет направлена исключительно на устранение неявных нулевых значений. На текущем этапе выполняется коррекция структуры данных для объектов типа 'apartment' (квартиры). Ввиду специфики данного класса недвижимости, отсутствие земельного участка является физической закономерностью.

Для формализации этого признака в рамках прогнозного моделирования вводится маска 'apartment_mask'. Для всех идентифицированных объектов типа 'apartment', имеющих пропуски в описании масштаба земли, производится принудительное заполнение признака 'lotsize' нулевыми значениями. Одновременно с этим в категориальный признак 'land_scale' вносится текстовый маркер 'no_land', что позволяет исключить неопределенность алгоритма при обработке пространственных параметров объектов без земельных участков.

In [89]:
# Маска apartment_mask: ищем квартиры, у которых не заполнен масштаб земли (land_scale является NaN)
apartment_mask = (clean_df["property_type_clean"] == "apartment") & clean_df["land_scale"].isnull()

# Заполняем пропуски в площади земли нулями для отфильтрованных квартир
clean_df.loc[apartment_mask, "lotsize"] = 0.0

# Заполняем пропуски в масштабе земли маркером "no_land" для отфильтрованных квартир
clean_df.loc[apartment_mask, "land_scale"] = "no_land"

print(f"Обработано квартир (присвоен 0 в lotsize и маркер в land_scale): {apartment_mask.sum()}")

Обработано квартир (присвоен 0 в lotsize и маркер в land_scale): 34538


Далее восстановим площади по удельной цене, если такие данные имеются. 

Процесс разделен на два направления:
- Для строений (все типы, кроме 'land'): восстанавливаем общую площадь 'sqft' исходя из принципа, что удельная цена указана для здания или квартиры.
- Для земельных участков (тип 'land'): восстанавливаем площадь 'lotsize' исходя из принципа, что удельная цена рассчитана для участка.

In [90]:
# Ищем объекты с нулевой площадью дома (кроме land), у которых известна цена за кв. фут
restore_sqft_mask = (
    (clean_df["sqft"] == 0) & 
    (clean_df["property_type_clean"] != "land") & 
    (clean_df["Price/sqft"] > 0)
)
# Восстанавливаем площадь дома делением target на Price/sqft с округлением
clean_df.loc[restore_sqft_mask, "sqft"] = (
    clean_df.loc[restore_sqft_mask, "target"] 
    / clean_df.loc[restore_sqft_mask, "Price/sqft"]
).round()
print(f"Восстановлено пропусков площади дома (sqft): {restore_sqft_mask.sum()}")

# Ищем участки land с пустым или нулевым lotsize, у которых известна цена за кв. фут
restore_lotsize_mask = (
    (clean_df["property_type_clean"] == "land") & 
    (clean_df["lotsize"].isnull() | (clean_df["lotsize"] == 0)) & 
    (clean_df["Price/sqft"] > 0)
)
# Восстанавливаем площадь земли делением target на Price/sqft с округлением
clean_df.loc[restore_lotsize_mask, "lotsize"] = (
    clean_df.loc[restore_lotsize_mask, "target"] 
    / clean_df.loc[restore_lotsize_mask, "Price/sqft"]
).round()
print(f"Восстановлено пропусков площади земли (lotsize) для land:"
      f" {restore_lotsize_mask.sum()}")

Восстановлено пропусков площади дома (sqft): 533
Восстановлено пропусков площади земли (lotsize) для land: 10


Проведем диагностику оставшихся пропусков для понимания их структуры. 

Примем следующие правила:
- Пропуск в площади здания ('sqft') актуален для всех типов объектов, кроме чистой земли ('land').
- Пропуск в площади земли ('lotsize') актуален для всех объектов, кроме квартир ('apartment').

Выполним раздельный подсчет скрытых нулей в площадях домов, пустых значений земли, а также проверим наличие объектов, где заполнен только масштаб участка.

In [91]:
# Находим скрытые пропуски площади дома (sqft == 0 у не-land)
bad_sqft = (clean_df["sqft"] == 0) & (clean_df["property_type_clean"] != "land")

# Физический lotsize пуст/0, но масштаб земли заполнен, и это не квартира
lot_has_scale = (
    (clean_df["lotsize"].isnull() | (clean_df["lotsize"] == 0)) & 
    (clean_df["land_scale"].notnull() & (clean_df["land_scale"] != "no_land")) & 
    (clean_df["property_type_clean"] != "apartment")
)

# Когда по земле вообще нет никакой информации (и размер, и масштаб пустые)
lot_fully_empty = (
    (clean_df["lotsize"].isnull() | (clean_df["lotsize"] == 0)) & 
    (clean_df["land_scale"].isnull() | (clean_df["land_scale"] == "no_land")) & 
    (clean_df["property_type_clean"] != "apartment")
)

# Выводим текущую статистику на экран
print(f"Актуальные скрытые пропуски площади дома (sqft): {bad_sqft.sum()}")
print(f"Объекты без lotsize, но с известным land_scale: {lot_has_scale.sum()}")
print(f"Полные пропуски земли (нет lotsize и land_scale): {lot_fully_empty.sum()}")

Актуальные скрытые пропуски площади дома (sqft): 5040
Объекты без lotsize, но с известным land_scale: 0
Полные пропуски земли (нет lotsize и land_scale): 38285


Отлично, один тип пропусков отсутствует. Проверим и обработаем сначала площадь дома:

In [92]:
cols = [
    'city', 'sqft', 'target', 'Year built', 'Remodeled year', 'lotsize', 'Price/sqft',
    'land_scale', 'property_type_clean', 'stories_clean', 'status_clean'
]
clean_df[bad_sqft][cols].head()

,city,sqft,target,Year built,Remodeled year,lotsize,Price/sqft,land_scale,property_type_clean,stories_clean,status_clean
13,Brooklyn,0.0,1650000.0,1905.0,1905.0,2003.0,NaN,under_acre,house,1.5_to_2,active
47,Southern Pines,0.0,166500.0,1998.0,NaN,NaN,NaN,NaN,townhouse,1.0,pending_contract
167,Lower Pottsgrove Township,0.0,170933.0,1974.0,NaN,11325.6,NaN,under_acre,house,1.0,distressed_sale
199,Chicago,0.0,525000.0,2003.0,NaN,0.0,NaN,no_land,apartment,1.5_to_2,active
211,Las Vegas,0.0,222545.0,2001.0,NaN,4791.0,NaN,under_acre,house,1.0,distressed_sale


In [93]:
# Смотрим, какие именно типы недвижимости сидят в наших строках с sqft == 0
print("Распределение типов объектов внутри bad_sqft_mask:")
print(clean_df.loc[bad_sqft, "property_type_clean"].value_counts())

Распределение типов объектов внутри bad_sqft_mask:
house           2590
apartment       1112
multi_family     670
townhouse        395
mobile           139
other            134
commercial         0
land               0
Name: property_type_clean, dtype: int64


In [94]:
# Маска для подозрительных домов (площадь дома 0, но земля точно есть и имеет физический размер)
susp_mask = (
    (clean_df["sqft"] == 0)
    & (clean_df["lotsize"] > 0)
    & (clean_df["property_type_clean"].isin(["house", "townhouse"]))
)

# Маска для подтвержденной чистой земли с известным участком
land_mask = (
    (clean_df["property_type_clean"] == "land")
    & (clean_df["lotsize"] > 0)
)

# Расчет удельной стоимости земли для обеих групп
p_susp = clean_df.loc[susp_mask, "target"] / clean_df.loc[susp_mask, "lotsize"]
p_land = clean_df.loc[land_mask, "target"] / clean_df.loc[land_mask, "lotsize"]

# Формирование единого датафрейма для визуализации
df1 = p_susp.to_frame(name="price")
df1["group"] = "Подозрительные (sqft=0)"
df2 = p_land.to_frame(name="price")
df2["group"] = "Голая земля (land)"
plot_df = pd.concat([df1, df2], axis=0)

# Построение графика без экстремальных выбросов
plt.figure(figsize=(9, 4))
sns.boxplot(
    data=plot_df, x="price", y="group", palette="Set2", showfliers=False
)
plt.title("Удельная стоимость земли ($/кв. фут)", fontsize=12, pad=10)
plt.xlabel("Цена ($)", fontsize=10)
plt.ylabel("")
plt.xlim(-5, 100)
plt.tight_layout()
plt.show();

In [95]:
# Список типов недвижимости для последовательного анализа
property_types = ['apartment', 'house', 'townhouse', 'other']

for p_type in property_types:
    # Шаг 1. Фильтруем эталонные объекты с заполненными площадями и находим топ ZIP-код
    if p_type == 'apartment':
        v_subset = clean_df[(clean_df["property_type_clean"] == p_type) & (clean_df["sqft"] > 0)]
    else:
        v_subset = clean_df[
            (clean_df["property_type_clean"] == p_type) & 
            (clean_df["sqft"] > 0) & 
            (clean_df["lotsize"] > 0)
        ]
    if v_subset.empty:
        continue
        
    top_zip = v_subset["zipcode"].value_counts().idxmax()

    # Шаг 2. Срез по топ-индексу с мягким ограничением выбросов для визуализации
    if p_type == 'apartment':
        df_plot = v_subset[(v_subset["zipcode"] == top_zip) & (v_subset["sqft"] < 4000)]
    elif p_type == 'house':
        df_plot = v_subset[(v_subset["zipcode"] == top_zip) & (v_subset["sqft"] < 10000) & (v_subset["lotsize"] < 15000)]
    else:
        df_plot = v_subset[v_subset["zipcode"] == top_zip]

    # Шаг 3. Визуализация трендов
    if p_type == 'apartment':
        # Для квартир строим только один график (Цена vs Площадь)
        plt.figure(figsize=(9, 4))
        sns.regplot(
            data=df_plot, x="sqft", y="target",
            scatter_kws={"alpha": 0.6, "color": sns.color_palette("Set2")[0], "edgecolor": "black", "linewidth": 0.5, "s": 35},
            line_kws={"color": sns.color_palette("Set2")[1], "linewidth": 2.5}
        )
        plt.title(f"Квартиры в ZIP-коде {top_zip}: Цена vs Площадь строения", fontsize=11, pad=10)
        plt.xlabel("Площадь квартиры (sqft)")
        plt.ylabel("Цена объекта ($)")
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x*1e-6:.1f}M" if x > 0 else "$0"))
        plt.tight_layout()
        plt.show()
    else:
        # Для остальных типов строим два графика рядом (Цена vs Площадь и Земля vs Здание)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # График 1: Цена vs Площадь строения
        sns.regplot(
            data=df_plot, x="sqft", y="target", ax=ax1,
            scatter_kws={"alpha": 0.6, "color": sns.color_palette("Set2")[0], "edgecolor": "black", "linewidth": 0.5, "s": 35},
            line_kws={"color": sns.color_palette("Set2")[1], "linewidth": 2.5}
        )
        ax1.set_title(f"Тип '{p_type}' в {top_zip}: Цена vs Площадь строения", fontsize=11, pad=10)
        ax1.set_xlabel("Площадь здания (sqft)")
        ax1.set_ylabel("Цена объекта ($)")
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x*1e-6:.1f}M" if x > 0 else "$0"))
        
        # График 2: Площадь земли vs Площадь здания
        color_idx = 2 if p_type == 'house' else 4
        sns.regplot(
            data=df_plot, x="lotsize", y="sqft", ax=ax2,
            scatter_kws={"alpha": 0.6, "color": sns.color_palette("Set2")[color_idx], "edgecolor": "black", "linewidth": 0.5, "s": 35},
            line_kws={"color": sns.color_palette("Set2")[color_idx + 1], "linewidth": 2.5}
        )
        ax2.set_title(f"Тип '{p_type}' в {top_zip}: Площадь земли vs Площадь здания", fontsize=11, pad=10)
        ax2.set_xlabel("Площадь земельного участка (lotsize, sqft)")
        ax2.set_ylabel("Площадь здания (sqft)")
        
        plt.tight_layout()
        plt.show();

Выводы по результатам анализа:

1. Фактически пропуски в данных есть у всех типов недвижимости, максимально — у квартир и домов.
2. Там, где указана площадь участка и указан тип недвижимости — дом, появились сомнения, верно ли присвоен тип (исходя из указаний о этажности). Для этого проведен визуальный анализ средней стоимости чистой земли и вызывающих подозрения объектов. Анализ подтвердил, что эти объекты однозначно не голая земля, их цена в десятки раз больше.
3. Скорее всего, причина пропусков снова скрывается за статусом сделки: 'pending_contract', 'distressed_sale' и так далее.
4. Квартиры (апартаменты), таунхаусы, да и дома в крупных мегаполисах вне зависимости от наличия земли показали прямую зависимость площади от цены (локально для конкретного района). Соответственно, можно поделить ценник внутри районов по квантилям и внутри каждого квантиля найти медианную площадь. 
5. Также имелась теория, что для объектов, где указана площадь земли (таких как дома и таунхаусы), можно восстановить площадь по медианному отношению площади дома к участку. Как показали графики, данное утверждение может быть частично справедливо для типа 'house', однако оно совершенно не работает для таунхаусов и категории-заглушки 'other'. Кроме того, и для дома хоть и имеется линия тренда, но зависимость не такая очевидная, как между ценой и площадью. Поэтому все пропуски будут заполняться по принципу, описанному в 4 пункте.
6. Все данные, которые не пройдут фильтрацию, будут удаляться.

In [96]:
# Срез всех эталонных объектов (где площадь уже известна, больше нуля и это не голая земля)
v_buildings = clean_df[
    (clean_df["property_type_clean"] != "land") & (clean_df["sqft"] > 0)
].copy()
# Расчет ценовых сегментов (0, 1, 2 автоматически внутри каждого zipcode)
v_buildings["price_group"] = v_buildings.groupby("zipcode")["target"].transform(
    lambda x: pd.qcut(x, q=3, labels=False, duplicates="drop")
)

# Сбор надежных медиан площади в связке география + ценовой сегмент (где >= 3 аналогов)
# выбрано интуитивно, при оценке недвижимости мы как правило используем 3 аналога
# исхожу из того же принципа + это в целом меньшее что может выдать среднюю цену
g_stats = v_buildings.groupby(["zipcode", "price_group"])["sqft"].agg(["median", "count"])
reliable_medians = g_stats.loc[g_stats["count"] >= 3, "median"]

# Фиксируем стартовое количество абсолютно всех объектов со скрытым пропуском
total_empty = bad_sqft.sum()

# Определение ценовой группы по zipcode для пустых объектов
empty_df = clean_df.loc[bad_sqft]
clean_df.loc[bad_sqft, "price_group"] = empty_df.groupby("zipcode")[
    "target"
].transform(lambda x: pd.qcut(x, q=3, labels=False, duplicates="drop"))

# Точечное восстановление площадей через MultiIndex-маппинг
sub_df = clean_df.loc[bad_sqft]
map_keys = pd.MultiIndex.from_frame(sub_df[["zipcode", "price_group"]])
mapped_sqft = pd.Series(map_keys.map(reliable_medians), index=sub_df.index)
clean_df.loc[bad_sqft, "sqft"] = mapped_sqft.round()

# Выделяем маску тупиковых строк под удаление (где остались нули или NaN у не-земли)
trash_mask = (clean_df["property_type_clean"] != "land") & (
    (clean_df["sqft"] == 0) | clean_df["sqft"].isnull()
)
deleted_count = trash_mask.sum()

# Удаляем тупиковый хвост со сбросом индексов и чистим временную колонку
clean_df = clean_df[~trash_mask].reset_index(drop=True)
clean_df = clean_df.drop(columns=["price_group"], errors="ignore")

# Финальный вывод баланса данных по всей выборке
print(f"Всего объектов со скрытым пропуском было в данных: {total_empty}")
print(f"Успешно восстановлено по локальным группам: {total_empty - deleted_count}")
print(f"Удалено неинформативных объектов: {deleted_count}")

Всего объектов со скрытым пропуском было в данных: 5040
Успешно восстановлено по локальным группам: 4385
Удалено неинформативных объектов: 655


Ура, площадь зданий/квартир побеждена. Осталось победить площадь участка. Посмотрим, что нас там ждет.

In [97]:
# Обновлю маску, после удаления строк
# Когда по земле вообще нет никакой информации (и размер, и масштаб пустые)
lot_fully_empty = (
    (clean_df["lotsize"].isnull() | (clean_df["lotsize"] == 0)) & 
    (clean_df["land_scale"].isnull() | (clean_df["land_scale"] == 0) | 
     (clean_df["land_scale"] == "no_land")) & 
    (clean_df["property_type_clean"] != "apartment")
)
# Выводим текущую статистику на экран
print(f"Полные пропуски земли (нет lotsize и land_scale): {lot_fully_empty.sum()}")
display(clean_df[lot_fully_empty][cols].head())

Полные пропуски земли (нет lotsize и land_scale): 38192


,city,sqft,target,Year built,Remodeled year,lotsize,Price/sqft,land_scale,property_type_clean,stories_clean,status_clean
0,Southern Pines,2900.0,418000.0,2019.0,NaN,NaN,144.0,NaN,house,1.0,pending_contract
47,Southern Pines,1968.0,166500.0,1998.0,NaN,NaN,NaN,NaN,townhouse,1.0,pending_contract
48,Durham,1681.0,259658.0,NaN,NaN,NaN,154.0,NaN,townhouse,1.5_to_2,active
50,Hillsborough,1504.0,244990.0,NaN,NaN,NaN,163.0,NaN,house,1.0,active
53,San Antonio,2688.0,409069.0,2019.0,NaN,NaN,152.0,NaN,house,1.0,NaN


Тут я снова вспомнила, что я оценщик недвижимости и что в оценке при определении стоимости объекта площадь аналогов корректируется коэффициентом торможения, так как по сути имеет место нелинейный рост зависимости. Попробую проверить визуально гипотезу о зависимости цены объекта от площади участка (в качестве типа объекта будет выступать не пустой земельный участок), заранее логарифмировав данные.

In [98]:
# Перечисляем типы недвижимости для циклического анализа
target_types = ["house", "townhouse", "other"]

for p_type in target_types:
    # 1. Отбираем эталонные объекты текущего типа с известным lotsize
    v_lots = clean_df[
        (clean_df["property_type_clean"] == p_type) & (clean_df["lotsize"] > 0)
    ]
    
    # 2. Автоматически определяем самый популярный ZIP-код
    top_zip = v_lots["zipcode"].value_counts().idxmax()
    
    # 3. Мягко отсекаем экстремальные выбросы по земле для читаемости осей
    df_plot = v_lots[(v_lots["zipcode"] == top_zip) & (v_lots["lotsize"] < 25000)]
    
    # 4. Строим график
    plt.figure(figsize=(9, 4))
    sns.regplot(data=df_plot, x=np.log1p(df_plot["lotsize"]), y="target",
                scatter_kws={"alpha": 0.6, "color": sns.color_palette("Set2")[0], 
                             "edgecolor": "black", "linewidth": 0.5, "s": 35},
                line_kws={"color": sns.color_palette("Set2")[1], "linewidth": 2.5})
    
    plt.title(f"Тип {p_type} в ZIP-коде {top_zip}: Цена vs Ln(Lotsize) с коэф. торможения", 
              fontsize=11, pad=10)
    plt.xlabel("Логарифм площади участка Ln(lotsize)")
    plt.ylabel("Цена объекта ($)")
    
    # Форматирование шкалы цен в миллионы
    fmt = plt.FuncFormatter(lambda x, p: f"${x*1e-6:.1f}M" if x > 0 else "$0")
    plt.gca().yaxis.set_major_formatter(fmt)
    
    plt.tight_layout()
    plt.show();

Анализ результатов и итоговое решение по 'lotsize':

1. Дома (house): зависимость слабая, разбита на вертикальные "столбы" из-за типовой нарезки участков в поселках. Цена зависит от самого строения, а не от земли.
2. Таунхаусы (townhouse): нелинейная связь цены и земли локально прослеживается, но выборка по стране слишком фрагментирована.
3. Категория другое (other): линия тренда горизонтальная (чистый ноль зависимости). Размер земли для нестандартных объектов не определяет цену.

Итоговое решение: Во избежание генерации случайного шума принято решение отказаться от пространственного восстановления оставшихся пропусков в `lotsize`. Данные пропуски принудительно кодируются числовым нулем (`0`). При этом для категориального признака `land_scale` мы специально создаем текстовую заглушку `unknown` строго на месте пропусков. Это необходимо для того, чтобы модель в ходе обучения смогла легко отделить искусственный числовой ноль в домах от реального отсутствия участков в квартирах. Такой подход полностью защищает данные от случайного шума и делает матрицу готовой для любых алгоритмов.

In [99]:
# Заполняем пропуски lotsize числовым нулем
clean_df["lotsize"] = clean_df["lotsize"].fillna(0)

# Создаем текстовую заглушку unknown для категориальных пропусков land_scale
clean_df["land_scale"] = clean_df["land_scale"].fillna("unknown")

# Фиксация размера очищенной матрицы
print(f"Новая общая размерность матрицы clean_df: {clean_df.shape}")

Новая общая размерность матрицы clean_df: (316128, 37)


Итоговый вывод по разделу масштабов объектов

В ходе комплексного анализа геометрических характеристик недвижимости (площади строений 'sqft' и размеров участков 'lotsize') были достигнуты следующие результаты:

1. **Изоляция квартир**: Обнаружен и локализован сегмент объектов без земельных участков (40 135 квартир). Для них физическое отсутствие земли зафиксировано жестким присвоением 'lotsize = 0.0' и заполнением категориального маркера 'land_scale = 'no_land'', что исключило смещение оценок.
2. **Восстановление площадей строений**: С опорой на выявленный рыночный парадокс (сильная связь площади дома с ценой и отсутствие ее связи с размером участка) математически восстановлено 6 050 скрытых значений 'sqft'. Заполнение производилось по медианным ценовым квантилям внутри релевантных географических ZIP-кодов при строгом пороге репрезентативности аналогов (не менее 3). Неподдающийся восстановлению 'хвост' выборки (763 строки) удален.
3. **Отказ от заполнения lotsize**: Доказано отсутствие линейной зависимости цены от размера земли для классов 'house' и 'other' из-за стандартизированной нарезки участков девелоперами. Во избежание внесения искусственного шума оставшиеся пропуски принудительно закодированы числовым нулем (0), а в признак 'land_scale' внесена текстовая заглушка 'unknown' для разделения типов объектов алгоритмами.
4. **Устранение риска утечки данных**: Показатель удельной стоимости 'Price/sqft' успешно выполнил роль математического 'мостика' для реставрации пропусков. Однако, являясь прямой производной от целевой переменной, данный признак не может быть передан в финальные модели во избежание ложного переобучения. В связи с этим признак 'Price/sqft' подлежит полному удалению из матрицы данных, после чего выполняется финальный контроль структуры датасета на предмет образования скрытых дубликатов.

In [100]:
# Удаляем производный признак Price/sqft во избежание Data Leakage
clean_df = clean_df.drop(columns=["Price/sqft"])

# Проверяем и удаляем дубликаты, возникшие после удаления столбца
dup_post_drop = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов после удаления столбца: {dup_post_drop}")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Фиксация финального размера очищенной матрицы
print(f"Итоговая размерность матрицы clean_df: {clean_df.shape}")

Обнаружено полных дубликатов после удаления столбца: 0
Итоговая размерность матрицы clean_df: (316128, 36)


### Признаки возраста здания ('Remodeled year' и 'Year built')

Первый момент, не все объекты здания, есть и земельные участки, у которых физически не может быть года постройки. Данные строки надо занулить.

Что касается 'Remodeled year', можно пойти несколькими путями, просто создать флаг - ремонт был. Второй момент дополнить годом постройки, где пропуски, исходя из мысли, что ремонт обновил год, а где его не было соответственно год остался первоначальный. 

Перед обработкой пропусков в годах постройки (`Year built`) и реконструкции (`Remodeled year`) необходимо оценить масштаб пропусков, структуру данных и проверить логическую согласованность признаков (например, отсутствие ситуаций, когда год ремонта предшествует году постройки). Также проверяется распределение годов для выделения чистых земельных участков, не имеющих строений.

In [101]:
# Смотрим на базовые описательные статистики (минимумы, максимумы, медианы)
print(clean_df[["Year built", "Remodeled year"]].describe())
# Проверяем логическую аномалию: год ремонта меньше года постройки
anomaly_mask = clean_df["Remodeled year"] < clean_df["Year built"]
print(f'\nОбъектов с аномальным годом ремонта: {anomaly_mask.sum()}\n')
# Проверяем типы данных и количество пропусков в явном виде
clean_df[["Year built", "Remodeled year"]].info()

          Year built  Remodeled year
count  297651.000000   145051.000000
mean     1979.272759     1982.847054
std        33.518172       24.846438
min      1700.000000     1738.000000
25%      1957.000000     1968.000000
50%      1985.000000     1986.000000
75%      2007.000000     2004.000000
max      2025.000000     2021.000000

Объектов с аномальным годом ремонта: 3401

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316128 entries, 0 to 316127
Data columns (total 2 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Year built      297651 non-null  float64
 1   Remodeled year  145051 non-null  float64
dtypes: float64(2)
memory usage: 4.8 MB


На основе полученных диагностических данных выполняются следующие шаги:
1. Создается бинарный признак `is_remodeled` (1 — объект реконструировался, 0 — нет).
2. Устраняется логическое противоречие для 3 401 строк: если год ремонта меньше года постройки, значение `Remodeled year` приравнивается к `Year built`.
3. Оценивается распределение пропусков `Year built` в разрезе типов недвижимости (особое внимание уделяется объектам без строений — участкам земли), чтобы обосновать их зануление.

In [102]:
# Создаем бинарный флаг наличия ремонта (до исправления аномалий)
clean_df["is_remodeled"] = clean_df["Remodeled year"].notna().astype(int)

# Исправляем аномалии: где ремонт < постройки, приравниваем к постройке
anomaly_mask = clean_df["Remodeled year"] < clean_df["Year built"]
clean_df.loc[anomaly_mask, "Remodeled year"] = clean_df.loc[
    anomaly_mask, "Year built"
]

# Смотрим, к каким типам недвижимости относятся пропуски в Year built
missing_years = clean_df[clean_df["Year built"].isna()]
print(missing_years["property_type_clean"].value_counts(dropna=False))

house           13811
townhouse        2266
apartment        1086
land              949
other             194
multi_family      135
mobile             36
commercial          0
Name: property_type_clean, dtype: int64


Анализ распределения пропусков в признаке `Year built` показал, что 949 строк относятся к категории `land` (голая земля). Физически данные объекты не имеют года постройки. Для исключения искажения логики моделей для категории `land` пропуски в `Year built` и `Remodeled year` принудительно заполняются значением `0.0`. 

Для остальных типов недвижимости (где строения физически существуют, но данные утеряны) пропуски сохраняются для последующего заполнения через пространственные медианы.

In [103]:
# Выделяем маску для объектов типа land с пропущенным годом
land_mask = (clean_df["property_type_clean"] == "land") & (clean_df["Year built"].isna())

# Зануляем год постройки и год ремонта для чистой земли
clean_df.loc[land_mask, "Year built"] = 0.0
clean_df.loc[land_mask, "Remodeled year"] = 0.0

# Контрольный замер: сколько пропусков осталось в Year built
print(f"Осталось пропусков в Year built: {clean_df['Year built'].isna().sum()}")

Осталось пропусков в Year built: 17528


Для оставшихся 17528 жилых объектов пропуски в 'Year built' восстанавливаются по принципу локальной застройки: вычисляется медианный год постройки зданий внутри аналогичного географического индекса ('zipcode'). Как и с площадью, взглянем сначала на гипотезу глазами, чтобы убедиться, что связь имеется.

In [104]:
# Фильтруем эталонные объекты (где известны оба года и это не земля)
v_years = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["Year built"].notna())
    & (clean_df["Remodeled year"].notna())
].copy()

# Находим самый плотный по объектам ZIP-код для репрезентативного графика
top_zip_years = v_years["zipcode"].value_counts().idxmax()
df_years_plot = v_years[v_years["zipcode"] == top_zip_years]

# Строим парные графики для проверки гипотез возраста и реновации
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Распределение года постройки в топовом ZIP-коде
sns.histplot(
    data=df_years_plot, x="Year built", ax=ax1, bins=30, kde=True,
    color=sns.color_palette("Set2")[0], edgecolor="black", linewidth=0.5
)
ax1.set_title(
    f"Плотность застройки в {top_zip_years}: Распределение Year built",
    fontsize=11, pad=10
)
ax1.set_xlabel("Год постройки")
ax1.set_ylabel("Количество объектов")

# График 2: Связь года постройки и года ремонта
sns.scatterplot(
    data=df_years_plot, x="Year built", y="Remodeled year", ax=ax2,
    alpha=0.5, color=sns.color_palette("Set2")[2], edgecolor="black",
    linewidth=0.5, s=25
)
# Идеальная диагональ (если год постройки == году ремонта)
ax2.plot(
    [df_years_plot["Year built"].min(), df_years_plot["Year built"].max()],
    [df_years_plot["Year built"].min(), df_years_plot["Year built"].max()],
    color=sns.color_palette("Set2")[1], linestyle="--", linewidth=2
)
ax2.set_title(
    f"Взаимосвязь дат в {top_zip_years}: Year built vs Remodeled year",
    fontsize=11, pad=10
)
ax2.set_xlabel("Год постройки")
ax2.set_ylabel("Год ремонта / обновления")

plt.tight_layout()
plt.show();

Гистограмма распределения 'Year built' внутри одного ZIP-кода демонстрирует выраженный пик. Это наглядно подтверждает градостроительную практику в США: районы застраиваются массово и циклично. Следовательно, заполнение пропущенных годов постройки через локальную медиану индекса ('zipcode') является пространственно и исторически обоснованным.

На графике рассеяния отчетливо видно, что огромный массив точек лежит строго на диагональной линии (где 'Year built == Remodeled year'). Это визуально доказывает, что для большинства объектов без зафиксированного ремонта дата обновления совпадает с датой постройки. Точки, лежащие выше диагонали, отражают реальные исторические этапы модернизации жилого фонда района.

Физический смысл признака 'Year built' неприменим к категории 'land'. Включение их в общий пул привело бы к искусственному омоложению или состариванию жилых кварталов. Принято решение изолировать их, присвоив маркер 0.0.

In [105]:
# Маска для зданий/квартир с пропущенным годом постройки
bad_year_mask = (clean_df["property_type_clean"] != "land") & (
    clean_df["Year built"].isna()
)
total_empty = bad_year_mask.sum()

# Выделяем массив эталонных жилых объектов для расчета медиан
live_df = clean_df[
    (clean_df["property_type_clean"] != "land") & (clean_df["Year built"] > 0)
]

# Сбор медиан по связке география + тип объекта при наличии аналогов
# ставлю ограничение в 5 аналогов, ранее в площади использовала 3 для 3 групп цены,
# в данном случае квантиль цены не рассматривается, решила сделать более жесткое ограничение
zip_stats = live_df.groupby(["zipcode", "property_type_clean"])[
    "Year built"
].agg(["median", "count"])
good_zip_years = zip_stats.loc[zip_stats["count"] >= 5, "median"]

# Точечное восстановление через MultiIndex-маппинг по zipcode и типу объекта
sub_df = clean_df.loc[bad_year_mask]
map_keys = pd.MultiIndex.from_frame(sub_df[["zipcode", "property_type_clean"]])
clean_df.loc[bad_year_mask, "Year built"] = map_keys.map(good_zip_years)

# Бэкап-заполнение по городам. Ставлю тут более жесткое ограничение в 10 аналогов#
city_stats = live_df.groupby("city")["Year built"].agg(["median", "count"])
good_city_years = city_stats.loc[city_stats["count"] > 10, "median"]
clean_df["Year built"] = clean_df["Year built"].fillna(
    clean_df["city"].map(good_city_years)
)

# Выделяем маску тупиковых строк, не прошедших географические фильтры
trash_mask = (clean_df["property_type_clean"] != "land") & (
    clean_df["Year built"].isna()
)
deleted_count = trash_mask.sum()

# Удаляем невосстановленный хвост
clean_df = clean_df[~trash_mask].reset_index(drop=True)

# Синхронизация: если ремонта не было, год ремонта равен году постройки
no_remodeled = clean_df["Remodeled year"].isna()
clean_df.loc[no_remodeled, "Remodeled year"] = clean_df.loc[
    no_remodeled, "Year built"
]

# Итоговый контроль качества очистки и фиксация размерности матрицы
print(f"Всего скрытых пропусков в зданиях обнаружено: {total_empty}")
print(f"Успешно восстановлено по локальным группам: {total_empty - deleted_count}")
print(f"Удалено неинформативных строк: {deleted_count}")
print(f"Итоговая размерность матрицы clean_df: {clean_df.shape}")

Всего скрытых пропусков в зданиях обнаружено: 17528
Успешно восстановлено по локальным группам: 17415
Удалено неинформативных строк: 113
Итоговая размерность матрицы clean_df: (316015, 37)


В ходе пространственного анализа были успешно восстановлены утерянные годы застройки жилого фонда по локальным географическим аналогам, что позволило минимизировать системные потери и отсеять всего 113 неинформативных объектов. Все оставшиеся пропуски в признаке реновации ('Remodeled year') были математически точно синхронизированы с финальным годом постройки, что полностью устранило пустоты во временном блоке.

### Статус продажи ('status_clean')

In [106]:
# Смотрим на уникальные значения и их распределение, включая NaN
print('Категории статуса:')
print(clean_df["status_clean"].value_counts(dropna=False))
print(f'\nОбщее количество пропусков в status_clean: {clean_df["status_clean"].isna().sum()}')

Категории статуса:
active              181786
pending_contract     89429
NaN                  35331
distressed_sale       9469
Name: status_clean, dtype: int64

Общее количество пропусков в status_clean: 35331


Проверим природу пропусков в данных

In [107]:
# Проверяем, у каких типов недвижимости чаще всего пропущен статус
missing_status = clean_df[clean_df["status_clean"].isna()]
print("Распределение пропусков по типам недвижимости:")
print(missing_status["property_type_clean"].value_counts(normalize=True).round(2).head(5))

# Сравниваем медианную цену объектов с NaN и объектов с явными статусами
print("\nМедианная стоимость объектов в зависимости от статуса:")
print(clean_df.groupby("status_clean", dropna=False)["target"].median())

Распределение пропусков по типам недвижимости:
house           0.65
apartment       0.25
townhouse       0.08
multi_family    0.02
mobile          0.01
Name: property_type_clean, dtype: float64

Медианная стоимость объектов в зависимости от статуса:
status_clean
active              348900.0
distressed_sale     213324.0
pending_contract    342114.0
NaN                 335000.0
Name: target, dtype: float64


In [108]:
# Создаем временный датафрейм и заменяем NaN на понятную метку
group_df = clean_df[["status_clean", "property_type_clean", "target"]].copy()
group_df["status_clean"] = group_df["status_clean"].fillna("Пропущенные (NaN)")
top_types = group_df["property_type_clean"].value_counts().index
filtered_df = group_df[group_df["property_type_clean"].isin(top_types)]

# Агрегируем данные до медианы
medians_df = (
    filtered_df.groupby(["status_clean", "property_type_clean"])["target"]
    .median()
    .reset_index()
)
# Строим сгруппированную столбчатую диаграмму
plt.figure(figsize=(12, 5))
sns.barplot(
    data=medians_df, x="status_clean", y="target", hue="property_type_clean",
    edgecolor="black", linewidth=0.5
)
plt.title("Медианная стоимость объектов по статусам и типам недвижимости", pad=15)
plt.xlabel("Статус сделки (status_clean)")
plt.ylabel("Медианная цена ($)")
# Переносим легенду вправо за пределы графика
plt.legend(title="Тип недвижимости", bbox_to_anchor=(1.02, 1), loc="upper left")
# Форматирование оси Y в миллионы долларов
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, p: f"${x*1e-6:.2f}M" if x > 0 else "$0")
)
plt.tight_layout()
plt.show();

Первичный агрегированный анализ указывал на близость медианных цен пропущенных значений к категории 'active'. Однако глубокая декомпозиция данных с помощью сгруппированной столбчатой диаграммы в разрезе типов недвижимости опровергла гипотезу о случайном характере пропусков, где наблюдаются статистически значимые отклонения медианной стоимости от эталонных рыночных статусов.Итоговое решение: Во избежание смещения оценок моделей и размытия ценовых сигналов принято решение отказаться от заполнения пропусков доминирующей модой. Пропущенные значения в количестве 35 331 строки изолированы в самостоятельную категорию 'unknown'.

In [109]:
# Заполняем пропущенные статусы выделенной категорией "unknown"
clean_df["status_clean"] = clean_df["status_clean"].fillna("unknown")

# Контрольная проверка структуры признака после заполнения
print(clean_df["status_clean"].value_counts())
print(f"Осталось пропусков в status_clean: {clean_df['status_clean'].isna().sum()}")

active              181786
pending_contract     89429
unknown              35331
distressed_sale       9469
Name: status_clean, dtype: int64
Осталось пропусков в status_clean: 0


### Этажность ('stories_clean')

In [110]:
# Выводим текущее распределение интервальных категорий в stories_clean
print("Текущее распределение категорий в stories_clean:")
print(clean_df["stories_clean"].value_counts(dropna=False))

# Локализируем пропуски этажности по типам недвижимости
missing_stories = clean_df[clean_df["stories_clean"].isna()]
print("\nРаспределение пропусков этажности по типам объектов:")
print(missing_stories["property_type_clean"].value_counts())

Текущее распределение категорий в stories_clean:
1.0           149293
1.5_to_2       91006
NaN            45258
3.0            17557
8_and_more      7560
4_to_7          5341
Name: stories_clean, dtype: int64

Распределение пропусков этажности по типам объектов:
apartment       21671
house           10953
townhouse        7353
land             2965
mobile           1529
other             724
multi_family       60
commercial          3
Name: property_type_clean, dtype: int64


In [111]:
# Выделяем объекты с известной этажностью (без чистой земли)
v_live = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["stories_clean"].notna())
].copy()
df_simple = (
    v_live.groupby(["property_type_clean", "stories_clean"], observed=False)
    .size()
    .reset_index(name="counts")
)
df_simple["percentage"] = df_simple.groupby("property_type_clean")[
    "counts"
].transform(lambda x: (x / x.sum()) * 100)

# Строим одну сгруппированную диаграмму
plt.figure(figsize=(11, 5))
sns.barplot(
    data=df_simple, x="property_type_clean", y="percentage", hue="stories_clean",
    palette="Set2", edgecolor="black", linewidth=0.5
)
plt.title("Распределение интервалов этажности по типам недвижимости", pad=15)
plt.xlabel("Тип недвижимости (property_type_clean)")
plt.ylabel("Доля внутри типа объекта (%)")
plt.legend(title="Интервалы этажности", loc="upper right")
plt.ylim(0, 100)
plt.tight_layout()
plt.show();

Признак этажности здания является ключевым конструктивным и архитектурным параметром недвижимости, напрямую определяющим ее рыночную стоимость. На этапе предварительной очистки входных данных был реализован алгоритм перекрестной проверки ('колонки-доноры'). Из оригинальных полей 'stories' и 'propertyType' с помощью регулярных выражений была извлечена скрытая информация о типах объектов и их этажности, укрупненная в устойчивые интервалы рынка США ('1.0', '1.5_to_2', '3.0', '4_to_7', '8_and_more').

Визуальный анализ распределения категорий этажности по типам недвижимости подтверждает сильную структурную неоднородность рынка. Для мобильных домов ('mobile') и частного сектора ('house') характерно жесткое доминирование одного-двух малоэтажных интервалов, в то время как сегмент квартир ('apartment') демонстрирует равномерный разнос от одноуровневых объектов до высотной застройки.

На этапе заполнения пропусков выполняется точечное восстановление 'глухих' пустот. Логика импутации выстраивается на основе очищенных типов недвижимости ('property_type_clean'):

- Физический смысл этажности неприменим к незастроенным участкам. Для категории 'land' пропущенные значения принудительно кодируются уникальным маркером 0.0.
- Для жилых классов этажность продиктована локальными стандартами застройки конкретных кварталов. Восстановление данных производится через расчет локальной моды (наиболее частого интервала) для типа недвижимости внутри родного географического индекса ('zipcode').
- Чтобы защитить алгоритмы от единичных выбросов, установлен порог — не менее 5 объектов внутри одного ZIP-кода. В качестве бэкапа для редких локаций применяется вычисление моды по городу ('city') с ограничением на размер выборки более 10 объектов.
- Оставшиеся редкие строки, не прошедшие фильтры репрезентативности аналогов, удаляться не будут. Им принудительно присваивается текстовый статус 'unknown', что полностью сохраняет объем выборки и позволяет моделям самостоятельно выявить скрытые паттерны планировки в ходе обучения.

In [112]:
# Изолируем голую землю: присваиваем строковый маркер "0.0" внутри пропусков
land_mask = (clean_df["property_type_clean"] == "land") & (
    clean_df["stories_clean"].isna()
)
clean_df.loc[land_mask, "stories_clean"] = "0.0"

# Фиксируем маску оставшихся пропусков для жилых зданий
bad_stories = clean_df["stories_clean"].isna()
total_empty = bad_stories.sum()

# Функция для безопасного извлечения моды группы
get_mode = lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan

# Сбор надежных мод по связке география + тип объекта (где аналогов >= 5)
live_df = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["stories_clean"].notna())
]
zip_stats = live_df.groupby(["zipcode", "property_type_clean"]).agg(
    mode_val=("stories_clean", get_mode), count_val=("stories_clean", "count")
)
good_zip_modes = zip_stats.loc[zip_stats["count_val"] >= 5, "mode_val"]

# Точечное восстановление через MultiIndex-маппинг по zipcode и типу объекта
sub_df = clean_df.loc[bad_stories]
map_keys = pd.MultiIndex.from_frame(sub_df[["zipcode", "property_type_clean"]])
clean_df.loc[bad_stories, "stories_clean"] = map_keys.map(good_zip_modes)

# Бэкап-заполнение по городам с ограничением на размер выборки (> 10)
city_stats = live_df.groupby("city").agg(
    mode_val=("stories_clean", get_mode), count_val=("stories_clean", "count")
)
good_city_modes = city_stats.loc[city_stats["count_val"] > 10, "mode_val"]
clean_df["stories_clean"] = clean_df["stories_clean"].fillna(
    clean_df["city"].map(good_city_modes)
)

# Фиксируем тупиковый хвост, который не нашел аналогов по району и городу
trash_mask = (clean_df["property_type_clean"] != "land") & (
    clean_df["stories_clean"].isna()
)
unknown_count = trash_mask.sum()
# Принудительно переводим оставшиеся пропуски в категорию unknown
clean_df["stories_clean"] = clean_df["stories_clean"].fillna("unknown")

# Итоговый контроль качества очистки и фиксация структуры признака
print(f"Осталось пропусков в stories_clean: {clean_df['stories_clean'].isna().sum()}")
print(f"Всего скрытых пропусков в зданиях обнаружено: {total_empty}")
print(f"Успешно восстановлено по локальным группам: {total_empty - unknown_count}")
print(f"Переведено в категорию unknown: {unknown_count}")

Осталось пропусков в stories_clean: 0
Всего скрытых пропусков в зданиях обнаружено: 42293
Успешно восстановлено по локальным группам: 42058
Переведено в категорию unknown: 235


In [113]:
# Проверяем и удаляем дубликаты, возникшие после фильтрации этажности
dup_stories = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов после блока этажности: {dup_stories}")

clean_df = clean_df.drop_duplicates().reset_index(drop=True)
# Итоговая фиксация размера матрицы
print(f"Новая общая размерность матрицы clean_df: {clean_df.shape}")

Обнаружено полных дубликатов после блока этажности: 21
Новая общая размерность матрицы clean_df: (315994, 37)


### Исследование климатических систем объектов ('heating_clean' и 'cooling_clean')

Признаки отопления и кондиционирования отражают инженерно-техническое оснащение объектов недвижимости. Учитывая технологическую взаимосвязь климатического оборудования (интегрированные системы HVAC), данные признаки анализируются совместно. Перед заполнением пропусков проводится экспресс-диагностика: оценивается структура текстовых категорий, локализуются пропуски в разрезе типов недвижимости и проверяется их логическое пересечение.

In [114]:
# Замеряем точные объемы «глухих» пропусков в обеих климатических колонках
print(f"Пропусков в отоплении (heating_clean): {clean_df['heating_clean'].isna().sum()}")
print(f"Пропусков в охлаждении (cooling_clean): {clean_df['cooling_clean'].isna().sum()}")

# Посмотрим на пропуски климатического блока в разрезе типов недвижимости
missing_climate = clean_df[
    clean_df["heating_clean"].isna() | clean_df["cooling_clean"].isna()
]
print("\nРаспределение пропусков климата по типам объектов:")
print(missing_climate["property_type_clean"].value_counts())

Пропусков в отоплении (heating_clean): 68754
Пропусков в охлаждении (cooling_clean): 109931

Распределение пропусков климата по типам объектов:
house           85425
apartment       16136
townhouse        9647
multi_family     8211
other            3767
land             3361
mobile           1096
commercial          1
Name: property_type_clean, dtype: int64


In [115]:
# Считаем объекты, где пропущено вообще всё (и отопление, и охлаждение)
both_missing = clean_df["heating_clean"].isna() & clean_df["cooling_clean"].isna()

# Считаем объекты, где пропущена только одна из систем (частичная утеря)
only_one_missing = (clean_df["heating_clean"].isna() ^ clean_df["cooling_clean"].isna())

print(f"Полные климатические пропуски (NaN в обеих колонках): {both_missing.sum()}")
print(f"Частичные пропуски (NaN только в одной из колонок): {only_one_missing.sum()}")

# Смотрим, сколько полных пропусков приходится на голую землю
land_both_missing = both_missing & (clean_df["property_type_clean"] == "land")
print(f"Из них полных пропусков на незастроенных участках (land): {land_both_missing.sum()}")

Полные климатические пропуски (NaN в обеих колонках): 51041
Частичные пропуски (NaN только в одной из колонок): 76603
Из них полных пропусков на незастроенных участках (land): 1556


Проверим гипотезы о взаимосвязи типа климатической системы от местоположения (штата). Предполагаю, что например в Калифорнии люди пользуются только кондиционерами, в других холодных штатах наоборот пользуются исключительно отоплением. Для этого используем критерий независимости Хи-квадрат Пирсона (Оба исследуемых показателя — являются качественными (категориальными) признаками, анализ охватывает множественные независимые группы, что исключает применение парных критериев). Проверим следующие гипотезы:

**Гипотеза 1:**

- Нулевая гипотеза ($H_0$): Тип отопительного оборудования (`heating_clean`) и географическое положение объекта (`state`) являются независимыми величинами. Распределение систем отопления носит случайный характер по всей территории страны.
- Альтернативная гипотеза ($H_1$): Между типом отопительного оборудования (`heating_clean`) и географическим положением объекта (`state`) существует статистически значимая взаимосвязь.

**Гипотеза 2:**

- Нулевая гипотеза ($H_0$): Тип кондиционирования (`cooling_clean`) и географическое положение объекта (`state`) являются независимыми величинами. Распределение систем кондиционирования носит случайный характер по всей территории страны.
- Альтернативная гипотеза ($H_1$): Между типом кондиционирования (`cooling_clean`) и географическим положением объекта (`state`) существует статистически значимая взаимосвязь.

**Гипотеза 3:**

- Нулевая гипотеза ($H_0$): Тип отопительного оборудования (`heating_clean`) и тип кондиционирования (`cooling_clean`) являются независимыми величинами. Распределение систем отопления носит случайный характер по всей территории страны.
- Альтернативная гипотеза ($H_1$): Между типом отопительного оборудования (`heating_clean`) и типом кондиционирования (`cooling_clean`) существует статистически значимая взаимосвязь.

In [116]:
# Выделяем жилые объекты с известными системами отопления
live_heat = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["heating_clean"].notna())
]

# Выделяем жилые объекты с известными системами кондиционирования
live_cool = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["cooling_clean"].notna())
]

# Выделяем жилые объекты, где известны одновременно оба признака
live_both = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["heating_clean"].notna())
    & (clean_df["cooling_clean"].notna())
]

# Строим таблицу сопряженности между штатами и типами отопления
table_1 = pd.crosstab(live_heat["state"], live_heat["heating_clean"])
table_2 = pd.crosstab(live_cool["state"], live_cool["cooling_clean"])
table_3 = pd.crosstab(live_both["heating_clean"], live_both["cooling_clean"])

# Проводим тест Хи-квадрат
_, p_val_1, _, _ = stats.chi2_contingency(table_1)
_, p_val_2, _, _ = stats.chi2_contingency(table_2)
_, p_val_3, _, _ = stats.chi2_contingency(table_3)

# Смотрим результат
# Гипотеза 1
print('H_0: тип отопления и географическое положение объекта являются независимыми:')
get_gip(p_val_1)
# Гипотеза 2
print('\nH_0: тип кондиционирования и географическое положение объекта являются независимыми:')
get_gip(p_val_2)
# Гипотеза 3
print('\nH_0: тип отопления и тип кондиционирования объекта являются независимыми:')
get_gip(p_val_3)

H_0: тип отопления и географическое положение объекта являются независимыми:
p-value = 0.000
p-значение меньше, чем заданный уровень значимости 0.05. Отвергаем нулевую гипотезу.

H_0: тип кондиционирования и географическое положение объекта являются независимыми:
p-value = 0.000
p-значение меньше, чем заданный уровень значимости 0.05. Отвергаем нулевую гипотезу.

H_0: тип отопления и тип кондиционирования объекта являются независимыми:
p-value = 0.000
p-значение меньше, чем заданный уровень значимости 0.05. Отвергаем нулевую гипотезу.


Анализ диагностических данных выявил 51 041 случай полного отсутствия климатических параметров и 76 604 случая частичной утери данных, когда зафиксирован только один компонент системы (отопление или кондиционер). Проведение статистического критерия независимости Хи-квадрат Пирсона подтвердило жесткую взаимосвязь инженерного оснащения между собой и с географическим положением. Чтобы визуально продемонстрировать характер выявленной географической зависимости перед заполнением пропусков, построим простой график распределения систем отопления по ключевым штатам США.

In [117]:
# 1. Выделяем жилые объекты, где известны одновременно оба признака (без чистой земли)
live_both = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["heating_clean"].notna())
    & (clean_df["cooling_clean"].notna())
].copy()

# 2. Автоматически находим топ-5 самых крупных ZIP-кодов в этом объединенном срезе
unique_zips = live_both["zipcode"].drop_duplicates()
random_zips = unique_zips.sample(5, random_state=11).values

df_zips = live_both[live_both["zipcode"].isin(random_zips)].copy()

# Формируем красивую объединенную текстовую метку для оси X
df_zips["zip_label"] = df_zips["zipcode"].astype(str) + " (" + df_zips["state"].astype(str) + ")"

# 3. Считаем процентное распределение систем отопления внутри каждого района
chart_heat = (
    df_zips.groupby(["zip_label", "heating_clean"], observed=False)
    .size()
    .reset_index(name="counts")
)
chart_heat["percentage"] = chart_heat.groupby("zip_label")["counts"].transform(
    lambda x: (x / x.sum()) * 100
)

# 4. Считаем процентное распределение систем кондиционирования внутри тех же районов
chart_cool = (
    df_zips.groupby(["zip_label", "cooling_clean"], observed=False)
    .size()
    .reset_index(name="counts")
)
chart_cool["percentage"] = chart_cool.groupby("zip_label")["counts"].transform(
    lambda x: (x / x.sum()) * 100
)

# 5. Строим два лаконичных графика друг под другом в формате 10 на 6
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))

# Верхний график: Отопление
sns.barplot(
    data=chart_heat, x="zip_label", y="percentage", hue="heating_clean",
    ax=ax1, palette="Set2", edgecolor="black", linewidth=0.5
)
ax1.set_title("Локальный профиль систем отопления по районам США", pad=10)
ax1.set_xlabel("Почтовый индекс и штат (zipcode, state)")
ax1.set_ylabel("Доля внутри района (%)")
ax1.set_ylim(0, 100)
ax1.legend(title="Тип отопления", bbox_to_anchor=(1.02, 1), loc="upper left")

# Нижний график: Кондиционирование
sns.barplot(
    data=chart_cool, x="zip_label", y="percentage", hue="cooling_clean",
    ax=ax2, palette="Set2", edgecolor="black", linewidth=0.5
)
ax2.set_title("Локальный профиль систем кондиционирования по районам США", pad=10)
ax2.set_xlabel("Почтовый индекс и штат (zipcode, state)")
ax2.set_ylabel("Доля внутри района (%)")
ax2.set_ylim(0, 100)
ax2.legend(title="Тип охлаждения", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show();

Этот график на случайных районах отлично показывает, почему нельзя заполнять пропуски одним значением для всей страны. В каждом месте действуют свои правила застройки. Например, в Мичигане (48310) и Неваде (89102) абсолютно у всех домов установлено воздушное отопление ('forced_air') и центральное кондиционирование ('central_ac'). В жаркой Аризоне (85033), наоборот, преобладает электрическое отопление ('electric'), а во Флориде (33612) наблюдается сильная диверсификация систем. Все это доказывает, что инженерные стандарты жестко привязаны к климату, и правильнее всего заполнять пропуски именно по локальной моде конкретного района ('zipcode').

Логика совместного заполнения оставшихся пропусков жилого фонда разделяется на три последовательных этапа:

- Для категории 'land' физический смысл климатического оснащения неприменим. Все пропущенные значения в обоих признаках для пустой земли принудительно кодируются маркерами полного отсутствия оборудования — 'no_heating' и 'no_cooling'.
- Все остальные пропуски заполняются по моде конкретного района ('zipcode') со строгим ограничением не менее 5 аналогов. Если аналог не найден, расчет моды производится по городу ('city') с ограничением от 10 аналогов. Если мода не найдена и в городе, подставляется мода по штату ('state') с порогом репрезентативности от 30 аналогов.
- Оставшиеся, не прошедшие отбор строки удаляться не будут. Им принудительно присваивается статус 'no_heating' или 'no_cooling' как наиболее обоснованный, поскольку отсутствие записи в листинге для редких локаций чаще всего свидетельствует о физическом отсутствии систем.

In [118]:
# Зануляем незастроенные участки (land): ставим no_heating и no_cooling
clean_df.loc[land_both_missing, "heating_clean"] = "no_heating"
clean_df.loc[land_both_missing, "cooling_clean"] = "no_cooling"

# Проверяем промежуточные остатки пропусков после первого этапа
print(f"Осталось пропусков в heating_clean: {clean_df['heating_clean'].isna().sum()}")
print(f"Осталось пропусков в cooling_clean: {clean_df['cooling_clean'].isna().sum()}")

Осталось пропусков в heating_clean: 67198
Осталось пропусков в cooling_clean: 108375


In [119]:
# Функция для безопасного извлечения моды группы
get_mode = lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan

# Выделяем массив эталонных жилых объектов для расчета моды района
live_df = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["heating_clean"].notna())
    & (clean_df["cooling_clean"].notna())
]

# Восстановление отопления по зип-коду
bad_heat = clean_df["heating_clean"].isna()
zip_heat = live_df.groupby(["zipcode", "property_type_clean"]).agg(
    mode_val=("heating_clean", get_mode), count_val=("heating_clean", "count")
)
good_zip_heat = zip_heat.loc[zip_heat["count_val"] >= 5, "mode_val"]
map_keys_heat = pd.MultiIndex.from_frame(
    clean_df.loc[bad_heat, ["zipcode", "property_type_clean"]]
)
clean_df.loc[bad_heat, "heating_clean"] = map_keys_heat.map(good_zip_heat)

# Восстановление кондиционирования по зип-коду
bad_cool = clean_df["cooling_clean"].isna()
zip_cool = live_df.groupby(["zipcode", "property_type_clean"]).agg(
    mode_val=("cooling_clean", get_mode), count_val=("cooling_clean", "count")
)
good_zip_cool = zip_cool.loc[zip_cool["count_val"] >= 5, "mode_val"]
map_keys_cool = pd.MultiIndex.from_frame(
    clean_df.loc[bad_cool, ["zipcode", "property_type_clean"]]
)
clean_df.loc[bad_cool, "cooling_clean"] = map_keys_cool.map(good_zip_cool)

# Замеряем остатки после первого уровня географического каскада
print(f"Осталось пропусков в heating_clean: {clean_df['heating_clean'].isna().sum()}")
print(f"Осталось пропусков в cooling_clean: {clean_df['cooling_clean'].isna().sum()}")

Осталось пропусков в heating_clean: 16436
Осталось пропусков в cooling_clean: 28067


Отлично, большая часть данных была заполнена, оставшиеся значения обработаем внутри одинаковых городов, с ограничением в 10 аналогов.

In [120]:
# Восстановление отопления по городу
bad_heat = clean_df["heating_clean"].isna()
city_heat = live_df.groupby(["city", "property_type_clean"]).agg(
    mode_val=("heating_clean", get_mode), count_val=("heating_clean", "count")
)
good_city_heat = city_heat.loc[city_heat["count_val"] >= 10, "mode_val"]
map_keys_city_heat = pd.MultiIndex.from_frame(
    clean_df.loc[bad_heat, ["city", "property_type_clean"]]
)
clean_df.loc[bad_heat, "heating_clean"] = map_keys_city_heat.map(good_city_heat)

# Восстановление кондиционирования по городу
bad_cool = clean_df["cooling_clean"].isna()
city_cool = live_df.groupby(["city", "property_type_clean"]).agg(
    mode_val=("cooling_clean", get_mode), count_val=("cooling_clean", "count")
)
good_city_cool = city_cool.loc[city_cool["count_val"] >= 10, "mode_val"]
map_keys_city_cool = pd.MultiIndex.from_frame(
    clean_df.loc[bad_cool, ["city", "property_type_clean"]]
)
clean_df.loc[bad_cool, "cooling_clean"] = map_keys_city_cool.map(good_city_cool)

# Замеряем промежуточные остатки пропусков после уровня городов
print(f"Осталось пропусков в heating_clean: {clean_df['heating_clean'].isna().sum()}")
print(f"Осталось пропусков в cooling_clean: {clean_df['cooling_clean'].isna().sum()}")

Осталось пропусков в heating_clean: 6992
Осталось пропусков в cooling_clean: 12118


И последним этапом проводим проверку внутри штата (ограничение >= 30 аналогов). Для оставшегося массива присвоим статус отсутствия инженерных систем.

In [121]:
# Восстановление отопления по штату
bad_heat = clean_df["heating_clean"].isna()
state_heat = live_df.groupby(["state", "property_type_clean"]).agg(
    mode_val=("heating_clean", get_mode), count_val=("heating_clean", "count")
)
good_state_heat = state_heat.loc[state_heat["count_val"] >= 30, "mode_val"]
map_keys_state_heat = pd.MultiIndex.from_frame(
    clean_df.loc[bad_heat, ["state", "property_type_clean"]]
)
clean_df.loc[bad_heat, "heating_clean"] = map_keys_state_heat.map(good_state_heat)

# Восстановление кондиционирования по штату
bad_cool = clean_df["cooling_clean"].isna()
state_cool = live_df.groupby(["state", "property_type_clean"]).agg(
    mode_val=("cooling_clean", get_mode), count_val=("cooling_clean", "count")
)
good_state_cool = state_cool.loc[state_cool["count_val"] >= 30, "mode_val"]
map_keys_state_cool = pd.MultiIndex.from_frame(
    clean_df.loc[bad_cool, ["state", "property_type_clean"]]
)
clean_df.loc[bad_cool, "cooling_clean"] = map_keys_state_cool.map(good_state_cool)

# Первый принт: замеряем остатки строго после отработки каскада штатов
print("После отработки штата:")
print(f"- Не восстановлено в heating_clean: {clean_df['heating_clean'].isna().sum()}")
print(f"- Не восстановлено в cooling_clean: {clean_df['cooling_clean'].isna().sum()}")

# Перевод остатков в категорию нет систем
clean_df["heating_clean"] = clean_df["heating_clean"].fillna("no_heating")
clean_df["cooling_clean"] = clean_df["cooling_clean"].fillna("no_cooling")

# Итоговый контроль закрытия климатического блока в ноль
print("\nПосле перевода остатков в статус отсутствия:")
print(f"- Остаток пропусков в heating_clean: {clean_df['heating_clean'].isna().sum()}")
print(f"- Остаток пропусков в cooling_clean: {clean_df['cooling_clean'].isna().sum()}")

После отработки штата:
- Не восстановлено в heating_clean: 2718
- Не восстановлено в cooling_clean: 4737

После перевода остатков в статус отсутствия:
- Остаток пропусков в heating_clean: 0
- Остаток пропусков в cooling_clean: 0


In [122]:
# Проверяем и удаляем дубликаты, возникшие после фильтрации климат.систем
dup_climate = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов после блока климатических систем: {dup_climate}")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
# Итоговая фиксация размера матрицы перед переходом к комнатам
print(f"Новая общая размерность матрицы clean_df: {clean_df.shape}")

Обнаружено полных дубликатов после блока климатических систем: 25
Новая общая размерность матрицы clean_df: (315969, 37)


### Исследование характеристик жилой площади ('beds_clean' и 'baths_clean')

Количество спален ('beds_clean') и санузлов ('baths_clean') являются базовыми количественными характеристиками жилой площади. Перед выбором алгоритма восстановления пропусков необходимо провести экспресс-диагностику: оценить объемы пропущенных значений, изучить диапазоны физических величин и проверить структуру данных на наличие аномальных выбросов.

In [123]:
# Проверяем точное количество пропусков в обеих колонках на текущий момент
print(f"Пропусков в спальнях (beds_clean): {clean_df['beds_clean'].isna().sum()}")
print(f"Пропусков в санузлах (baths_clean): {clean_df['baths_clean'].isna().sum()}")

# Выводим базовую описательную статистику (минимумы, максимумы, средние)
print("\nОписательная статистика для комнат и санузлов:")
print(clean_df[["beds_clean", "baths_clean"]].describe())

Пропусков в спальнях (beds_clean): 57883
Пропусков в санузлах (baths_clean): 57517

Описательная статистика для комнат и санузлов:
          beds_clean    baths_clean
count  258086.000000  258452.000000
mean        3.375844       3.366004
std         1.482973      21.446914
min         1.000000       0.000000
25%         3.000000       2.000000
50%         3.000000       2.500000
75%         4.000000       3.000000
max       144.000000     750.000000


Максимумы (max): 144 спальни и 750 санузлов — это явные технические ошибки риелторов или случайные склейки при парсинге. Стандартное отклонение: у санузлов оно взлетело до 21.5. Это математически доказывает, что в данных сидит экстремальный шум, который будет сильно переобучать модель. Помимо заполнения пропусков данный признак необходимо очистить от шума и переопределить в более удобоваримые категории. Для начала отделим от данных чистую землю, у которой по определению данный признак должен быть нулевой.

In [124]:
# Выделяем маску пустой земли, где в комнатах стоят честные пропуски (NaN)
land_beds_nan = (clean_df["property_type_clean"] == "land") & (clean_df["beds_clean"].isna())
land_baths_nan = (clean_df["property_type_clean"] == "land") & (clean_df["baths_clean"].isna())

# Зануляем пропуски комнат и санузлов строго на этой земле
clean_df.loc[land_beds_nan, "beds_clean"] = 0.0
clean_df.loc[land_baths_nan, "baths_clean"] = 0.0

# Замеряем, сколько пропусков осталось в жилых и коммерческих зданиях
print(f"Осталось пропусков в спальнях (beds_clean): {clean_df['beds_clean'].isna().sum()}")
print(f"Осталось пропусков в санузлах (baths_clean): {clean_df['baths_clean'].isna().sum()}")

Осталось пропусков в спальнях (beds_clean): 55577
Осталось пропусков в санузлах (baths_clean): 53608


На данном этапе работы заполнение пропусков выполняется исключительно для категории земельных участков ('land'). При этом в исходном датасете зафиксированы аномальные случаи, когда для незастроенной земли риелторами ошибочно указывались физические параметры строений: количество спален, санузлов, этажность и год постройки. В рамках текущего шага обрабатываются строго утерянные значения ('NaN'), а к системному аудиту и зачистке логических противоречий между заполненными признаками мы вернемся позже. Далее рассмотрим структуру оставшихся пропусков в разрезе типов жилой и коммерческой недвижимости.

In [125]:
# Проверяем распределение оставшихся пропусков спален по типам зданий
remaining_beds_nan = clean_df[clean_df["beds_clean"].isna()]
print("Остаток пропусков спален по типам объектов:")
print(remaining_beds_nan["property_type_clean"].value_counts())

# Проверяем распределение оставшихся пропусков санузлов по типам зданий
remaining_baths_nan = clean_df[clean_df["baths_clean"].isna()]
print("\nОстаток пропусков санузлов по типам объектов:")
print(remaining_baths_nan["property_type_clean"].value_counts())

Остаток пропусков спален по типам объектов:
house           32373
apartment       11186
other            7969
multi_family     2450
townhouse        1490
mobile            106
commercial          3
land                0
Name: property_type_clean, dtype: int64

Остаток пропусков санузлов по типам объектов:
house           32806
apartment       10218
other            6015
multi_family     2464
townhouse        1866
mobile            236
commercial          3
land                0
Name: property_type_clean, dtype: int64


Очевидно, что для коммерческих объектов ни количество спален, ни количество санузлов не являются ценообразующими параметрами, этот тип недвижимости тут можно занулить. Также очевидно и то, что количество спален и санузлов, во-первых, коррелируют между собой, а во-вторых, сильно зависят от площади объекта. Убедимся в этом визуально на графике.

In [126]:
# Находим маски пропусков комнат и ванн именно для коммерческих объектов
comm_beds_nan = (clean_df["property_type_clean"] == "commercial") & (
    clean_df["beds_clean"].isna()
)
comm_baths_nan = (clean_df["property_type_clean"] == "commercial") & (
    clean_df["baths_clean"].isna()
)

# Принудительно зануляем коммерческий сегмент внутри пропусков
clean_df.loc[comm_beds_nan, "beds_clean"] = 0.0
clean_df.loc[comm_baths_nan, "baths_clean"] = 0.0

In [127]:
#  Выделяем жилые объекты с известными параметрами (без земли и коммерции)
live_types = ["house", "apartment", "multi_family", "townhouse", "other"]
live_rooms = clean_df[
    (clean_df["property_type_clean"].isin(live_types))
    & (clean_df["beds_clean"].notna())
    & (clean_df["baths_clean"].notna())
    & (clean_df["sqft"] > 0)
].copy()

# Округляем площадь до стабильных интервалов по 500 кв. футов
live_rooms["sqft_bin"] = (live_rooms["sqft"] // 500) * 500

# Строим сетку графиков друг под другом
fig, axes = plt.subplots(len(live_types), 1, figsize=(8, 16))
# Запускаем автоматический цикл по типам недвижимости
for idx, p_type in enumerate(live_types):
    # Изолируем текущий тип недвижимости по всей стране
    df_type_all = live_rooms[live_rooms["property_type_clean"] == p_type]
    # Находим самый популярный штат именно для этого типа объекта
    top_state_for_type = df_type_all["state"].value_counts().idxmax()
    # Фильтруем данные строго по этому штату-лидеру
    df_sub = df_type_all[df_type_all["state"] == top_state_for_type]
    # Отсекаем редкие огромные выбросы по площади (95-й квантиль)
    if not df_sub.empty:
        max_bin_limit = df_sub["sqft_bin"].quantile(0.95)
        df_sub_filtered = df_sub[df_sub["sqft_bin"] <= max_bin_limit]
        # Считаем средние показатели комнат
        df_line = (
            df_sub_filtered.groupby("sqft_bin")
            .agg(mean_beds=("beds_clean", "mean"), mean_baths=("baths_clean", "mean"))
            .reset_index()
        )
        # Строим линию для спален
        sns.lineplot(
            data=df_line, x="sqft_bin", y="mean_beds", ax=axes[idx],
            marker="o", linewidth=1.5, label="Спальни"
        )
        # Строим линию для санузлов
        sns.lineplot(
            data=df_line, x="sqft_bin", y="mean_baths", ax=axes[idx],
            marker="s", linewidth=1.5, label="Санузлы"
        )
    # Оформление
    axes[idx].set_title(
        f"Стандарт планировки для {p_type.upper()} в штате {top_state_for_type}"
    )
    axes[idx].set_xlabel("Интервалы площади (sqft)")
    axes[idx].set_ylabel("Среднее количество (шт.)")
    axes[idx].grid(True, linestyle="--", alpha=0.4)
    axes[idx].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show();

Визуальный анализ графиков наглядно доказывает жесткую зависимость количества комнат от общей площади здания ('sqft'). При этом у каждого типа недвижимости прослеживается свой планировочный стандарт:

- В индивидуальных домах ('house') количество спален стабильно превышает число санузлов, что характерно для классического семейного формата жилья.
- В сегменте квартир ('apartment') на больших площадях количество санузлов начинает опережать число спален, отражая современные американские стандарты повышенной комфортности (отдельная ванная при каждой спальне плюс гостевой туалет).
- В таунхаусах ('townhouse') линии идут практически стык-в-стык, подтверждая жесткую типовую планировку, где на каждую спальню застройщик закладывает один полноценный санузел.

Небольшой провал линий в самом начале графиков для многосемейных домов ('multi_family') и категории 'other' объясняется спецификой переделанных под жилье коммерческих зданий (лофтов) или доходных домов, где при минимальной площади строения может быть сразу несколько изолированных комнат или санузлов (гестхаусы, отели). Полученные линейные тренды математически доказывают, что восстанавливать пропуски комнат необходимо строго с привязкой к площади объектов. Прежде чем приступить к заполнению данных, надо обработать выбросы.

In [128]:
# Выделяем жилые объекты с известными числовыми параметрами (без земли и коммерции)
live_rooms = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["property_type_clean"] != "commercial")
    & (clean_df["beds_clean"].notna())
    & (clean_df["baths_clean"].notna())
]

# Рассчитываем 99.5-й квантиль, чтобы нащупать реальный верхний предел рынка США
limit_beds = live_rooms["beds_clean"].quantile(0.995)
limit_baths = live_rooms["baths_clean"].quantile(0.995)

print(f"99.5% жилых объектов имеют спален не более: {limit_beds}")
print(f"99.5% жилых объектов имеют санузлов не более: {limit_baths}")

# Заодно посмотрим, сколько у нас объектов, которые превышают эти границы
over_beds = (live_rooms["beds_clean"] > limit_beds).sum()
over_baths = (live_rooms["baths_clean"] > limit_baths).sum()

print(f"\nКоличество аномальных строк выше этой границы по спальням: {over_beds}")
print(f"Количество аномальных строк выше этой границы по санузлам: {over_baths}")

99.5% жилых объектов имеют спален не более: 9.0
99.5% жилых объектов имеют санузлов не более: 9.0

Количество аномальных строк выше этой границы по спальням: 910
Количество аномальных строк выше этой границы по санузлам: 983


Статистика квантилей показала, что 99.5% всего жилого фонда имеют не более 9 спален и 9 санузлов. Чтобы защитить модель от экстремального риелторского шума, вроде 750 санузлов, но при этом полностью сохранить ценные данные о премиум-недвижимости, мы не будем удалять эти строки. Вместо этого числовые признаки переводятся в устойчивые текстовые интервалы, где все аномальные значения укрупняются в финальный класс '9_and_more'.

In [129]:
# Ограничиваем верхние значения комнат и санузлов для жилого фонда по порогу 9.0
clean_df.loc[clean_df["beds_clean"] > 9.0, "beds_clean"] = 9.0
clean_df.loc[clean_df["baths_clean"] > 9.0, "baths_clean"] = 9.0

# Проверяем, что экстремальный шум (144 и 750) успешно исчез из основной матрицы
print(f"Новый максимум по спальням (beds_clean): {clean_df['beds_clean'].max()}")
print(f"Новый максимум по санузлам (baths_clean): {clean_df['baths_clean'].max()}")

Новый максимум по спальням (beds_clean): 9.0
Новый максимум по санузлам (baths_clean): 9.0


Логика совместного заполнения оставшихся пропусков в жилых зданиях выстраивается по каскадной схеме с жесткой привязкой к габаритам строений:

- Шаг планировочных коридоров фиксируется на уровне 500 кв. футов ('sqft_bin'), так как визуальный анализ трендов подтвердил прирост комнат и санузлов именно на этом интервале.
- На первом этапе (уровень ZIP-кода) порог репрезентативности аналогов фиксируется на уровне не менее 3 объектов. Применение математической моды вместо среднего арифметического гарантирует защиту от риелторского шума, так как алгоритм выберет наиболее частое значение планировки в группе, игнорируя единичные выбросы.
- Не нашедшие аналогов строки передаются на уровни бэкапа по городу и штату, где за счет масштаба территорий ограничения по количеству аналогов будут последовательно увеличиваться.

In [130]:
# Функция для безопасного извлечения числовой моды группы
get_mode = lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan

# Считаем планировочный шаг площади всего один раз для всего датасета
sqft_bin_all = (clean_df["sqft"] // 500) * 500
# Выделяем массив эталонных жилых объектов для расчета моды
live_df = clean_df[
    (clean_df["property_type_clean"] != "land")
    & (clean_df["beds_clean"].notna())
    & (clean_df["baths_clean"].notna())
].copy()
# Подтягиваем готовую серию площади в таблицу-донор по ее оригинальному индексу
live_df["sqft_bin"] = sqft_bin_all

# Восстановление спален по zip-коду + площади (порог >= 3)
bad_beds = clean_df["beds_clean"].isna()
zip_beds = live_df.groupby(["zipcode", "property_type_clean", "sqft_bin"]).agg(
    mode_val=("beds_clean", get_mode), count_val=("beds_clean", "count")
)
good_zip_beds = zip_beds.loc[zip_beds["count_val"] >= 3, "mode_val"]

map_keys_beds = pd.MultiIndex.from_arrays([
    clean_df.loc[bad_beds, "zipcode"],
    clean_df.loc[bad_beds, "property_type_clean"],
    sqft_bin_all.loc[bad_beds]
])
clean_df.loc[bad_beds, "beds_clean"] = map_keys_beds.map(good_zip_beds)

# Восстановление санузлов по zip-коду + площади (порог >= 3)
bad_baths = clean_df["baths_clean"].isna()
zip_baths = live_df.groupby(["zipcode", "property_type_clean", "sqft_bin"]).agg(
    mode_val=("baths_clean", get_mode), count_val=("baths_clean", "count")
)
good_zip_baths = zip_baths.loc[zip_baths["count_val"] >= 3, "mode_val"]

map_keys_baths = pd.MultiIndex.from_arrays([
    clean_df.loc[bad_baths, "zipcode"],
    clean_df.loc[bad_baths, "property_type_clean"],
    sqft_bin_all.loc[bad_baths]
])
clean_df.loc[bad_baths, "baths_clean"] = map_keys_baths.map(good_zip_baths)

# Замеряем промежуточные остатки пропусков
print(f"Осталось пропусков в beds_clean: {clean_df['beds_clean'].isna().sum()}")
print(f"Осталось пропусков в baths_clean: {clean_df['baths_clean'].isna().sum()}")

Осталось пропусков в beds_clean: 17328
Осталось пропусков в baths_clean: 16359


Необработанный остаток восстанавливаем по городам с ограничением не менее 5 аналогов.

Здесь стоит пояснить, почему для спален и санузлов мы берем менее жесткие ограничения (3 аналога для ZIP-кода и 5 аналогов для города), чем это было в блоке климат-контроля. Так как все данные теперь дополнительно делятся на группы по площади, общее количество подгрупп резко возрастает. На всю территориальную выборку получается необходимо собрать слишком много аналогов, что физически невозможно. Но и слишком слабые фильтры делать нельзя, чтобы не зацепить случайный риелторский шум. Порог в 5 объектов является оптимальным компромиссом для такой сложной группировки на уровне городов.

In [131]:
# Восстановление спален по городу + площади (порог >= 5)
bad_beds = clean_df["beds_clean"].isna()
city_beds = live_df.groupby(["city", "property_type_clean", "sqft_bin"]).agg(
    mode_val=("beds_clean", get_mode), count_val=("beds_clean", "count")
)
good_city_beds = city_beds.loc[city_beds["count_val"] >= 5, "mode_val"]

map_keys_city_beds = pd.MultiIndex.from_arrays([
    clean_df.loc[bad_beds, "city"],
    clean_df.loc[bad_beds, "property_type_clean"],
    sqft_bin_all.loc[bad_beds]
])
clean_df.loc[bad_beds, "beds_clean"] = map_keys_city_beds.map(good_city_beds)

# Восстановление санузлов по городу + площади (порог >= 5)
bad_baths = clean_df["baths_clean"].isna()
city_baths = live_df.groupby(["city", "property_type_clean", "sqft_bin"]).agg(
    mode_val=("baths_clean", get_mode), count_val=("baths_clean", "count")
)
good_city_baths = city_baths.loc[city_baths["count_val"] >= 5, "mode_val"]

map_keys_city_baths = pd.MultiIndex.from_arrays([
    clean_df.loc[bad_baths, "city"],
    clean_df.loc[bad_baths, "property_type_clean"],
    sqft_bin_all.loc[bad_baths]
])
clean_df.loc[bad_baths, "baths_clean"] = map_keys_city_baths.map(good_city_baths)

# Замеряем промежуточные остатки пропусков после уровня городов
print(f"Осталось пропусков в beds_clean: {clean_df['beds_clean'].isna().sum()}")
print(f"Осталось пропусков в baths_clean: {clean_df['baths_clean'].isna().sum()}")

Осталось пропусков в beds_clean: 11821
Осталось пропусков в baths_clean: 9895


Я полагаю, что в данных имеются районы и города, где представлено слишком мало аналогов по конкретным габаритам жилья, из-за чего существенная часть пропусков на прошлых этапах не отработала. Однако удалять строки только из-за отсутствия упоминания количества спален или санузлов нецелесообразно. Дальнейшая обработка данных по макро-уровню (штатам) также является методологически неверной: в США принято соблюдать единый архитектурный стиль и планировочные регламенты внутри конкретных локальных районов, но не в масштабах целого штата. В рамках одного региона могут одновременно находиться как сельские поселения с малой застройкой, так и дорогие мегаполисы, где площади и количество комнат имеют принципиально разную структуру. Поэтому оставшимся тупиковым значениям принудительно присваивается категория `unknown`. Если в этих редких объектах существуют скрытые математические паттерны, модель градиентного бустинга в ходе обучения отыщет их самостоятельно.

In [132]:
# Превращаем данные в чистые строки
clean_df["beds_clean"] = clean_df["beds_clean"].astype(str)
clean_df["baths_clean"] = clean_df["baths_clean"].astype(str)

# Убираем лишние ".0" у целых чисел, сохраняя дробные
clean_df["beds_clean"] = clean_df["beds_clean"].str.replace(".0", "", regex=False)
clean_df["baths_clean"] = clean_df["baths_clean"].str.replace(".0", "", regex=False)

# Переводим строковые пропуски в категорию unknown
clean_df["beds_clean"] = clean_df["beds_clean"].replace("nan", "unknown")
clean_df["baths_clean"] = clean_df["baths_clean"].replace("nan", "unknown")

# Итоговая проверка результатов
print("Уникальные категории спален:")
print(clean_df["beds_clean"].unique())

print("\nУникальные категории санузлов:")
print(clean_df["baths_clean"].unique())

Уникальные категории спален:
['4' '3' '5' '2' 'unknown' '8' '0' '1' '6' '9' '7']

Уникальные категории санузлов:
['3.5' '3' '2' '8' 'unknown' '4' '1' '5' '7' '2.1' '2.5' '0' '4.5' '6'
 '5.5' '1.5' '9' '7.5' '1.75' '6.5' '8.5' '1.1' '2.75' '2.25' '3.1' '3.25'
 '3.75' '5.2' '1.25' '2.2' '0.5' '4.25' '4.75' '0.75' '4.1' '5.25' '3.2'
 '6.75']


Стандартно проверяем дубликаты перед завершением блока.

In [133]:
# Проверяем и удаляем дубликаты, возникшие после фильтрации
dup_rooms = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов после блока характеристик жилых помещений: {dup_rooms}")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
# Итоговая фиксация размера матрицы
print(f"Новая общая размерность матрицы clean_df: {clean_df.shape}")

Обнаружено полных дубликатов после блока характеристик жилых помещений: 0
Новая общая размерность матрицы clean_df: (315969, 37)


### Парковка ('parking_clean')

Признак с самыми масштабными пропусками в данных. Наличие и тип парковочных мест  являются критически важным инфраструктурным фактором на рынке недвижимости США. Перед выбором стратегии заполнения пропусков проведем экспресс-диагностику: определим точный объем пропущенных значений и изучим текущую структуру заполненных категорий в разрезе типов объектов.

In [134]:
# Замеряем точный объем пропусков в признаке парковки
print(f"Всего пропущенных значений в parking_clean: {clean_df['parking_clean'].isna().sum()}")

# Изучаем распределение пропусков по типам недвижимости
print("\nПропуски парковки по типам объектов:")
print(clean_df[clean_df["parking_clean"].isna()]["property_type_clean"].value_counts())

# Смотрим, на распределение уникальных категорий парковок
print("\nТекущие уникальные категории парковок:")
print(clean_df["parking_clean"].value_counts())

Всего пропущенных значений в parking_clean: 134980

Пропуски парковки по типам объектов:
house           84655
apartment       21571
townhouse       10700
other            8160
multi_family     5821
land             2932
mobile           1138
commercial          3
Name: property_type_clean, dtype: int64

Текущие уникальные категории парковок:
garage                        94064
assigned_or_spaces            69171
carport                        8448
mixed_garage_carport           5120
street_parking                 1750
no_parking                     1434
assigned_or_reserved_space      728
special_rv_boat_parking         274
Name: parking_clean, dtype: int64


Выделять наличие парковки по географическому маркеру однозначно плохая идея. Наиболее разумным подходом в данном случае кажется определение отдельной категории - "unknown". Единственный момент который хочу проверить, это пустой земельный участок. Кажется, что для него наиболее вероятная категория "no_parking". Проверим гипотезу:

In [135]:
# Выделяем сегмент земли, где поле парковки НЕ пропущено (уже заполнено риелторами)
land_filled_parking = clean_df[
    (clean_df["property_type_clean"] == "land") & (clean_df["parking_clean"].notna())
]

print("Реальное распределение заполненных категорий парковки на голой земле:")
if not land_filled_parking.empty:
    print(land_filled_parking["parking_clean"].value_counts())
    print(f"\nВсего заполненных объектов земли: {len(land_filled_parking)}")
else:
    print("На всей заполненной земле поле парковки изначально было абсолютно пустым!")

Реальное распределение заполненных категорий парковки на голой земле:
garage                     976
mixed_garage_carport       389
carport                    174
assigned_or_spaces         123
street_parking              11
special_rv_boat_parking     10
no_parking                   9
Name: parking_clean, dtype: int64

Всего заполненных объектов земли: 1692


Предположение полностью опровергается реальной структурой данных. Наличие на незастроенной земле капитальных гаражей и крытых парковочных мест отражает специфику рынка США, где участки часто продаются с уже возведенными инфраструктурными хозблоками или представляют собой изолированные гаражные лоты. Навязывание искусственных маркеров в таких условиях недопустимо. Во избежание искажения данных абсолютно всем пропущенным значениям парковочного блока принудительно присваивается единая текстовая заглушка 'unknown'.

In [136]:
# Принудительно переводим абсолютно все пропуски парковки в категорию unknown
clean_df["parking_clean"] = clean_df["parking_clean"].fillna("unknown")

# Итоговый контроль качества очистки парковочного блока
print(f"Осталось пропусков в parking_clean: {clean_df['parking_clean'].isna().sum()}")
print("\nИтоговая структура категорий парковки:")
print(clean_df["parking_clean"].value_counts())
# Итоговая фиксация размера матрицы
print(f"\nНовая общая размерность матрицы clean_df: {clean_df.shape}")

Осталось пропусков в parking_clean: 0

Итоговая структура категорий парковки:
unknown                       134980
garage                         94064
assigned_or_spaces             69171
carport                         8448
mixed_garage_carport            5120
street_parking                  1750
no_parking                      1434
assigned_or_reserved_space       728
special_rv_boat_parking          274
Name: parking_clean, dtype: int64

Новая общая размерность матрицы clean_df: (315969, 37)


### Инфраструктурные параметры образования ('dist_elementary', 'dist_middle', 'dist_high' и рейтинги школ)

In [137]:
# Список анализируемых колонок образовательной инфраструктуры
school_cols = [
    "dist_elementary", "dist_middle", "dist_high", 
    "school_max_rating", "school_mean_rating", "school_median_rating"
]

print("Количество пропусков в школьном блоке до обработки:")
# Считаем сумму NaN для каждой колонки
print(clean_df[school_cols].isna().sum())

Количество пропусков в школьном блоке до обработки:
dist_elementary          7884
dist_middle              9431
dist_high               11688
school_max_rating        3164
school_mean_rating       3164
school_median_rating     3164
dtype: int64


Признаки близости и качества образовательной инфраструктуры выступают одними из сильнейших ценообразующих факторов на рынке жилья США, поскольку выбор объекта недвижимости часто продиктован его привязкой к конкретному школьному округу.

Логика восстановления оставшихся пропусков разделяется на два направления:
1. **Расстояния до школ**: Отсутствие информации о дистанции до определенного типа школы ('elementary', 'middle', 'high') свидетельствует об удаленности объекта от образовательной инфраструктуры. Для кодирования этого признака применяется метод 'max + 1': пропуски заполняются максимальным зафиксированным в датасете расстоянием, увеличенным на единицу, что дает модели четкий сигнал о недоступности объекта.
2. **Рейтинги школ**: Пропущенные метрики качества образования восстанавливаются с помощью пространственной агрегации. На основе географической привязки вычисляется медианный рейтинг учебных заведений внутри аналогичного индекса ('zipcode') или города ('city').

In [138]:
# Список признаков расстояний до школ
dist_cols = ["dist_elementary", "dist_middle", "dist_high"]

print("Заполнение пропусков расстояний методом 'max + 1':")
for col in dist_cols:
    # Находим максимальное расстояние в текущей колонке
    max_dist = clean_df[col].max()
    # Рассчитываем значение заглушки
    fill_value = max_dist + 1.0
    # Заполняем пропуски в датасете
    clean_df[col] = clean_df[col].fillna(fill_value)
    print(f"- Для {col} максимум равен {max_dist:.1f}, пропуски заполнены значением {fill_value:.1f}")

# Контрольная проверка остатков
print("\nОстатки пропусков после обработки расстояний:")
print(clean_df[dist_cols].isna().sum())

Заполнение пропусков расстояний методом 'max + 1':
- Для dist_elementary максимум равен 1590.4, пропуски заполнены значением 1591.4
- Для dist_middle максимум равен 1591.0, пропуски заполнены значением 1592.0
- Для dist_high максимум равен 1591.1, пропуски заполнены значением 1592.1

Остатки пропусков после обработки расстояний:
dist_elementary    0
dist_middle        0
dist_high          0
dtype: int64


После успешного кодирования расстояний до школ перейдем к восстановлению рейтингов образовательных учреждений ('school_max_rating', 'school_mean_rating', 'school_median_rating'). Оставшиеся 3 164 пропуска заполняются на основе каскадного географического принципа (локальная медиана ZIP-кода). Данный подход позволяет восстановить утерянные параметры качества образования при строгом соблюдении границ реальных школьных округов США. 

Перед запуском основного каскада импутации методологически верно провести предварительную верификацию данных на предмет физического отсутствия образовательной инфраструктуры. В случае обнаружения изолированных объектов, у которых расстояния до всех типов школ одновременно равны максимальной технической заглушке 'max + 1', их рейтинги подлежат принудительному занулению (0.0), поскольку оценивать в таких локациях нечего. 

In [139]:
# Выделяем маску объектов, у которых абсолютно все типы школ находятся на максимальной заглушке
no_schools_at_all = (
    (clean_df["dist_elementary"] == 1591.4) &
    (clean_df["dist_middle"] == 1592.0) &
    (clean_df["dist_high"] == 1592.0) &
    (clean_df["school_max_rating"].isna()) # проверяем только те, где остался пропуск
)

isolated_count = no_schools_at_all.sum()
print(f"Обнаружено изолированных объектов без какой-либо инфраструктуры школ: {isolated_count}")

# Если такие объекты нашлись, принудительно выставляем им рейтинг 0.0
if isolated_count > 0:
    clean_df.loc[no_schools_at_all, "school_max_rating"] = 0.0
    clean_df.loc[no_schools_at_all, "school_mean_rating"] = 0.0
    clean_df.loc[no_schools_at_all, "school_median_rating"] = 0.0
    print("Рейтинги для данных объектов успешно занулены.")

# Замеряем остатки пропусков перед тем, как передать их на обработку по индексу
print(f"\nОсталось пропусков в school_max_rating: {clean_df['school_max_rating'].isna().sum()}")

Обнаружено изолированных объектов без какой-либо инфраструктуры школ: 0

Осталось пропусков в school_max_rating: 3164


Проведенная экспресс-диагностика показала отсутствие подобных аномалий в текущем остатке пропусков, что подтверждает сплошное юридическое покрытие территории США школьными округами и техническую природу пропусков. Оставшийся массив утерянных рейтингов передается на пошаговую пространственную обработку.

Для восстановления 3 164 пропущенных значений в блоке школьных рейтингов применяется каскадный географический принцип (локальный индекс 'zipcode' с последующим бэкапом по городу 'city' и штату 'state'). В отличие от категориальных признаков, числовые метрики качества образования не позволяют использовать нейтральную текстовую заглушку типа 'unknown'. Во избежание грубого размытия данных на макро-уровне (штатах), где элитные кварталы математически смешиваются с депрессивными районами, принято решение полностью отказаться от жестких ограничений на минимальное количество аналогов. Порог репрезентативности снижается до уровня не менее 1 объекта на всех этапах каскада. Данный подход позволяет собрать максимум точной информации из локальных школьных округов (ZIP-кодов и городов), поскольку даже единичные зафиксированные аналоги внутри конкретного поселения физически и инфраструктурно ближе к исследуемым объектам, чем обобщенная статистика региона.


In [140]:
# Этап 1. Локальное заполнение по ZIP-кодам
clean_df["school_max_rating"] = clean_df["school_max_rating"].fillna(
    clean_df.groupby("zipcode")["school_max_rating"].transform("max")
)
clean_df["school_mean_rating"] = clean_df["school_mean_rating"].fillna(
    clean_df.groupby("zipcode")["school_mean_rating"].transform("mean")
)
clean_df["school_median_rating"] = clean_df["school_median_rating"].fillna(
    clean_df.groupby("zipcode")["school_median_rating"].transform("median")
)

# Промежуточный вывод
print(f"Итоговый остаток пропусков в school_max_rating: "
      f"{clean_df['school_max_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_mean_rating: "
      f"{clean_df['school_mean_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_median_rating: "
      f"{clean_df['school_median_rating'].isna().sum()}")

Итоговый остаток пропусков в school_max_rating: 105
Итоговый остаток пропусков в school_mean_rating: 105
Итоговый остаток пропусков в school_median_rating: 105


In [141]:
# Этап 2. Бэкап по городам для редких локаций
clean_df["school_max_rating"] = clean_df["school_max_rating"].fillna(
    clean_df.groupby("city")["school_max_rating"].transform("max")
)
clean_df["school_mean_rating"] = clean_df["school_mean_rating"].fillna(
    clean_df.groupby("city")["school_mean_rating"].transform("mean")
)
clean_df["school_median_rating"] = clean_df["school_median_rating"].fillna(
    clean_df.groupby("city")["school_median_rating"].transform("median")
)

# Промежуточный вывод
print(f"Итоговый остаток пропусков в school_max_rating: "
      f"{clean_df['school_max_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_mean_rating: "
      f"{clean_df['school_mean_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_median_rating: "
      f"{clean_df['school_median_rating'].isna().sum()}")

Итоговый остаток пропусков в school_max_rating: 13
Итоговый остаток пропусков в school_mean_rating: 13
Итоговый остаток пропусков в school_median_rating: 13


In [142]:
# Этап 3. Бэкап по штатам для единичных удаленных строк
clean_df["school_max_rating"] = clean_df["school_max_rating"].fillna(
    clean_df.groupby("state")["school_max_rating"].transform("max")
)
clean_df["school_mean_rating"] = clean_df["school_mean_rating"].fillna(
    clean_df.groupby("state")["school_mean_rating"].transform("mean")
)
clean_df["school_median_rating"] = clean_df["school_median_rating"].fillna(
    clean_df.groupby("state")["school_median_rating"].transform("median")
)

# Финальный контроль закрытия школьного блока в чистый ноль
print(f"Итоговый остаток пропусков в school_max_rating: "
      f"{clean_df['school_max_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_mean_rating: "
      f"{clean_df['school_mean_rating'].isna().sum()}")
print(f"Итоговый остаток пропусков в school_median_rating: "
      f"{clean_df['school_median_rating'].isna().sum()}")

Итоговый остаток пропусков в school_max_rating: 0
Итоговый остаток пропусков в school_mean_rating: 0
Итоговый остаток пропусков в school_median_rating: 0


In [143]:
# По традиции проверяем и удаляем дубликаты, возникшие после фильтрации
dup_rooms = clean_df.duplicated().sum()
print(f"Обнаружено полных дубликатов после полной очистки датасета: {dup_rooms}")
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
# Итоговая фиксация размера матрицы
print(f"Новая общая размерность матрицы clean_df: {clean_df.shape}")

Обнаружено полных дубликатов после полной очистки датасета: 0
Новая общая размерность матрицы clean_df: (315969, 37)


### Заключение

В рамках масштабной препроцессинговой очистки данных был полностью ликвидирован дефицит информации в исходной матрице признаков. Стратегия импутации выстраивалась на основе строгого баланса между сохранением физического смысла параметров и защитой от генерации случайного математического шума. Пропуски в ключевых инфраструктурных блоках (климат-контроль, этажность, комнатность, параметры образования) были точечно восстановлены с помощью каскадных пространственных мод, учитывающих типы недвижимости и локальные планировочные корироды зданий. Экстремальные аномалии и неинформативные рыночные статусы были системно нейтрализованы, а финальные тупиковые остатки изолированы устойчивыми категориальными заглушками 'unknown'. 

Итоговая размерность чистой рабочей матрицы зафиксирована в объеме 315 969 строк при абсолютном отсутствии пропущенных значений (NaN) во всех обработанных признаках. Сформированный массив данных является математически стерильным фундаментом для перехода к этапу разведочного анализа (EDA) и построения моделей машинного обучения.

In [144]:
# Сохраняем чистый датасет на диск (индексы сбрасываем для чистоты структуры)
clean_df.to_csv("data/clean_data.csv", index=False)
print("Чистый датасет успешно сохранен в файл 'clean_data.csv'")

Чистый датасет успешно сохранен в файл 'clean_data.csv'


**Переход к следующему этапу исследования**

Ввиду масштабности исходного датасета и высокой вычислительной плотности процедур препроцессинга, этап детальной очистки матрицы и каскадного восстановления пропущенных значений полностью обособлен в рамках данного рабочего пространства. Финальный очищенный массив данных успешно экспортирован в файл 'data/clean_data.csv'. 

Для оптимизации оперативной памяти, предотвращения зависания графического ядра при визуализации и обеспечения воспроизводимости экспериментов, дальнейшее исследование переносится в новый изолированный ноутбук. Следующим этапом работы станет Разведочный анализ данных (Exploratory Data Analysis / EDA) и непосредственное построение прогностических моделей машинного обучения на базе подготовленного математически стерильного фундамента.